In [1]:
# ===== CELL 0: SETUP =====
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings
warnings.filterwarnings("ignore")      # hide non-critical warnings

# TECH: a seed fixes the random number generator, so any random step later
# (splits, shuffles, simulations) gives the same result on every run.
SEED = 42
np.random.seed(SEED)

sns.set_theme(style="whitegrid")                    # one consistent plot style
pd.set_option("display.max_columns", 50, "display.width", 140)   # stop pandas hiding columns

# File sits next to the notebook, so the name alone is enough.
# (If your local file name differs, e.g. has spaces, change it here.)
DATA_PATH = "../data/Amazon Pay India Analytics Case Dataset.csv"


### Setup

**Procedure**
We loaded the analysis libraries, fixed a random seed (42) and set one plot style. The seed makes every random step (model splits, shuffles) repeatable, so anyone re-running the notebook gets the same numbers.

In [2]:
# ===== CELL 1: LOAD + AUDIT =====
raw = pd.read_csv(DATA_PATH)
df = raw.copy()        # TECH: work on a copy, so `raw` stays as the untouched original

# TECH: snake_case names ("Monthly income inr" -> "monthly_income_inr") are easier to type
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Shape:", df.shape)               # BIZ: should be the 5,000 customers in the case
print("\nDtypes:\n", df.dtypes)          # TECH: wrong types (numbers stored as text) break maths

# BIZ: duplicate customers or missing values would silently bias every average and churn rate
print("\nNulls:", int(df.isna().sum().sum()), "| Duplicates:", int(df.duplicated().sum()),
      "| Customer id unique:", df["customer_id"].is_unique)

# TECH: numeric columns get averages/correlations; categorical ones get counts per category.
# customer_id (a label) and churned (the target) are left out of both lists.
NUM = ["age", "monthly_income_inr", "tenure_months", "avg_monthly_transactions",
       "avg_transaction_value_inr", "cashback_used_pct", "app_engagement_score",
       "customer_support_contacts", "net_promoter_score"]
CAT = ["gender", "city_tier", "amazon_prime_member"]

# TECH: describe() = count, mean, std (spread), min, quartiles, max. If mean ~ median and the
# range is spread evenly, the variable is roughly uniform, like your pair-plot showed.
print("\nNumeric summary:\n", df[NUM].describe().T.round(2))

# BIZ: segment sizes. "Other" gender is ~2% (~100 people), so any pattern there is fragile.
for c in CAT:
    print(f"\n{c}:\n", df[c].value_counts().to_frame("n")
                          .assign(pct=lambda x: (x.n / len(df) * 100).round(1)))

# TECH: the mean of a 0/1 column is the share of 1s, i.e. the churn rate.
# BIZ: this 19% is the baseline every segment and model must beat.
print("\nChurn rate:", round(df.churned.mean(), 4), "| churners:", int(df.churned.sum()))

# --- Integrity checks: does the data obey real-world rules? ---
# BIZ: monthly spend = transactions x average value = money moved through Amazon Pay per month.
spend = df.avg_monthly_transactions * df.avg_transaction_value_inr

checks = {
    # tenure (months) / 12 = years with us; age - 18 = years as an adult
    "tenure longer than adult life": int((df.tenure_months / 12 > df.age - 18).sum()),
    "monthly spend > income":        int((spend > df.monthly_income_inr).sum()),
    "monthly spend > 3x income":     int((spend > 3 * df.monthly_income_inr).sum()),
}
print("\nIntegrity checks (rows flagged):")
for k, v in checks.items():
    print(f"  {v:>5}  {k}")

# TECH: if customer_id is ordered by sign-up, a non-zero correlation with churn would hint at a
# hidden time effect. Near 0 means the ID carries no information.
print("\nCorr(customer_id, churned):", round(df.customer_id.corr(df.churned), 4))

# TECH: quantiles show spread without being distorted by extremes ({0.5: 3.08} = median ratio).
print("Spend/income quantiles:",
      (spend / df.monthly_income_inr).quantile([.1, .25, .5, .75, .9]).round(2).to_dict())

Shape: (5000, 14)

Dtypes:
 customer_id                  int64
age                          int64
gender                         str
city_tier                      str
monthly_income_inr           int64
tenure_months                int64
amazon_prime_member          int64
avg_monthly_transactions     int64
avg_transaction_value_inr    int64
cashback_used_pct            int64
app_engagement_score         int64
customer_support_contacts    int64
net_promoter_score           int64
churned                      int64
dtype: object

Nulls: 0 | Duplicates: 0 | Customer id unique: True

Numeric summary:
                             count      mean       std      min      25%      50%       75%       max
age                        5000.0     40.83     13.66     18.0     29.0     41.0      53.0      64.0
monthly_income_inr         5000.0  98834.13  47051.33  18020.0  58110.5  98603.5  139514.0  179981.0
tenure_months              5000.0     36.50     20.44      1.0     19.0     37.0      54.0   

### Load and audit the data

**Procedure**
We loaded the 5,000-customer file and checked it before analysing it: size, data types, missing values, duplicates and unique IDs. We then summarised each column and ran logic checks on combinations of columns (can tenure exceed adult life? can spend exceed income?). Analysing unaudited data risks building conclusions on errors.

**Observations**
- **The file is clean:** 5,000 unique customers, no missing values, no duplicates.
- **1 in 5 customers churns:** the churn rate is 19.0% (950 customers). This is our baseline, and every segment and model must beat it to be useful.
- **Customer mix:** 54.5% male, 43.6% female, 1.9% other (only 97 people). 42% of customers are in Tier 1 cities and 46% are Prime members.
- **Typical customer:** income of about ₹99k a month, 36 months with Amazon Pay, about 76 transactions a month at roughly ₹5,000 each.
- **Data-quality flags:**
  - 383 customers (7.7%) have a tenure longer than their adult life.
  - 77% have monthly spend above monthly income, and about half above 3x income.
  - The median spend-to-income ratio is about 3.
- **What this means:** spend above income is plausible for a payment platform. Monthly spend is payment *volume*, not out-of-pocket money, so it can exceed income when customers fund payments with credit cards or loans, when family or friends use the account, or when it is used for small-business payments. The data cannot confirm which applies, so we treat these as plausible explanations, not facts. We keep all rows and do not treat this as a data error. Because spend does not measure how much of a customer's income they commit to Amazon Pay, we call the ratio **payment intensity** and use it only as a ranking, not as a literal percentage.
- **No hidden time effect:** customer ID has no link with churn (correlation −0.002).

In [3]:
# ===== CELL 2: HOW IS EACH FEATURE DISTRIBUTED? =====
# BIZ: before looking for patterns, know the shape of your customer base.
C_ACTIVE, C_CHURN = "#2B7BBF", "#D2591A"     # fixed colours: blue = baseline, orange = churn
def pretty(c): return c.replace("_", " ").replace(" inr", " (INR)").replace(" pct", " (%)").capitalize()

# TECH: a histogram counts customers per value band. subplots(3, 3) = a 3x3 grid of charts.
fig, axes = plt.subplots(3, 3, figsize=(14, 9))
for ax, col in zip(axes.ravel(), NUM):
    sns.histplot(df[col], bins=25, color=C_ACTIVE, edgecolor="white", ax=ax)
    ax.axvline(df[col].mean(), color=C_CHURN, lw=2, ls="--")      # dashed line = the mean
    ax.set_title(pretty(col), fontsize=11, weight="bold"); ax.set_xlabel(""); ax.set_ylabel("Customers")
fig.suptitle("Every numeric feature is spread almost evenly (dashed line = mean)",
             fontsize=14, weight="bold", y=1.01)
plt.tight_layout(); plt.show()

# TECH: skewness = lopsidedness. ~0 means symmetric; above +1 means a long right tail.
# BIZ: real customer data usually has a few big spenders (right skew). Flat ~0 skew across
# every column is unusual and hints the data is synthetic or generated uniformly.
print("Skewness:\n", df[NUM].skew().round(2).to_string())

Skewness:
 age                          0.03
monthly_income_inr          -0.01
tenure_months               -0.03
avg_monthly_transactions    -0.03
avg_transaction_value_inr    0.01
cashback_used_pct            0.01
app_engagement_score        -0.02
customer_support_contacts   -0.02
net_promoter_score          -0.00


### Distribution of each numeric feature

**Procedure**
We plotted a histogram for each of the 9 numeric features, with a dashed line at the mean, and computed skewness (how lopsided a distribution is). This shows what the customer base looks like before we look for patterns.

**Observations**
- **Every feature is spread evenly across its range.** Age, income, tenure, transactions, transaction value, cashback use and NPS are all roughly flat. Engagement (1 to 10) and support contacts (0 to 9) have about 500 customers at each score.
- **Skewness is about zero for all features** (between −0.03 and +0.03).
- **This is unusual for real payment data.** Real customer bases normally have a long tail of a few very heavy spenders and many light ones. A flat spread suggests a synthetic or evenly generated dataset.
- **The small dips (age near 30, 41 and 51; tenure near 15 and 55) are not real gaps.** Age and tenure are whole numbers, and some histogram bins happen to contain one whole number while others contain two. We treat them as a plotting artefact.
- **Implication:** there are no obvious outliers or "power users" that dominate averages. There is also no natural "typical customer", so segments will have to be defined by business rules.

In [4]:
# ===== CELL 3: CHURN BY SEGMENT (WITH UNCERTAINTY) =====
from scipy.stats import chi2_contingency
BASE = df.churned.mean()

# TECH: a churn rate from a small group is noisy. A 95% confidence interval (CI) is the range the
# true rate likely sits in. The Wilson method behaves well for proportions and small groups.
# BIZ: if a segment's CI still covers the 19% baseline, we cannot call it different.
def wilson(k, n, z=1.96):
    p = k / n
    centre = (p + z**2 / (2*n)) / (1 + z**2 / n)
    half = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / (1 + z**2 / n)
    return centre - half, centre + half

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
rows = []
for ax, col in zip(axes, CAT):
    g = df.groupby(col).churned.agg(["sum", "count"])        # churners and customers per segment
    g["rate"] = g["sum"] / g["count"]
    g[["lo", "hi"]] = [wilson(k, n) for k, n in zip(g["sum"], g["count"])]
    x = np.arange(len(g))
    ax.bar(x, g.rate * 100, color=C_CHURN, width=0.5)
    ax.errorbar(x, g.rate * 100, yerr=[(g.rate-g.lo)*100, (g.hi-g.rate)*100],
                fmt="none", ecolor="#333", capsize=4, lw=1.2)  # whiskers = the CI
    ax.axhline(BASE * 100, color=C_ACTIVE, ls="--", lw=2)      # overall churn = the benchmark
    ax.set_xticks(x); ax.set_xticklabels([f"{i}\n(n={n:,})" for i, n in zip(g.index, g["count"])])
    ax.set_title(pretty(col), weight="bold"); ax.set_ylim(0, 40)
    # TECH: chi-square test asks "could these churn differences arise by pure chance?"
    # p < 0.05 = unlikely by chance; p > 0.05 = consistent with chance.
    pval = chi2_contingency(pd.crosstab(df[col], df.churned))[1]
    ax.text(0.97, 0.94, f"chi-square p = {pval:.2f}", transform=ax.transAxes, ha="right", fontsize=10)
    rows.append({"feature": col, "p_value": round(pval, 3)})
axes[0].set_ylabel("Churn rate (%)")
fig.suptitle("Churn rate by segment (bars = rate, whiskers = 95% CI, dashed = 19% overall)",
             fontsize=13, weight="bold")
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).to_string(index=False))

            feature  p_value
             gender    0.052
          city_tier    0.204
amazon_prime_member    0.286


### Churn rate by customer segment

**Procedure**
We compared churn across gender, city tier and Prime membership. Small groups give noisy rates, so each bar shows a 95% confidence interval (the range the true rate likely falls in) and a chi-square test p-value. A p-value above 0.05 means the difference is consistent with chance. The dashed line is the overall 19% churn rate.

**Observations**
- **City tier makes no real difference:** Tier 1 20.0%, Tier 2 17.7%, Tier 3 18.9% (p = 0.20).
- **Prime membership makes no real difference:** Prime 19.7% vs non-Prime 18.4% (p = 0.29). Prime members do not churn less, so Prime on its own is not acting as a retention lock-in. This is an early hint for Q2.
- **Gender is borderline (p = 0.052), but only because of the "Other" group:** 27.8% churn, from just 97 customers. Its confidence interval is very wide (about 20% to 37%) and only just clears the 19% line. Female (18.2%) and Male (19.3%) are effectively the same.
- **Treatment:** with 97 customers and several segments tested, "Other" is a watch-item, not a finding. We would not design a retention programme around it. Cell 9A shows why: a group of 97 customers would need a churn gap of about 11 points to be detected reliably, and the observed gap is 8.8 points.

In [5]:
# ===== CELL 4: CHURN ACROSS THE RANGE OF EACH NUMERIC FEATURE =====
from scipy.stats import mannwhitneyu
from matplotlib.ticker import EngFormatter        # shows 100000 as "100 k"

# TECH: box plots hide small shifts in a RATE. Better: cut each feature into 5 equal-sized bands
# (quintiles, via pd.qcut), then plot the churn rate per band. A real driver shows as a slope.
# BIZ: "do heavy users churn less than light users?" is exactly a slope question.
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
rows = []
for ax, col in zip(axes.ravel(), NUM):
    bins = pd.qcut(df[col], q=5, duplicates="drop")      # duplicates="drop": discrete features repeat values
    g = df.groupby(bins, observed=True).agg(x=(col, "mean"), k=("churned", "sum"), n=("churned", "count"))
    g["rate"] = g.k / g.n
    g[["lo", "hi"]] = [wilson(k, n) for k, n in zip(g.k, g.n)]
    ax.fill_between(g.x, g.lo * 100, g.hi * 100, color=C_CHURN, alpha=0.2)   # shaded = CI
    ax.plot(g.x, g.rate * 100, color=C_CHURN, marker="o", lw=2)
    ax.axhline(BASE * 100, color=C_ACTIVE, ls="--", lw=1.5)
    if df[col].max() > 5000: ax.xaxis.set_major_formatter(EngFormatter())
    ax.set_ylim(10, 30); ax.set_title(pretty(col), fontsize=11, weight="bold"); ax.set_xlabel("")
    # TECH: Mann-Whitney U compares churners vs actives without assuming a bell curve.
    # (Chosen over a t-test because these features are flat/uniform, not bell-shaped.)
    p = mannwhitneyu(df.loc[df.churned == 1, col], df.loc[df.churned == 0, col],alternative="two-sided").pvalue
    rows.append({"feature": col, "median_active": df.loc[df.churned == 0, col].median(),
                 "median_churned": df.loc[df.churned == 1, col].median(), "p_value": round(p, 3)})
fig.suptitle("Churn rate across 5 equal-sized bands of each feature (band = CI, dashed = 19%)",
             fontsize=14, weight="bold", y=1.0)
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).to_string(index=False))

                  feature  median_active  median_churned  p_value
                      age           41.0            40.0    0.698
       monthly_income_inr        98592.0         99291.5    0.964
            tenure_months           37.0            37.0    0.953
 avg_monthly_transactions           75.0            80.0    0.280
avg_transaction_value_inr         4968.0          5087.0    0.996
        cashback_used_pct           49.0            47.0    0.250
     app_engagement_score            6.0             6.0    0.544
customer_support_contacts            5.0             5.0    0.925
       net_promoter_score            0.0            -3.5    0.542


### Churn across the range of each numeric feature

**Procedure**
For each numeric feature we split customers into 5 equal-sized bands (lowest 20% to highest 20%) and plotted the churn rate in each band. If a feature drives churn, the line should slope up or down. The shaded area is the 95% confidence interval and the dashed line is the 19% baseline. We also compared churners with active customers using a two-sided Mann-Whitney test (it does not assume a bell curve, which suits these flat distributions).

**Observations**
- **No feature shows a trend.** Every line stays inside the uncertainty band around 19%, from the lowest income band to the highest, and from new to long-tenured customers.
- **Median gaps are tiny and not significant:** for example, churners make 80 transactions a month vs 75 for actives, and have a median NPS of −3.5 vs 0. All p-values are between 0.25 and 0.996.
- **The few wiggles are noise.** The 100 to 120 transactions band (about 21%) and the top engagement and support-contact scores (about 20.5%) have wide bands that overlap the baseline. With 9 features × 5 bands we expect a few points to drift by chance.
- **No tipping point exists in the raw data.** There is no rule like "customers with fewer than X transactions churn more", so a simple threshold-based churn alert would not work with these columns.

In [6]:
# ===== CELL 5: CORRELATIONS =====
# TECH: Spearman correlation works on RANKS, so it catches any steadily rising/falling link
# (not just straight lines) and is not thrown off by outliers. Range: -1 to +1; 0 = no link.
cols = NUM + ["amazon_prime_member", "churned"]
corr = df[cols].corr(method="spearman")
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)       # hide the duplicate upper triangle
plt.figure(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, linewidths=.5, cbar_kws={"shrink": .8})
plt.title("Spearman correlation: near zero everywhere, including with churn", weight="bold")
plt.tight_layout(); plt.show()
# BIZ: the bottom row (churned) is the one that matters: how well does each feature track churn?
print("Strongest |correlation| with churn:\n",
      corr["churned"].drop("churned").abs().sort_values(ascending=False).head(3).round(3).to_string())



Strongest |correlation| with churn:
 cashback_used_pct           0.016
amazon_prime_member         0.016
avg_monthly_transactions    0.015


### Correlation between features (and with churn)

**Procedure**
We computed Spearman correlations between all numeric features and churn. Spearman uses ranks, so it catches any steady rising or falling relationship and is not distorted by outliers. 0 means no link and ±1 means a perfect link.

**Observations**
- **Churn has no linear or rank link with any feature.** The strongest correlations are 0.016 (cashback use), 0.016 (Prime) and 0.015 (transactions), which are essentially zero.
- **The features are also independent of each other:** all pairs sit within ±0.03. In real customer data we would expect income to move with spend, and tenure with engagement. Here it does not.
- **Business meaning:**
  - Features do not overlap, so there is no duplicated information to remove.
  - Customers do not fall into natural clusters, so segments must be built from business logic (value, engagement, cashback behaviour), not discovered from the data.
- **Limit of this test:** correlation only captures monotonic relationships.

In [7]:
# ===== CELL 5A: PAIR PLOTS (bivariate view of every feature pair) =====
from scipy.stats import spearmanr

# Colours are fixed per meaning: blue = Active / Prime, orange = Churned, grey = Non-Prime
C_ACTIVE, C_CHURN = "#2B7BBF", "#D2591A"
C_PRIME, C_NON = "#2B7BBF", "#7F7F7F"

# Short, readable labels for the 7 numeric features
PAIR_COLS = {"age": "Age", "monthly_income_inr": "Income", "tenure_months": "Tenure",
             "avg_monthly_transactions": "Txns / month", "avg_transaction_value_inr": "Ticket size",
             "cashback_used_pct": "Cashback %", "net_promoter_score": "NPS"}

pp = df[list(PAIR_COLS)].rename(columns=PAIR_COLS)
pp["Churn status"] = df.churned.map({0: "Active", 1: "Churned"})
pp["Prime status"] = df.amazon_prime_member.map({1: "Prime", 0: "Non-Prime"})

# TECH: Spearman uses ranks, so it catches any steadily rising or falling relationship,
#       not just straight lines, and extreme incomes or ticket sizes do not distort it.
# BIZ:  "Do customers with a higher X also tend to have a higher Y?" +1 = always, 0 = no link, -1 = opposite.
rho = pp[list(PAIR_COLS.values())].corr(method="spearman")

# TECH: plotting 5,000 points per panel turns into a solid blob (overplotting).
#       A random 2,000-row sample keeps the pattern visible, and the sample is for drawing only.
#       The correlations above use all 5,000 rows.
pp_s = pp.sample(2000, random_state=SEED)

def pair_grid(hue, hue_order, palette, title):
    pg = sns.PairGrid(pp_s, vars=list(PAIR_COLS.values()), hue=hue, hue_order=hue_order,
                      palette=palette, diag_sharey=False, height=1.7, aspect=1)
    pg.map_lower(sns.scatterplot, s=9, alpha=0.45, linewidth=0)             # lower triangle: scatter
    pg.map_diag(sns.kdeplot, fill=True, alpha=0.25, common_norm=False, linewidth=1.5)  # diagonal: density per group

    def show_rho(x, y, **kw):                                               # upper triangle: the number
        ax = plt.gca()
        if not ax.texts:                                                    # print once, not once per hue
            r = rho.loc[x.name, y.name]
            ax.text(0.5, 0.5, f"{r:+.2f}", ha="center", va="center", transform=ax.transAxes,
                    fontsize=15, weight="bold", color="#333")
        ax.set_axis_off()
    pg.map_upper(show_rho)
    pg.add_legend(title="")
    # TECH: seaborn renamed .fig to .figure in v0.11.2, so this line works on both old and new versions
    fig_obj = getattr(pg, "figure", None) or pg.fig
    fig_obj.suptitle(title, y=1.02, fontsize=13, weight="bold")
    return pg

pair_grid("Churn status", ["Active", "Churned"], {"Active": C_ACTIVE, "Churned": C_CHURN},
          "Pairs of features, coloured by churn (upper right = Spearman correlation)")
plt.show()

pair_grid("Prime status", ["Non-Prime", "Prime"], {"Non-Prime": C_NON, "Prime": C_PRIME},
          "Pairs of features, coloured by Prime membership (upper right = Spearman correlation)")
plt.show()

off = rho.values[np.triu_indices(len(rho), k=1)]                           # the 21 unique pairs
print(f"Across all {len(off)} feature pairs: largest |correlation| = {np.abs(off).max():.3f}, "
      f"median |correlation| = {np.median(np.abs(off)):.3f}")

Across all 21 feature pairs: largest |correlation| = 0.025, median |correlation| = 0.010


### Procedure
1. Took the 7 numeric features (Age, Income, Tenure, Txns/month, Ticket size, Cashback %, NPS) and plotted every pair of them against each other, 21 pairs in total.
2. Drew the grid twice: once coloured by churn status (Active vs Churned) and once by Prime status (Non-Prime vs Prime).
3. Lower triangle shows the scatter of 2,000 randomly sampled customers (the sample is for drawing only). Diagonal shows each feature's distribution per group. Upper triangle prints the Spearman rank correlation of each pair, computed on all 5,000 customers.

### Observations
- **No two features move together.** All 21 correlations lie between -0.025 and +0.025, with a median of about 0.01. Older customers do not earn more, longer-tenured customers do not transact more, and higher spenders do not give higher NPS.
- **The scatter plots are flat, uniform rectangles.** There is no slope, cluster or curve in any panel. The features look independently generated, not like a real customer base where income, age and ticket size usually move together.
- **Churned and Active customers are mixed together.** The orange points sit on top of the blue ones in every panel, and the two density curves on the diagonal overlap almost completely. No single feature, and no pair of features, picks out the churners.
- **Prime and Non-Prime customers are mixed together as well.** Prime members are no older, richer, longer-tenured, heavier users or happier than non-members, so Prime membership does not mark a distinct customer type.
- **Business reading:** there is no hidden "type of customer" that leaves, and Prime does not make a customer type. Any retention or cashback action has to rest on how customers behave (for example, redemption patterns), not on who they are.

In [8]:
# ===== CELL 5B: PAIRWISE COMBINATIONS vs CHURN and PRIME =====
# TECH: Cell 5A looks at pairs by eye. Here we test them. For each pair of features we cut
#       both into quartiles (4 x 4 = 16 cells), then run a chi-square test of whether the
#       churn rate (or Prime share) differs across those 16 cells. This catches interactions
#       such as "high income AND low tenure", which single-feature tests can miss.
# BIZ:  "Is there any customer type, defined by two traits together, that behaves differently?"
from scipy.stats import chi2_contingency

names = list(PAIR_COLS); labels = list(PAIR_COLS.values()); k = len(names)
binned = {c: pd.qcut(df[c], q=4, labels=False, duplicates="drop") for c in names}

def pair_pvalue(a, b, target):
    cell = binned[a] * 10 + binned[b]                  # unique id for each quartile combination
    return chi2_contingency(pd.crosstab(cell, target))[1]

targets = {"Churn": df.churned, "Prime membership": df.amazon_prime_member}

# TECH: 21 tests means some will look "significant" by luck alone.
#       Bonferroni divides the 5% threshold by the number of tests.
n_pairs = k * (k - 1) // 2
alpha_adj = 0.05 / n_pairs

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (tname, target) in zip(axes, targets.items()):
    P = np.full((k, k), np.nan)
    for i in range(k):
        for j in range(i):
            P[i, j] = pair_pvalue(names[i], names[j], target)
    low = P[np.tril_indices(k, -1)]
    print(f"{tname}: smallest pair p-value = {np.nanmin(P):.3f} | pairs below 0.05: {(low < 0.05).sum()} of {n_pairs} "
          f"| below corrected {alpha_adj:.4f}: {(low < alpha_adj).sum()}")

    # Darker = stronger evidence of a pattern. -log10(p): p=0.05 -> 1.3, p=0.0024 -> 2.6.
    strength = pd.DataFrame(-np.log10(P), index=labels, columns=labels)
    sns.heatmap(strength, mask=np.isnan(P), cmap="Blues", vmin=0, vmax=3,
                annot=pd.DataFrame(P, index=labels, columns=labels).round(2), fmt="",
                linewidths=.5, annot_kws={"fontsize": 9},
                cbar_kws={"label": "Evidence strength (-log10 p); 2.6 = corrected threshold"}, ax=ax)
    ax.set_title(f"Does any pair of features separate {tname.lower()}? (p-values)", weight="bold")
plt.tight_layout(); plt.show()

Churn: smallest pair p-value = 0.219 | pairs below 0.05: 0 of 21 | below corrected 0.0024: 0


Prime membership: smallest pair p-value = 0.158 | pairs below 0.05: 0 of 21 | below corrected 0.0024: 0


### Procedure
1. Cut each feature into 4 equal-sized groups (quartiles). For every pair of features this gives 16 customer types, such as "high income and short tenure". There are 21 pairs, so 336 types in all.
2. For each pair, ran a chi-square test of whether the churn rate (and, separately, the Prime share) differs across the 16 types.
3. Because 21 tests are run per label, some would look significant by luck, so we also used a Bonferroni-corrected threshold of 0.05 / 21 = 0.0024.
4. Plotted the p-values as a heatmap. Darker blue means stronger evidence of a pattern, and white means nothing there.

### Observations
- **Churn: no pair of features separates churners.** The smallest p-value across the 21 pairs is 0.219 (Tenure x NPS). Zero pairs are below 0.05, so not even a single uncorrected "lucky" hit appears, and none comes anywhere near the corrected threshold of 0.0024.
- **Prime: no pair of features separates Prime members.** The smallest p-value is 0.158 (Income x Ticket size). Zero pairs are below 0.05.
- **Both heatmaps are almost white.** None of the 21 cells even approaches the corrected threshold.
- **This closes the interaction gap.** Earlier tests looked at one feature at a time. This shows that combining two features does not reveal a hidden pattern either, in case one existed, such as "high income AND short tenure".
- **Fair limit of this test:** each of the 16 types holds about 300 customers, so it can detect moderate differences but not tiny ones. The earlier models agree with it, though: churn prediction stays at chance level (AUC about 0.5) even when many features are combined.
- **Business reading:** neither churn nor Prime membership follows any pattern in customer profile, whether features are taken singly or in pairs. This supports the Q1 conclusion of "no strong driver found". It also supports shifting effort to the questions the data can answer: cashback efficiency, behaviour-based segments and a test-and-learn rollout.

In [9]:
# ===== CELL 6: CAN A MODEL PREDICT CHURN FROM THE RAW COLUMNS? =====
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# TECH: one-hot encoding turns text categories into 0/1 columns (drop_first avoids a redundant one).
X = pd.get_dummies(df.drop(columns=["customer_id", "churned"]), drop_first=True)
y = df.churned

# TECH: 5-fold cross-validation = train on 4/5 of customers, test on the unseen 1/5, repeat 5 times.
# Stratified keeps the 19% churn share in every fold. This avoids fooling ourselves with a model
# that just memorised the data.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Two very different model families: a straight-line one and a flexible tree-based one that can
# find curves and interactions. If NEITHER works, the signal is not in the raw columns.
models = {"Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
          "Gradient boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)}

# TECH: AUC = the chance the model ranks a random churner above a random non-churner.
# 0.5 = coin flip, 1.0 = perfect. BIZ: AUC ~0.5 means a retention team targeting by this model
# would do no better than calling customers at random.
# TECH: "shuffled" control = same test with churn labels randomly scrambled. This shows what
# "pure noise" scores, so we know the real baseline.
y_shuffled = y.sample(frac=1, random_state=42).reset_index(drop=True)
res = []
for name, m in models.items():
    real = cross_val_score(m, X, y, cv=cv, scoring="roc_auc")
    fake = cross_val_score(m, X, y_shuffled.values, cv=cv, scoring="roc_auc")
    res.append({"model": name, "AUC_real": real.mean().round(3),
                "AUC_real_sd": real.std().round(3), "AUC_shuffled": fake.mean().round(3)})
res = pd.DataFrame(res); print(res.to_string(index=False))

ax = res.set_index("model")[["AUC_real", "AUC_shuffled"]].plot.bar(
        color=[C_CHURN, "#999999"], figsize=(7, 4), rot=0)
ax.axhline(0.5, color="black", ls="--", lw=1); ax.set_ylim(0.4, 0.7); ax.set_ylabel("AUC (0.5 = coin flip)")
ax.set_title("Raw features predict churn no better than chance", weight="bold")
plt.tight_layout(); plt.show()

              model  AUC_real  AUC_real_sd  AUC_shuffled
Logistic regression     0.491        0.017         0.469
  Gradient boosting     0.468        0.018         0.490


### Can a model predict churn from the raw columns?

**Procedure**
We trained two very different models on all raw columns: logistic regression (straight-line relationships) and gradient boosting (curves and interactions between features). We scored them with 5-fold cross-validation (train on 4/5 of customers, test on the unseen 1/5, repeated 5 times). The score is AUC: the chance the model ranks a random churner above a random non-churner (0.5 = coin flip, 1.0 = perfect). As a control we repeated the test with churn labels randomly shuffled, which shows what pure noise scores.

**Observations**
- **Neither model beats chance:** logistic regression 0.491 and gradient boosting 0.468, against 0.469 and 0.490 for shuffled labels.
- **The gaps are within noise.** The fold-to-fold variation is about ±0.017, so a score of 0.468 is not "worse than random" in any meaningful way.
- **Business meaning:** a retention team using these 12 raw columns to decide who to call would do no better than calling at random. Raw customer attributes do not explain who leaves.
- **Likely reasons:** the real churn drivers are not in this dataset (for example failed payments, competitor offers, app issues or recent behaviour change), or the data is synthetic with randomly assigned churn.
- **Next step:** combine the raw columns into business-meaningful features (spend relative to income, engagement vs cashback, tenure stage) and test whether any of them do better than chance.

## EDA Summary

### 1. What we examined
We audited and explored the 5,000-customer Amazon Pay India dataset (12 customer attributes, plus an ID and the churn flag) in six steps: data audit, feature distributions, churn by segment, churn across feature ranges, correlations, and a model-based test of whether the raw columns can predict churn. The goal was to find what drives churn and to decide how to approach the five business questions.

### 2. Data quality and integrity
- **Clean file:** no missing values, no duplicate rows, every customer ID unique. No rows needed removal.
- **Baseline churn:** 19.0% (950 of 5,000 customers), roughly 1 in 5. Every segment and model is judged against this number.
- **Customer mix:** 54.5% male, 43.6% female, 1.9% other (97 customers); Tier 1 / Tier 2 / Tier 3 cities are 42% / 32% / 25%; 45.7% are Prime members.
- **Typical customer:** about ₹99k monthly income, 36 months of tenure, about 76 transactions a month at roughly ₹5,000 each.
- **Plausibility flags:**
  - 383 customers (7.7%) have a tenure longer than their adult life.
  - 77% have monthly spend above monthly income, and about half above 3x income. The median spend-to-income ratio is about 3.
- **Our reading:** spend is payment volume, not out-of-pocket money. It can exceed income when customers pay by credit card or loan, when family or friends use the account, or for small-business use. The data cannot confirm which applies. We keep all rows, call the ratio "payment intensity", and use it only as a ranking.
- **No ordering effect:** customer ID has no link with churn (correlation −0.002).

### 3. Shape of the data
- Every numeric feature (age, income, tenure, transactions, transaction value, cashback use, NPS) is spread almost evenly across its range, with skewness near zero. Engagement (1 to 10) and support contacts (0 to 9) have about 500 customers at each score.
- Real payment data usually has a long tail of heavy spenders. A flat spread suggests synthetic or evenly generated data, so we do not expect natural "power user" clusters.
- Small dips in the age and tenure histograms are a plotting artefact of whole-number values, not real gaps.

### 4. Churn by segment
| Segment | Churn rate | Finding |
|---|---|---|
| Gender: Female / Male / Other | 18.2% / 19.3% / 27.8% | p = 0.052. Driven only by "Other" (97 customers, wide confidence interval). A watch-item, not a finding |
| City tier: 1 / 2 / 3 | 20.0% / 17.7% / 18.9% | p = 0.20. No real difference |
| Prime: yes / no | 19.7% / 18.4% | p = 0.29. Prime members do not churn less |

### 5. Churn across feature ranges
- We split each numeric feature into 5 equal bands and plotted the churn rate in each. All lines stay inside the uncertainty band around 19%, with no upward or downward trend.
- Churners and active customers have almost identical medians (for example 80 vs 75 transactions a month, and an NPS of −3.5 vs 0). Two-sided Mann-Whitney p-values range from 0.25 to 0.996.
- The few wiggles (for example the 100 to 120 transactions band at about 21%) are within noise: with 9 features x 5 bands, a few points will drift by chance.
- There is no tipping point such as "customers below X transactions churn more", so a simple threshold alert would not work on these columns.

### 6. Relationships between variables
- **With churn:** every Spearman correlation is within ±0.02 (strongest: cashback use 0.016, Prime 0.016, transactions 0.015).
- **Between features:** all pairs are within ±0.03. In real customer data, income would normally move with spend and tenure with engagement. Here it does not.
- **Consequence:** the features carry no duplicated information, and customers do not form natural clusters. Segments must therefore be built from business logic (value, engagement, cashback behaviour), not discovered from the data.

### 7. Can a model predict churn from the raw columns?
- Logistic regression (straight-line relationships) scored an AUC of 0.491, and gradient boosting (curves and interactions) scored 0.468. Shuffled-label controls scored 0.469 and 0.490.
- Fold-to-fold variation is about ±0.017, so all of these are consistent with a coin flip (AUC 0.5).
- A retention team using these columns to pick whom to call would do no better than calling at random.

### 8. What this means for the five business questions
| Question | What the EDA tells us | Next step |
|---|---|---|
| **Q1: What drives churn?** | No strong driver in the available attributes | Report this honestly. Re-test with engineered features (Stage 1 association tests); investigate joint patterns (e.g. copula) only if something appears |
| **Q2: Prime effect on frequency and loyalty** | Prime churn is not lower (19.7% vs 18.4%) | Compare transactions, value and tenure for matched Prime and non-Prime customers, with confidence intervals and cashback cost (Prime earns 5% vs 3% on Amazon.in) |
| **Q3: Highest-lifetime-value segments** | Value will be driven by spend and tenure, since churn is flat | Build a value proxy (monthly spend x take rate x expected lifetime) and rank segments |
| **Q4: Cashback: real engagement or deal-seeking?** | Cashback use shows no link with churn | Use the cashback x engagement quadrants (real engager, deal-seeker, organic loyalist, passive) and size the cashback cost per quadrant |
| **Q5: Personalised offers to improve retention** | No churn signal to target on, so targeting must rest on behaviour and value | Segment-level playbook, cashback reallocation with rupee impact, and an A/B test design to prove it before rollout |

### 9. Limitations to state in the report
- **Snapshot only:** no time dimension, no churn date and no behaviour trend, so we cannot see customers drifting toward churn.
- **No spend-by-category** (Amazon.in vs partner merchants vs other), so cashback cost relies on an assumed spend mix.
- **No offer-exposure data**, so we cannot measure the true effect of any past offer, only propose a test.
- **No revenue or margin data**, so lifetime value uses an assumed take rate.
- **Evenly spread, uncorrelated features** suggest synthetic data, which limits how far findings can be generalised.
- **The "Other" gender group is only 97 customers**, too small for reliable conclusions.

### 10. Bridge to feature engineering
Because no raw attribute explains churn, the next step is to combine columns into business-meaningful features: payment intensity, payment style, lifecycle stage, friction rate, engagement index, cashback segment, NPS group and say-do gap. We will test whether any of them separate churners from active customers, and use them mainly to answer Q2 to Q5.

In [10]:
# ===== CELL 7: FEATURE ENGINEERING =====
# BIZ: raw columns describe a customer; engineered features describe BEHAVIOUR
# (how much they use us, how engaged they are, what rewards they chase).
# TECH: no feature below uses `churned`. Building features from the target would leak
# the answer into the inputs and fake a great model.

# All business assumptions live in ONE dictionary, so changing one updates everything
# and the assumptions are easy to state on a slide.
CFG = {
    "tenure_bins":   [0, 12, 36, np.inf],                          # months
    "tenure_labels": ["New (<1y)", "Growing (1-3y)", "Mature (3y+)"],
    "age_bins":      [17, 25, 35, 50, 64],
    "age_labels":    ["18-25", "26-35", "36-50", "51-64"],
    "nps_detractor": -33,     # below this = unhappy (judgment call, say so in the deck)
    "nps_promoter":   33,     # above this = happy
    # Cashback rates from the published Amazon Pay ICICI card table (ASSUMPTION: we apply
    # them to all customers; wallet/UPI cashback may differ)
    "rate_amazon":  {"prime": 0.05, "nonprime": 0.03},   # eligible Amazon.in orders only (per TapNow guide, Oct 2026)
    "rate_partner": 0.01,                                # partner merchants outside Amazon: 1% (was 2%)
    "eligible_share_amazon": 0.85,                       # ASSUMPTION: 85% of Amazon.in spend is eligible (EMI, wallet loads etc. excluded)
    "rate_other":   0.01,
    # ASSUMPTION: the data has no spend-by-category, so we assume a spend mix (must sum to 1)
    "mix_amazon": 0.40, "mix_partner": 0.30, "mix_other": 0.30,
}

fe = df.copy()    # `df` stays clean; `fe` is the feature table

# --- 1. Value features ---
# BIZ: monthly spend = money moved through Amazon Pay each month (the base of customer value)
fe["monthly_spend"] = fe.avg_monthly_transactions * fe.avg_transaction_value_inr
# BIZ: payment intensity = spend relative to income (how central we are to their payments).
# Spend can exceed income (credit cards, family or business use), so it is a RANKING, not a %.
fe["payment_intensity"] = fe.monthly_spend / fe.monthly_income_inr
# TECH: rank(pct=True) converts values to 0-1 positions (0.9 = higher than 90% of customers),
# which tames extreme values.
fe["payment_intensity_rank"] = fe.payment_intensity.rank(pct=True)

# --- 2. Payment style (frequency x ticket size) ---
# TECH: split each at its MEDIAN to get 4 equal-sized groups; np.select is a multi-way if/else
# applied to every row at once.
# BIZ: daily small payers (bills, recharges) and occasional big shoppers need different offers.
hi_freq   = fe.avg_monthly_transactions  >= fe.avg_monthly_transactions.median()
hi_ticket = fe.avg_transaction_value_inr >= fe.avg_transaction_value_inr.median()
fe["payment_style"] = np.select(
    [hi_freq & hi_ticket, hi_freq & ~hi_ticket, ~hi_freq & hi_ticket],
    ["Power user", "Habitual small payer", "Big-ticket occasional"], default="Light user")

# --- 3. Lifecycle and life stage ---
# TECH: pd.cut turns a number into labelled bands using the bin edges in CFG.
# BIZ: a 4-month customer needs onboarding; a 5-year customer needs recognition.
fe["lifecycle_stage"] = pd.cut(fe.tenure_months, CFG["tenure_bins"], labels=CFG["tenure_labels"])
fe["age_band"]        = pd.cut(fe.age, CFG["age_bins"], labels=CFG["age_labels"])

# --- 4. Friction ---
# BIZ: 8 support contacts in 6 months = struggling; over 5 years = normal. So measure per year.
# CAUTION: very new customers inflate the value (max ~108), so the feature is right-skewed.
fe["friction_per_year"] = fe.customer_support_contacts / (fe.tenure_months / 12)

# --- 5. Engagement and cashback behaviour (Q4) ---
# TECH: an INDEX blends two measures; averaging their 0-1 ranks puts both on one scale.
# BIZ: real engagement = opens the app AND transacts.
fe["engagement_index"] = (fe.app_engagement_score.rank(pct=True)
                          + fe.avg_monthly_transactions.rank(pct=True)) / 2
hi_cb  = fe.cashback_used_pct >= fe.cashback_used_pct.median()   # redeems a lot of cashback
hi_eng = fe.engagement_index  >= 0.5                             # above-median engagement
# BIZ: the 2x2 for Q4: does cashback go with real engagement, or only deal-hunting?
fe["cashback_segment"] = np.select(
    [hi_cb & hi_eng, hi_cb & ~hi_eng, ~hi_cb & hi_eng],
    ["Real engager", "Deal-seeker", "Organic loyalist"], default="Passive")
# BIZ: cashback_used_pct = share of EARNED cashback redeemed, so it measures reward sensitivity.
# reward_pull scales it by activity: heavy users who chase rewards score high.
fe["reward_pull"] = fe.cashback_used_pct / 100 * fe.avg_monthly_transactions

# --- 6. Cashback cost ---
# BIZ: blended rate = sum of (share of spend x cashback rate on that spend).
# Prime earns 5% on Amazon.in spend, non-Prime 3%, so Prime costs more per rupee spent.
rate_prime    = (CFG["mix_amazon"]*CFG["eligible_share_amazon"]*CFG["rate_amazon"]["prime"]    + CFG["mix_partner"]*CFG["rate_partner"] + CFG["mix_other"]*CFG["rate_other"])
rate_nonprime = (CFG["mix_amazon"]*CFG["eligible_share_amazon"]*CFG["rate_amazon"]["nonprime"] + CFG["mix_partner"]*CFG["rate_partner"] + CFG["mix_other"]*CFG["rate_other"])
print(f"Blended cashback rate -> Prime: {rate_prime:.1%} | Non-Prime: {rate_nonprime:.1%}")
fe["cashback_rate"] = np.where(fe.amazon_prime_member == 1, rate_prime, rate_nonprime)
# Monthly cost per customer = spend x rate x share of cashback redeemed.
# CAUTION: spend is inflated for many customers (Cell 1), so use this cost to COMPARE segments,
# not as an absolute rupee budget.
fe["est_cashback_cost"] = fe.monthly_spend * fe.cashback_rate * fe.cashback_used_pct / 100

# --- 7. Satisfaction ---
# BIZ: three groups are actionable; a raw -100..100 score is not.
fe["nps_group"] = pd.cut(fe.net_promoter_score,
                         [-101, CFG["nps_detractor"], CFG["nps_promoter"], 101],
                         labels=["Detractor", "Passive", "Promoter"])
# BIZ: "say vs do" = what customers SAY (NPS) vs what they DO (engagement).
# Happy-but-idle -> nudge them. Active-but-unhappy -> hidden risk, fix the experience.
fe["say_do_gap"] = np.select(
    [(fe.nps_group == "Promoter") & ~hi_eng, (fe.nps_group == "Detractor") & hi_eng],
    ["Happy but idle", "Active but unhappy"], default="Aligned")

# --- Checks ---
NEW_NUM = ["monthly_spend", "payment_intensity", "payment_intensity_rank", "friction_per_year",
           "engagement_index", "reward_pull", "cashback_rate", "est_cashback_cost"]
print(fe[NEW_NUM].describe().T.round(3))
print("\nShape:", fe.shape, "| Nulls:", int(fe.isna().sum().sum()))
print("Churn column untouched:", fe.churned.equals(df.churned))
print("Avg monthly cashback cost by Prime (0/1):", fe.groupby("amazon_prime_member").est_cashback_cost.mean().round(0).to_dict())


Blended cashback rate -> Prime: 2.3% | Non-Prime: 1.6%
                         count        mean         std      min         25%         50%         75%          max
monthly_spend           5000.0  376046.867  324285.220  420.000  106142.500  282964.500  568955.250  1445449.000
payment_intensity       5000.0       5.411       7.113    0.003       1.136       3.082       6.640       71.993
payment_intensity_rank  5000.0       0.500       0.289    0.000       0.250       0.500       0.750        1.000
friction_per_year       5000.0       3.667       9.094    0.000       0.676       1.500       2.880      108.000
engagement_index        5000.0       0.500       0.206    0.027       0.353       0.500       0.650        0.974
reward_pull             5000.0      37.291      33.147    0.000       9.660      28.055      56.952      149.000
cashback_rate           5000.0       0.019       0.003    0.016       0.016       0.016       0.023        0.023
est_cashback_cost       5000.0    3577.89

### Feature engineering

**Procedure**
We combined the raw columns into 14 business-meaningful features, with every assumption kept in one configuration block so it is easy to change and to state in the report:
- **Value:** monthly spend (transactions x average value) and payment intensity (spend relative to income, kept as a ranking).
- **Behaviour:** payment style (frequent small payers vs occasional big shoppers), lifecycle stage (new, growing, mature), age band, friction per year (support contacts relative to tenure) and an engagement index (app engagement + transactions).
- **Cashback:** a cashback segment (real engager, deal-seeker, organic loyalist, passive), reward pull (reward sensitivity x activity), and an estimated monthly cashback cost per customer. The cost uses the Amazon Pay ICICI card rates as summarised in the TapNow guide (5% Prime / 3% non-Prime on eligible Amazon.in orders, 1% on partner merchants, 1% elsewhere), an assumed spend mix of 40% / 30% / 30% and an assumed 85% of Amazon.in spend being eligible.
- **Satisfaction:** NPS group (detractor, passive, promoter) and a "say vs do" gap that compares satisfaction with activity.
No feature uses the churn column, so nothing leaks the answer into the inputs.

**Observations**
- **Build is clean:** 14 new columns (28 in total), no missing values, and the churn column is unchanged.
- **Prime cashback costs more:** the blended cashback rate is 2.3% for Prime vs 1.6% for non-Prime (about 42% higher per rupee spent). The average estimated monthly cashback cost is ₹4,214 for Prime vs ₹3,043 for non-Prime (about 38% higher per customer, a little less than the rate gap because Prime members spend 4% less). Almost all of the gap comes from the higher Prime rate, not from Prime members spending or redeeming more. This sets up the Prime question (Q2): is the extra cost buying anything?
- **Engineered features are skewed even though raw ones were flat.** Multiplying two evenly spread features creates a tail: monthly spend has a median of ₹2.8 lakh vs a mean of ₹3.8 lakh and a maximum of ₹14.5 lakh. This gives us a real "high-value" group for Q3 even though the raw data had none.
- **Payment intensity (median 3.1, maximum 72) is only usable as a ranking,** as the spend-above-income caveat explained.
- **Friction per year is skewed** (median 1.5, mean 3.7, maximum 108) because very new customers with a few support contacts look extreme when scaled to a year. We read it as a ranking, not an exact rate.
- **Cashback cost is a comparison tool, not a budget.** Because spend is inflated for many customers, absolute rupee amounts (average ₹3,578 per customer per month) are overstated. We use cost to compare segments and rank where cashback goes, not to claim a rupee saving.
- **Assumptions to flag:** the cashback rates come from a published summary of the ICICI card (not the issuer's terms), and the 40/30/30 spend mix and the 85% eligible share are assumptions. Cell 7A stress-tests them.

In [11]:
# ===== CELL 7A: STRESS TEST OF THE CASHBACK ASSUMPTIONS =====
from IPython.display import display

# TECH: A sensitivity analysis re-runs one calculation under several assumption sets and checks which
#       conclusions survive. A conclusion that holds in every scenario is "robust".
# BIZ:  We do not know the true rate card, the spend mix or the eligible share, so we show how far the
#       headline numbers move across plausible cases.
HEAVY = fe.cashback_segment.isin(["Real engager", "Deal-seeker"])   # heavy redeemers = real engagers + deal-seekers
CUT = 0.25                                                          # the 25% cashback cut used in Cell 19

def run_scenario(mix_amazon, eligible, mix_partner, mix_other, r_prime=0.05, r_non=0.03, r_partner=0.01, r_other=0.01):
    # blended rate = share of spend in each bucket x rate of that bucket.
    # Amazon.in spend earns the 5% / 3% rate only on its ELIGIBLE share; the rest earns nothing.
    rate_p = mix_amazon * eligible * r_prime + mix_partner * r_partner + mix_other * r_other
    rate_n = mix_amazon * eligible * r_non   + mix_partner * r_partner + mix_other * r_other
    rate   = np.where(fe.amazon_prime_member == 1, rate_p, rate_n)
    earned   = fe.monthly_spend * rate                       # cashback EARNED (accrued, a liability)
    redeemed = earned * fe.cashback_used_pct / 100           # cashback actually REDEEMED (cash-out cost)
    cp, cn = redeemed[fe.amazon_prime_member == 1].mean(), redeemed[fe.amazon_prime_member == 0].mean()
    share_red, share_earn = redeemed[HEAVY].sum() / redeemed.sum(), earned[HEAVY].sum() / earned.sum()
    return {"Prime rate": f"{rate_p:.1%}", "Non-Prime rate": f"{rate_n:.1%}",
            "Prime costs more by": f"{cp / cn - 1:+.0%}",
            "Heavy redeemers: cost per Rs100": round(redeemed[HEAVY].sum() / fe.monthly_spend[HEAVY].sum() * 100, 2),
            "Their share of redeemed bill": f"{share_red:.0%}", "Their share of earned cashback": f"{share_earn:.0%}",
            "25% cut saves (redeemed view)": f"{CUT * share_red:.0%}", "25% cut saves (earned view)": f"{CUT * share_earn:.0%}"}

SCENARIOS = {
    "Old assumption (partner 2%)":            run_scenario(0.40, 1.00, 0.30, 0.30, r_partner=0.02),
    "BASE: guide rates, 85% eligible":        run_scenario(CFG["mix_amazon"], CFG["eligible_share_amazon"], CFG["mix_partner"], CFG["mix_other"]),
    "Guide rates, all Amazon spend eligible": run_scenario(0.40, 1.00, 0.30, 0.30),
    "Conservative: 30% Amazon, 70% eligible": run_scenario(0.30, 0.70, 0.35, 0.35),
    "Generous: 60% Amazon, all eligible":     run_scenario(0.60, 1.00, 0.20, 0.20),
}
# TECH: flipping the table (metrics as rows, scenarios as columns) gives 5 narrow columns instead of
#       8 wide ones, so it fits the screen. display() draws a normal HTML table in Jupyter.
from IPython.display import display
sens = pd.DataFrame(SCENARIOS)                                   # rows = metrics, columns = scenarios
sens.columns = ["Old (partner 2%)", "BASE", "All eligible", "Conservative", "Generous"]
display(sens)

print("Scenario key:")
print("  Old           = partner merchants at 2%, all Amazon spend eligible")
print("  BASE          = guide rates, 85% of Amazon.in spend eligible (the CFG setting)")
print("  All eligible  = guide rates, 100% of Amazon.in spend eligible")
print("  Conservative  = 30% of spend on Amazon.in, 70% of it eligible")
print("  Generous      = 60% of spend on Amazon.in, all eligible")

,Old (partner 2%),BASE,All eligible,Conservative,Generous
Prime rate,2.9%,2.3%,2.6%,1.8%,3.4%
Non-Prime rate,2.1%,1.6%,1.8%,1.3%,2.2%
Prime costs more by,+35%,+38%,+41%,+28%,+51%
Heavy redeemers: cost per Rs100,1.83,1.43,1.6,1.13,2.04
Their share of redeemed bill,76%,76%,76%,76%,76%
Their share of earned cashback,51%,51%,51%,51%,51%
25% cut saves (redeemed view),19%,19%,19%,19%,19%
25% cut saves (earned view),13%,13%,13%,13%,13%


Scenario key:
  Old           = partner merchants at 2%, all Amazon spend eligible
  BASE          = guide rates, 85% of Amazon.in spend eligible (the CFG setting)
  All eligible  = guide rates, 100% of Amazon.in spend eligible
  Conservative  = 30% of spend on Amazon.in, 70% of it eligible
  Generous      = 60% of spend on Amazon.in, all eligible


### Procedure 
1. Re-ran the cashback cost calculation under five assumption sets: the old assumption (partner merchants at 2%), the base case from the TapNow guide (partner merchants 1%, 85% of Amazon.in spend eligible), all Amazon spend eligible, a conservative case (30% of spend on Amazon.in, 70% of it eligible) and a generous case (60% on Amazon.in, all eligible).
2. For each, measured the blended Prime and Non-Prime rates, how much more Prime customers cost, heavy redeemers' cost per ₹100, and their share of the bill.
3. Showed heavy redeemers' share in two views: redeemed cashback (what is paid out) and earned cashback (what has been accrued, since points do not expire).

### Observations 
- **Base case: blended rates of 2.3% (Prime) and 1.6% (Non-Prime).** The old assumption gave 2.9% and 2.1%, so the corrected partner rate and the eligibility limit lower every cost figure.
- **The absolute level moves a lot, but the shares do not.** Heavy redeemers' cost per ₹100 ranges from ₹1.13 to ₹2.04 across scenarios, yet they take 76% of the redeemed bill (51% of earned cashback) and a 25% cut saves 19% (13%) in every scenario. All segments are paid on the same rate card, so the rate card cannot change who the cost falls on.
- **Prime costs more in every scenario:** +28% (conservative) to +51% (generous), +38% in the base case. The conclusion "Prime is more expensive to serve" does not depend on the assumptions.
- **Two views of the saving.** Counting only what is redeemed now, a 25% cut on heavy redeemers saves 19% of the bill. Counting everything earned (points do not expire, so unredeemed points are still a liability), it saves 13%. We report both as a range of 13% to 19%.
- **Business reading:** the case for re-tiering cashback is robust to the rate-card uncertainty. Only the rupee size of the saving depends on the true rates, which we do not have.

In [12]:
# ===== CELL 8: DO THE NEW SEGMENTS SEPARATE CHURNERS? =====
from scipy.stats import chi2_contingency
import textwrap

# Fixed display order (low to high) so charts read the same way every time.
ORDER = {
    "payment_style":    ["Light user", "Habitual small payer", "Big-ticket occasional", "Power user"],
    "lifecycle_stage":  CFG["tenure_labels"],
    "age_band":         CFG["age_labels"],
    "cashback_segment": ["Passive", "Organic loyalist", "Deal-seeker", "Real engager"],
    "nps_group":        ["Detractor", "Passive", "Promoter"],
    "say_do_gap":       ["Aligned", "Happy but idle", "Active but unhappy"],
}

# Same method as Cell 3: churn rate per segment, 95% confidence interval, chi-square p-value.
# (Uses wilson() and BASE from Cell 3.)
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
rows = []
for ax, (col, order) in zip(axes.ravel(), ORDER.items()):
    g = fe.groupby(col, observed=True).churned.agg(["sum", "count"]).reindex(order)
    g["rate"] = g["sum"] / g["count"]
    g[["lo", "hi"]] = [wilson(k, n) for k, n in zip(g["sum"], g["count"])]
    x = np.arange(len(g))
    ax.bar(x, g.rate * 100, color=C_CHURN, width=0.55)
    ax.errorbar(x, g.rate * 100, yerr=[(g.rate - g.lo) * 100, (g.hi - g.rate) * 100],
                fmt="none", ecolor="#333", capsize=4, lw=1.2)
    ax.axhline(BASE * 100, color=C_ACTIVE, ls="--", lw=2)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{textwrap.fill(i, 12)}\n(n={n:,})" for i, n in zip(g.index, g["count"])], fontsize=9)
    ax.set_title(col.replace("_", " ").capitalize(), weight="bold"); ax.set_ylim(0, 35)
    p = chi2_contingency(pd.crosstab(fe[col], fe.churned))[1]
    ax.text(0.97, 0.94, f"p = {p:.2f}", transform=ax.transAxes, ha="right", fontsize=10)
    rows.append({"feature": col, "p_value": round(p, 3),
                 "max_gap_pts": round((g.rate.max() - g.rate.min()) * 100, 1)})
for ax in axes[:, 0]: ax.set_ylabel("Churn rate (%)")
fig.suptitle("Churn rate by engineered segment (whiskers = 95% CI, dashed = 19% overall)",
             fontsize=14, weight="bold")
plt.tight_layout(); plt.show()

# TECH: MULTIPLE TESTING. Testing 6 features at p < 0.05 gives a ~26% chance of at least one
# false alarm by luck alone. Bonferroni correction divides the threshold by the number of tests
# (0.05 / 6 = 0.0083), so a finding must be much stronger to count.
res = pd.DataFrame(rows)
alpha_adj = 0.05 / len(res)
res["significant_after_correction"] = res.p_value < alpha_adj
print(res.to_string(index=False))
print(f"\nCorrected threshold: {alpha_adj:.4f}")

         feature  p_value  max_gap_pts  significant_after_correction
   payment_style    0.323          2.7                         False
 lifecycle_stage    0.891          0.6                         False
        age_band    0.988          0.5                         False
cashback_segment    0.537          2.2                         False
       nps_group    0.969          0.3                         False
      say_do_gap    0.837          0.8                         False

Corrected threshold: 0.0083


### Do the engineered segments separate churners?

**Procedure**
For each of the six engineered segment features, we plotted the churn rate per segment with 95% confidence intervals against the 19% baseline and ran a chi-square test. Because we test six features at once, a "significant" result can appear by luck, so we applied a Bonferroni correction: the threshold drops from 0.05 to 0.0083 (0.05 / 6), so a result must be much stronger to count.

**Observations**
- **None of the six features is significant:** p-values range from 0.32 to 0.99, and none gets near the corrected 0.0083 threshold.
- **The largest gaps are small:** payment style 2.7 points (power users 20.6% vs big-ticket occasional shoppers 17.9%) and cashback segment 2.2 points (organic loyalists 20.4% vs real engagers 18.2%). Lifecycle stage, age band, NPS group and the say-do gap differ by 0.3 to 0.8 points. Every confidence interval overlaps the 19% line.
- **New customers do not churn more:** new (<1 year) is 18.7% vs mature (3+ years) at 19.3%, so the data shows no early-life drop-off problem.
- **Deal-seekers do not churn more** than real engagers (18.7% vs 18.2%). Cashback redemption is not associated with loyalty in either direction. This matters for Q4: the data gives no evidence that cashback is buying loyalty from deal-seekers, so the case for reallocating it rests on cost efficiency, not on a proven retention effect.
- **Satisfaction does not predict churn:** detractors, passives and promoters all churn at about 19%. The "happy but idle" and "active but unhappy" groups (about 17% of customers each) look no different from "aligned" customers.
- **The segments are well sized and usable:** about 25% per payment-style and cashback group, thirds for NPS groups, and 17% / 17% / 66% for the say-do gap, so no segment is too small for analysis.
- **Business meaning:** the segments are not "retention-risk" groups, since all churn at about 19%. Their value for this case lies in customer value, engagement and cashback efficiency (Q2 to Q5), not in predicting who leaves.

In [13]:
# ===== CELL 8A: DOES SERVICE FRICTION EXPLAIN CHURN? =====
from scipy.stats import spearmanr, chi2_contingency
C_PRIME, C_NON, C_COST = "#2B7BBF", "#7F7F7F", "#D2591A"      # fixed colour per meaning (re-declared for safety)

# 1) Is friction_per_year a fair measure? Compare raw contacts with the per-year version across tenure.
# TECH: dividing a count by tenure builds in a mechanical link: a 1-month customer with 4 contacts becomes
#       "48 per year". We check whether that is a real pattern or an artefact of the division.
# BIZ:  A new customer with a few contacts is not necessarily more frustrated than a long-term one.
fe["tenure_band"] = pd.qcut(fe.tenure_months, q=5, labels=["Newest 20%", "20-40%", "40-60%", "60-80%", "Longest 20%"])
by_tenure = fe.groupby("tenure_band", observed=True).agg(avg_contacts=("customer_support_contacts", "mean"),
                                                          avg_friction_per_year=("friction_per_year", "mean"))
print(by_tenure.round(2).to_string())
r_c, _ = spearmanr(fe.tenure_months, fe.customer_support_contacts)
r_f, _ = spearmanr(fe.tenure_months, fe.friction_per_year)
print(f"\nCorrelation with tenure: raw contacts {r_c:+.2f} | contacts per year {r_f:+.2f}")

# 2) Does friction relate to churn, satisfaction or activity? Test both versions of the measure.
rows = []
for col in ["customer_support_contacts", "friction_per_year"]:
    for label, tgt in {"Churned": "churned", "NPS": "net_promoter_score", "Engagement score": "app_engagement_score",
                       "Transactions / month": "avg_monthly_transactions"}.items():
        r, p = spearmanr(fe[col], fe[tgt])
        rows.append({"measure": col, "against": label, "spearman_rho": round(r, 3), "p_value": round(p, 3)})
print(pd.DataFrame(rows).to_string(index=False))

# 3) Churn rate by number of support contacts (95% Wilson intervals)
fe["contact_group"] = pd.cut(fe.customer_support_contacts, [-1, 0, 1, 2, 3, 4, 5, 99], labels=["0", "1", "2", "3", "4", "5", "6+"])
grp = fe.groupby("contact_group", observed=True).churned.agg(["sum", "size"])
print(f"\nChurn vs number of support contacts: chi-square p = {chi2_contingency(pd.crosstab(fe.contact_group, fe.churned))[1]:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
ax = axes[0]
ax.plot(range(5), by_tenure.avg_contacts, "o-", color=C_PRIME, lw=2.5, ms=7)
ax.set_ylim(0, 6); ax.set_xticks(range(5)); ax.set_xticklabels(by_tenure.index, fontsize=8.5)
ax.set_title("Raw support contacts: flat across tenure", weight="bold", fontsize=11); ax.set_ylabel("Average contacts")
ax = axes[1]
ax.plot(range(5), by_tenure.avg_friction_per_year, "o-", color=C_COST, lw=2.5, ms=7)
ax.set_ylim(0, 14); ax.set_xticks(range(5)); ax.set_xticklabels(by_tenure.index, fontsize=8.5)
ax.set_title("Contacts per year: falls with tenure by construction", weight="bold", fontsize=11); ax.set_ylabel("Average contacts per year")
ax = axes[2]
for i, (g_, row) in enumerate(grp.iterrows()):
    m = row["sum"] / row["size"]; lo, hi = wilson(row["sum"], row["size"])
    ax.errorbar(i, m, yerr=[[m - lo], [hi - m]], fmt="o", color=C_PRIME, capsize=4, ms=7, lw=2)
ax.axhline(BASE, color="black", ls="--", lw=1); ax.set_ylim(0.10, 0.30)
ax.set_xticks(range(len(grp))); ax.set_xticklabels(grp.index); ax.set_xlabel("Support contacts (all time)")
ax.set_title("Churn by support contacts (95% CI)", weight="bold", fontsize=11); ax.set_ylabel("Churn rate")
plt.tight_layout(); plt.show()

             avg_contacts  avg_friction_per_year
tenure_band                                     
Newest 20%           4.58                  12.20
20-40%               4.49                   2.40
40-60%               4.54                   1.46
60-80%               4.54                   1.06
Longest 20%          4.54                   0.84

Correlation with tenure: raw contacts +0.00 | contacts per year -0.59
                  measure              against  spearman_rho  p_value
customer_support_contacts              Churned        -0.001    0.925
customer_support_contacts                  NPS         0.017    0.229
customer_support_contacts     Engagement score        -0.031    0.030
customer_support_contacts Transactions / month        -0.030    0.034
        friction_per_year              Churned        -0.007    0.599
        friction_per_year                  NPS         0.007    0.632
        friction_per_year     Engagement score        -0.029    0.044
        friction_per_year 

### Procedure 
1. Tested whether `friction_per_year` (support contacts divided by years of tenure) is a fair measure of service friction, by comparing it with raw support contacts across five tenure bands.
2. Tested both versions against churn, NPS, engagement score and transactions per month (Spearman correlation).
3. Grouped customers by number of support contacts (0 to 6+) and compared churn rates with 95% confidence intervals and a chi-square test.

### Observations 
- **`friction_per_year` is distorted for new customers.** Raw support contacts are flat across tenure (about 4.5 for every band, correlation +0.00), but contacts per year fall from 12.2 for the newest 20% to 0.84 for the longest-tenured (correlation -0.59). Dividing a fixed count by a short tenure inflates the rate. A new customer with a few contacts is not necessarily more frustrated than an older one. Raw contacts are the cleaner measure in this data.
- **Service friction does not predict churn.** Raw contacts vs churn: rho -0.001 (p = 0.93). Contacts per year vs churn: rho -0.007 (p = 0.60). Churn stays near the overall 19% in every contact group (0 to 6+), with all intervals overlapping the average line (chi-square p = 0.94).
- **Friction does not predict satisfaction either.** NPS vs contacts: rho +0.017 and +0.007 (both p > 0.2). Customers who contact support more are not less happy, so support volume is not a signal of unhappiness.
- **Two tiny links exist but are not usable.** More contacts go with slightly lower engagement and fewer transactions (rho -0.03 to -0.04, p between 0.004 and 0.04). The smallest (contacts per year vs transactions, p = 0.004) is even below the corrected threshold of 0.006 (0.05 / 8), so it is statistically detectable, but at rho -0.04 it explains well under 1% of the variation. We treat these as too small to act on, not as exactly zero.
- **Business reading:** support logs are not an early-warning system for churn here. This is one more independent lens that finds no churn driver (Q1 stays "no strong driver"), and unhappy customers will need to be found through NPS surveys, not through support logs.

In [14]:
# ===== CELL 9: CLOSING Q1, DO RAW + ENGINEERED FEATURES TOGETHER BEAT CHANCE? =====
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
try:
    from sklearn.experimental import enable_hist_gradient_boosting  # noqa: F401
    from sklearn.ensemble import HistGradientBoostingClassifier
    USE_HIST = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    USE_HIST = False
print("Fast boosting available:", USE_HIST)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif

# Model input = ALL raw + engineered columns, with text categories one-hot encoded
# (0/1 columns). customer_id (a label) and churned (the target) are excluded.
# Helper columns that later cells add to `fe` (tenure_band, contact_group, ...) are excluded on purpose, so the
# model input is the same 37 columns no matter which cells were run before this one.
MODEL_DROP = ["customer_id", "churned", "tenure_band", "contact_group", "value_projected", "value_index",
              "volume_to_date", "prime_label", "cashback_band", "pull_band"]
X = pd.get_dummies(fe.drop(columns=MODEL_DROP, errors="ignore"), drop_first=True).astype(float)
y = fe.churned.values
print("Model input:", X.shape[1], "features")

# --- PART A: model accuracy vs a "pure luck" benchmark ---
# TECH: cross-validation = train on 4/5 of customers, test on the unseen 1/5. Repeating the whole
# thing 3 times with different splits (RepeatedStratifiedKFold) makes the score more stable.
# Fallback model is slower, so we use 1 repeat of 5-fold CV instead of 3 to keep runtime sensible.
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3 if USE_HIST else 1, random_state=SEED)

models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Gradient boosting": (HistGradientBoostingClassifier(max_depth=3, max_iter=100,
                                                         learning_rate=0.05, random_state=SEED)
                          if USE_HIST else
                          GradientBoostingClassifier(n_estimators=60, max_depth=3,
                                                     learning_rate=0.1, random_state=SEED)),
}
# TECH: PERMUTATION TEST. Shuffle the churn labels so they carry no information, re-run the SAME
# pipeline, and record the AUC. Do this 30 times to see what "pure luck" scores. If our real AUC
# is not above most shuffled scores, there is no evidence of signal.
# p-value = share of shuffled runs that scored at least as well as the real one (+1 correction).
# BIZ: this gives a clear, explainable statement: "our model did no better than random."
N_PERM = 30 if USE_HIST else 20
rng = np.random.default_rng(SEED)
rows = []
for name, m in models.items():
    real = cross_val_score(m, X, y, cv=cv, scoring="roc_auc").mean()
    null = np.array([cross_val_score(m, X, rng.permutation(y), cv=cv, scoring="roc_auc").mean()
                     for _ in range(N_PERM)])
    p = (np.sum(null >= real) + 1) / (N_PERM + 1)
    rows.append({"model": name, "AUC_real": round(real, 3), "null_mean": round(null.mean(), 3),
                 "null_95th": round(np.quantile(null, .95), 3), "p_value": round(p, 3)})
auc_res = pd.DataFrame(rows)
print(auc_res.to_string(index=False))

# --- PART B: feature-by-feature check for ANY kind of relationship ---
# TECH: mutual information (MI) measures how much knowing a feature reduces uncertainty about
# churn. Unlike correlation, it also catches curved and irregular relationships. 0 = independent.
# discrete_features tells the estimator which columns have few distinct values (<= 12).
discrete = (X.nunique() <= 12).values
mi_real = pd.Series(mutual_info_classif(X, y, discrete_features=discrete, random_state=SEED),
                    index=X.columns)

# TECH: with 37 features, the BEST one will always look somewhat good by luck (multiple testing).
# So we shuffle the labels 50 times, record the best MI across ALL features each time, and take
# the 95th percentile of those "best by luck" scores as the bar. A real feature must clear it.
N_MI = 50
null_max = np.array([mutual_info_classif(X, rng.permutation(y), discrete_features=discrete,
                                         random_state=SEED).max() for _ in range(N_MI)])
thr = np.quantile(null_max, .95)
top = mi_real.sort_values(ascending=False).head(10)
print("\nTop 10 features by mutual information:\n", top.round(4).to_string())
print(f"\n'Best by luck' threshold: {thr:.4f} | Features above it: {int((mi_real > thr).sum())}")

# --- Chart: left = model vs chance, right = top features vs the luck line ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
a = auc_res.set_index("model"); xx = np.arange(len(a))
ax.bar(xx - .18, a.AUC_real, .35, color=C_CHURN, label="Real labels")
ax.bar(xx + .18, a.null_95th, .35, color="#999999", label="Shuffled labels (95th percentile)")
ax.axhline(.5, color="black", ls="--", lw=1)
ax.set_xticks(xx); ax.set_xticklabels(a.index); ax.set_ylim(.4, .7)
ax.set_ylabel("AUC (0.5 = coin flip)"); ax.legend(loc="upper right")
ax.set_title("Model accuracy vs chance", weight="bold")
ax = axes[1]
t = top[::-1]
ax.barh(t.index, t.values, color=C_CHURN); ax.axvline(thr, color=C_ACTIVE, ls="--", lw=2)
ax.set_xlabel("Mutual information with churn (dashed = best-by-luck line)")
ax.set_title("Top 10 features vs the 'best by luck' line", weight="bold")
plt.tight_layout(); plt.show()

Fast boosting available: True
Model input: 37 features


              model  AUC_real  null_mean  null_95th  p_value
Logistic regression     0.488      0.498      0.517    0.806
  Gradient boosting     0.489      0.503      0.525    0.903



Top 10 features by mutual information:
 monthly_spend                0.0099
engagement_index             0.0077
age                          0.0058
est_cashback_cost            0.0047
avg_monthly_transactions     0.0029
reward_pull                  0.0010
customer_support_contacts    0.0006
gender_Other                 0.0005
payment_style_Power user     0.0003
city_tier_Tier 2             0.0003

'Best by luck' threshold: 0.0140 | Features above it: 0


### Closing Q1: Do raw and engineered features together beat chance?

**Procedure**
We fed all 37 inputs (raw columns plus engineered features) into two models: logistic regression (straight-line relationships) and gradient boosting (curves and interactions). Each was scored with repeated 5-fold cross-validation (train on 4/5 of customers, test on the unseen 1/5). As a benchmark we ran a **permutation test**: shuffle the churn labels so they carry no information, re-run the same pipeline 30 times, and see what "pure luck" scores. The p-value is the share of shuffled runs that scored at least as well as the real one.

We then checked each feature on its own with **mutual information**, which measures how much a feature tells us about churn and also catches curved or irregular relationships (0 = no link). With 37 features the best one always looks somewhat good by luck, so we shuffled the labels 50 times, recorded the best feature score each time, and used the 95th percentile of those "best by luck" scores as the bar a real feature must clear.

**Observations**
- **Neither model beats chance.** Logistic regression scored an AUC of 0.488 and gradient boosting 0.489 (0.5 = coin flip). Pure luck averages 0.498 and 0.503, and its 95th percentile is 0.517 and 0.525, so both real scores sit inside the range of random results.
- **The p-values confirm it:** 0.81 and 0.90 (they shift by a few hundredths between runs). Roughly 8 to 9 in 10 shuffled runs did as well as or better than the real model.
- **A score slightly below 0.5 is not "worse than random".** When there is no signal, cross-validated scores scatter around 0.5 and often land a little under it.
- **No single feature clears the bar.** The best is monthly spend (mutual information 0.0099), followed by engagement index (0.0077), age (0.0058) and transactions (0.0030). The "best by luck" threshold is 0.0140, and none of the 37 features reaches it.
- **Engineered features add nothing over raw ones.** The two top engineered features (monthly spend and engagement index) are built from raw columns and still fall short.
- **Business meaning:** no combination of the available attributes, whether straight-line, curved or interacting, identifies who will churn. Using these attributes to target a retention campaign would perform no better than random targeting.

In [15]:
# ===== CELL 9A: HOW BIG A CHURN GAP COULD THIS DATA HAVE DETECTED? =====
from scipy.stats import norm
# BIZ: "we found no driver" is only convincing if the data COULD have shown one. This cell says how big a gap in
# churn a group would need before our tests would reliably catch it.
# TECH: minimum detectable effect (MDE) = the smallest gap that a test flags 80% of the time (power) while
# keeping false alarms at 5%. We compare a group of n customers with all the others. Because we ran 12 tests
# (9 numeric + 3 categorical features), we also show the stricter Bonferroni threshold 0.05 / 12.
p0, N = df.churned.mean(), len(df)
def mde_pts(n, alpha, power=0.80):
    z = norm.ppf(1 - alpha / 2) + norm.ppf(power)
    return z * np.sqrt(p0 * (1 - p0) * (1 / n + 1 / (N - n))) * 100

groups = {"Gender 'Other'": 97, "Tier 3 customers": 1271, "One fifth of customers": 1000,
          "Prime members": 2283, "Half of customers": 2500}
mde_tbl = pd.DataFrame({"group size": list(groups.values()),
                        "gap detectable (5% level), pts":   [round(mde_pts(n, 0.05), 1)      for n in groups.values()],
                        "gap detectable (corrected), pts":  [round(mde_pts(n, 0.05 / 12), 1) for n in groups.values()]},
                       index=list(groups))
print(mde_tbl.to_string())
print(f"\nBaseline churn {p0:.1%}. A fifth of customers would need to churn above {p0*100 + mde_pts(1000, 0.05/12):.1f}% "
      f"or below {p0*100 - mde_pts(1000, 0.05/12):.1f}% to be flagged after correction.")

                        group size  gap detectable (5% level), pts  gap detectable (corrected), pts
Gender 'Other'                  97                            11.3                             14.9
Tier 3 customers              1271                             3.6                              4.7
One fifth of customers        1000                             3.9                              5.1
Prime members                 2283                             3.1                              4.1
Half of customers             2500                             3.1                              4.1

Baseline churn 19.0%. A fifth of customers would need to churn above 24.1% or below 13.9% to be flagged after correction.


### Procedure
1. For a group of *n* customers compared with everyone else, we calculated the smallest churn gap our tests would catch 80% of the time (power), while keeping false alarms at 5%. This is the **minimum detectable effect**.
2. We repeated it with the stricter threshold we need after running 12 tests on the raw features (0.05 / 12).
3. We applied it to groups of the sizes we actually tested: gender "Other", Tier 3, a fifth of customers, Prime members and half of customers.

### Observations
- **A fifth of customers (1,000) would need to churn at 24% or more, or 14% or less, to be flagged after correction** (a gap of about 5 points, or 3.9 points before correction). Tier 3 (1,271) needs 4.7 points, Prime members (2,283) and half of customers need 4.1 points.
- **Gender "Other" (97 customers) would need a gap of 11 points (15 after correction).** The observed gap is 8.8 points, so it is below what these data can reliably detect. This is why it stays a watch-item.
- **What "no driver found" rules out:** any single group whose churn differs from average by about 4 to 5 points or more. Gaps of 1 to 3 points, which on 5,000 customers means 50 to 150 customers, cannot be excluded.
- **Business meaning:** the Q1 answer is "no large driver in this data", not "no driver". Any small effect has to be found with a larger sample or a proper experiment, which is why the playbook ends with an A/B test.

### Q1 Conclusion: What factors most strongly influence customer churn?

**Answer:** In the data provided, no attribute or combination of attributes reliably predicts churn.

**Evidence (all point the same way)**

| Test | Result |
|---|---|
| Churn by gender, city tier, Prime (chi-square) | p = 0.05 to 0.29; the one borderline result comes from 97 customers |
| Churn across 5 bands of each numeric feature | No trend; all bands overlap the 19% baseline |
| Churners vs active (two-sided Mann-Whitney) | All p-values 0.25 to 0.996 |
| Spearman correlation with churn | All within ±0.02 |
| Models on raw features (Cell 6) | AUC 0.49 and 0.47, same as shuffled labels |
| Six engineered segments (Bonferroni-corrected chi-square) | All p-values 0.32 to 0.99; none significant |
| Models on raw + engineered features (permutation test) | AUC 0.49 and 0.49; p = 0.81 and 0.90 |
| Mutual information vs the "best by luck" bar | 0 of 37 features above the bar |

**How to read this**
- **Large effects would have shown up.** With roughly 1,000 customers per segment, a churn gap of about 4 points (5 after correcting for 12 tests) would normally be visible, and Prime-sized groups need about 3 to 4 points (Cell 9A). Effects of 1 to 2 points cannot be ruled out.
- **"No evidence" is not "no drivers".** Likely real drivers are not in this dataset: failed or declined payments, app problems, competitor offers, a recent drop in activity, or the reason for leaving. The data is a single snapshot with no time dimension or churn date.
- **The data may also be synthetic,** with evenly spread, uncorrelated features and randomly assigned churn.

**Recommendations**
1. **Do not build a churn-targeting programme on these attributes.** It would perform no better than random targeting.
2. **Collect the data that could explain churn:** monthly transaction history (recency and trend), payment failures, app events, offer exposure, support-contact reasons, and churn date with the stated reason.
3. **Redirect effort to what the data supports:** customer value, engagement and cashback efficiency (Q2 to Q5).
4. **Prove any retention action with an experiment,** since there is no historical signal to rely on (the A/B test design is part of the Q5 playbook).

In [16]:
# ===== CELL 10: DOES PRIME CHANGE BEHAVIOUR? (Q2) =====
from scipy.stats import mannwhitneyu

# Fixed colours for this section: blue = Prime, grey = Non-Prime.
C_PRIME, C_NON = "#2B7BBF", "#7F7F7F"
prime = fe[fe.amazon_prime_member == 1]
non   = fe[fe.amazon_prime_member == 0]
print(f"Prime: {len(prime):,} customers | Non-Prime: {len(non):,} customers")

# BIZ: Q2 asks about "transaction frequency and loyalty", so we compare the groups on
# frequency (transactions), value (ticket size, spend), loyalty (tenure, engagement, NPS, churn),
# reward use (cashback redeemed) and friction (support contacts).
METRICS = {
    "Transactions / month":  "avg_monthly_transactions",
    "Avg transaction value": "avg_transaction_value_inr",
    "Monthly spend":         "monthly_spend",
    "Tenure (months)":       "tenure_months",
    "App engagement":        "app_engagement_score",
    "Cashback redeemed (%)": "cashback_used_pct",
    "Net promoter score":    "net_promoter_score",
    "Support contacts":      "customer_support_contacts",
    "Churned (0/1)":         "churned",
}

# TECH: Standardised Mean Difference (SMD) = (Prime mean - Non-Prime mean) / typical spread.
# It puts every metric on one scale (rupees, months and scores are not comparable otherwise).
# Rule of thumb: |SMD| < 0.1 negligible, 0.2 small, 0.5 medium, 0.8 large.
def smd(a, b):
    pooled_sd = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return (a.mean() - b.mean()) / pooled_sd

# TECH: bootstrap confidence interval = re-draw each group with replacement 1,000 times,
# recompute the SMD each time, and take the middle 95%. It shows how much the estimate wobbles.
# BIZ: if the interval covers 0, we cannot say Prime is different on that metric.
rng = np.random.default_rng(SEED)
rows = []
for label, col in METRICS.items():
    a, b = prime[col].values, non[col].values
    boots = [smd(rng.choice(a, len(a)), rng.choice(b, len(b))) for _ in range(1000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = mannwhitneyu(a, b, alternative="two-sided").pvalue
    rows.append({"metric": label, "prime_mean": a.mean(), "non_prime_mean": b.mean(),
                 "diff": a.mean() - b.mean(), "smd": smd(a, b), "ci_lo": lo, "ci_hi": hi, "p_value": p})
cmp_ = pd.DataFrame(rows)
print(cmp_.round(3).to_string(index=False))

# Forest plot: one dot per metric, bar = 95% CI, grey band = "negligible difference" zone.
fig, ax = plt.subplots(figsize=(9, 5.5))
y = np.arange(len(cmp_))[::-1]
ax.axvspan(-0.1, 0.1, color="#DDDDDD", alpha=.6, label="Negligible (|SMD| < 0.1)")
ax.axvline(0, color="black", lw=1)
ax.errorbar(cmp_.smd, y, xerr=[cmp_.smd - cmp_.ci_lo, cmp_.ci_hi - cmp_.smd],
            fmt="o", color=C_PRIME, ecolor=C_PRIME, capsize=4, lw=2, ms=7)
ax.set_yticks(y); ax.set_yticklabels(cmp_.metric); ax.set_xlim(-0.3, 0.3)
ax.set_xlabel("Prime minus Non-Prime, in standard deviations (dot = estimate, bar = 95% CI)")
ax.set_title("Prime members look like non-Prime members on every behaviour", weight="bold")
ax.legend(loc="upper right"); plt.tight_layout(); plt.show()



Prime: 2,283 customers | Non-Prime: 2,717 customers


               metric  prime_mean  non_prime_mean       diff    smd  ci_lo  ci_hi  p_value
 Transactions / month      74.823          76.529     -1.706 -0.040 -0.094  0.019    0.160
Avg transaction value    4949.429        5042.533    -93.103 -0.033 -0.088  0.024    0.243
        Monthly spend  367330.300      383371.093 -16040.793 -0.050 -0.106  0.006    0.174
      Tenure (months)      36.786          36.256      0.530  0.026 -0.030  0.083    0.354
       App engagement       5.522           5.541     -0.019 -0.007 -0.060  0.048    0.818
Cashback redeemed (%)      49.801          49.127      0.673  0.023 -0.026  0.078    0.423
   Net promoter score      -0.393           0.445     -0.838 -0.014 -0.068  0.036    0.605
     Support contacts       4.569           4.506      0.063  0.022 -0.031  0.079    0.440
        Churned (0/1)       0.197           0.184      0.012  0.031 -0.026  0.084    0.270


### Does Prime change customer behaviour? (Q2)

**Procedure**
We split customers into Prime (2,283) and non-Prime (2,717) and compared them on frequency (transactions per month), value (transaction size, monthly spend), loyalty (tenure, app engagement, NPS, churn), reward use (cashback redeemed) and friction (support contacts). To compare metrics measured in different units on one scale, we used the **standardised mean difference (SMD)**: the gap between group averages divided by the typical spread (below 0.1 is negligible, 0.2 small, 0.5 medium). Each estimate comes with a bootstrap 95% confidence interval (the data re-sampled 1,000 times to see how much the estimate wobbles). We also ran a two-sided Mann-Whitney test per metric.

**Observations**
- **Prime does not raise transaction frequency.** Prime members average 74.8 transactions a month vs 76.5 for non-Prime (SMD −0.04, p = 0.16). If anything it is slightly lower, but the difference is not significant.
- **Prime does not raise customer value.** Average transaction value (₹4,949 vs ₹5,043) and monthly spend (₹3.67 lakh vs ₹3.83 lakh) are slightly lower for Prime, again not significant.
- **Prime shows no loyalty advantage.** Tenure (36.8 vs 36.3 months), app engagement (5.52 vs 5.54), NPS (−0.4 vs +0.4) and churn (19.7% vs 18.4%, p = 0.27) are all statistically indistinguishable.
- **Prime and non-Prime use cashback and support alike:** cashback redeemed 49.8% vs 49.1%, and support contacts 4.6 vs 4.5.
- **Every difference is negligible.** All confidence intervals sit inside the "negligible" band (below 0.1 SD); monthly spend only touches its edge (−0.11 to +0.01) and still includes zero. Even the most favourable end of the transactions interval is about 0.02 SD, under one transaction a month.
- **Business meaning:** in this data, being a Prime member does not make a customer transact more often, spend more, stay longer or be happier. This is a comparison between existing members, not proof of cause, since customers chose whether to join.

In [17]:
# ===== CELL 11: IS THE COMPARISON FAIR, AND DOES THE GAP VARY BY SEGMENT? =====
from scipy.stats import chi2_contingency

# TECH: CONFOUNDING. If Prime members were richer or older, any gap could be about wealth or age,
# not Prime. A balance check compares background traits across the two groups.
# BIZ: if the groups look alike on background, a simple comparison is fair and no matching is needed.
bal = [{"background": c, "smd": round(smd(prime[c].values, non[c].values), 3)}
       for c in ["age", "monthly_income_inr", "tenure_months"]]
print(pd.DataFrame(bal).to_string(index=False))
for c in ["gender", "city_tier", "lifecycle_stage"]:
    print(f"{c}: chi-square p = {chi2_contingency(pd.crosstab(fe[c], fe.amazon_prime_member))[1]:.2f}")

def mean_ci(s):                      # mean +/- 1.96 standard errors = 95% CI
    m, se = s.mean(), s.std(ddof=1) / np.sqrt(len(s))
    return m, m - 1.96 * se, m + 1.96 * se

# BIZ: does Prime pay off at certain stages (e.g. loyalty building up over years) or in certain
# cities? We split the comparison by lifecycle stage and city tier. Churn uses the Wilson CI (Cell 3).
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for r, (seg, order) in enumerate([("lifecycle_stage", CFG["tenure_labels"]),
                                  ("city_tier", ["Tier 1", "Tier 2", "Tier 3"])]):
    for c, (metric, title) in enumerate([("avg_monthly_transactions", "Transactions per month"),
                                         ("churned", "Churn rate")]):
        ax = axes[r, c]
        for flag, color, name in [(1, C_PRIME, "Prime"), (0, C_NON, "Non-Prime")]:
            xs, ms, los, his = [], [], [], []
            for i, level in enumerate(order):
                s = fe[(fe[seg] == level) & (fe.amazon_prime_member == flag)][metric]
                if metric == "churned":
                    k, n = s.sum(), len(s); m = k / n; lo, hi = wilson(k, n)
                else:
                    m, lo, hi = mean_ci(s)
                xs.append(i + (-0.1 if flag == 1 else 0.1)); ms.append(m)
                los.append(m - lo); his.append(hi - m)
            ax.errorbar(xs, ms, yerr=[los, his], fmt="o", color=color, capsize=4, lw=2, ms=7, label=name)
        ax.set_xticks(range(len(order))); ax.set_xticklabels(order)
        ax.set_title(f"{title} by {seg.replace('_', ' ')}", weight="bold")
        if metric == "churned": ax.axhline(BASE, color="black", ls="--", lw=1)
        ax.legend()
fig.suptitle("Does the Prime gap change across customer stages or city tiers? (bars = 95% CI)",
             fontsize=13, weight="bold")
plt.tight_layout(); plt.show()



        background    smd
               age -0.008
monthly_income_inr -0.050
     tenure_months  0.026
gender: chi-square p = 0.42
city_tier: chi-square p = 0.96
lifecycle_stage: chi-square p = 0.28


### Is the comparison fair, and does the Prime gap vary by segment?

**Procedure**
First we checked for **confounding**: if Prime members were richer, older or in different cities, any gap could reflect those traits instead of Prime. We compared their backgrounds (age, income, tenure, gender, city tier). Then we split the Prime vs non-Prime comparison by lifecycle stage and by city tier, for transactions per month and churn rate, to see whether Prime pays off in particular groups (for example loyalty building over the years).

**Observations**
- **The two groups have near-identical backgrounds** (age SMD −0.01, income −0.05, tenure 0.03; gender p = 0.42, city tier p = 0.96, lifecycle stage p = 0.28). Tenure is balanced whether measured in months or as stages. The simple comparison in Cell 10 is therefore fair, and no matching or adjustment is needed.
- **How to read the charts:** each dot is the average (left) or the churn rate (right) for one group, and each bar is the 95% confidence interval, the range the true value plausibly sits in. The gap between a blue (Prime) and a grey (non-Prime) dot is what we saw in this sample. Heavily overlapping bars mean we cannot tell the groups apart.
- **Prime shows no advantage in any stage or city tier.** Prime members make slightly fewer or the same number of transactions everywhere (for example mature customers: 76.3 vs 76.3). Loyalty does not appear to build up with tenure for Prime members.
- **Three gaps stand out, but none is statistically reliable:**
  - Growing customers (1 to 3 years): Prime makes 73.5 transactions a month vs 77.4 for non-Prime (p = 0.07).
  - Tier 3 cities: 73.8 vs 77.2 transactions a month (p = 0.17).
  - New customers (under 1 year): Prime churns at 20.8% vs 17.0% (p = 0.17).
  A p-value of 0.17 means a gap this large would appear about 1 time in 6 by luck even if Prime made no difference.
- **Why we do not treat them as findings:** the grid contains 12 subgroup comparisons (3 stages and 3 tiers, each for transactions and churn). Even if Prime did nothing, about 0.6 of them would look significant by luck. After correcting for 12 comparisons the threshold is about 0.004, and the smallest p-value in the grid is 0.07. The gaps also do not repeat consistently: mature customers show no gap at all.
- **How we use them:** as leads for a follow-up test (for example, whether new Prime members churn more), not as conclusions.
- **City tier does not change the picture.** The Prime gap in transactions and churn is similar across Tier 1, 2 and 3, with overlapping intervals.

In [18]:
# ===== CELL 12: WHAT DOES PRIME COST? (SPEND VS CASHBACK BILL) =====
# TECH: bootstrap CI for a relative difference (Prime mean / Non-Prime mean - 1).
def boot_diff_pct(a, b, n=2000):
    rng = np.random.default_rng(SEED)
    out = [rng.choice(a, len(a)).mean() / rng.choice(b, len(b)).mean() - 1 for _ in range(n)]
    return np.percentile(out, [2.5, 97.5])

sp, sn = prime.monthly_spend.values, non.monthly_spend.values
cp, cn = prime.est_cashback_cost.values, non.est_cashback_cost.values
spend_gap = sp.mean() / sn.mean() - 1;  spend_lo, spend_hi = boot_diff_pct(sp, sn)
cost_gap  = cp.mean() / cn.mean() - 1;  cost_lo, cost_hi   = boot_diff_pct(cp, cn)
print(f"Spend per customer:     Prime {sp.mean():,.0f} vs Non-Prime {sn.mean():,.0f} -> {spend_gap:+.1%} (95% CI {spend_lo:+.1%} to {spend_hi:+.1%})")
print(f"Cashback cost/customer: Prime {cp.mean():,.0f} vs Non-Prime {cn.mean():,.0f} -> {cost_gap:+.1%} (95% CI {cost_lo:+.1%} to {cost_hi:+.1%})")
# BIZ: cashback actually paid per Rs 100 of spend (after redemption).
print(f"Cashback paid per Rs100 spent: Prime {cp.sum()/sp.sum()*100:.2f} vs Non-Prime {cn.sum()/sn.sum()*100:.2f}")

# BIZ: the key picture. Cashback is a cost; spend is what could pay for it. If Prime members spent
# much more, the higher bill would be justified. If spend is the same, the extra is pure cost.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, (title, a, b) in zip(axes, [("Avg monthly spend per customer (INR)", sp, sn),
                                    ("Avg monthly cashback cost per customer (INR)", cp, cn)]):
    for i, (vals, color) in enumerate([(a, C_PRIME), (b, C_NON)]):
        m, lo, hi = mean_ci(pd.Series(vals))
        ax.bar(i, m, color=color, width=.5)
        ax.errorbar(i, m, yerr=[[m - lo], [hi - m]], fmt="none", ecolor="#222", capsize=5, lw=1.5)
        ax.text(i, hi * 1.02, f"{m:,.0f}", ha="center", fontsize=11, weight="bold")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Prime", "Non-Prime"])
    ax.set_title(title, weight="bold"); ax.set_ylim(0, max(a.mean(), b.mean()) * 1.25)
fig.suptitle("Prime members spend no more, but cost more in cashback", fontsize=13, weight="bold")
plt.tight_layout(); plt.show()

Spend per customer:     Prime 367,330 vs Non-Prime 383,371 -> -4.2% (95% CI -8.6% to +0.6%)
Cashback cost/customer: Prime 4,214 vs Non-Prime 3,043 -> +38.5% (95% CI +29.7% to +47.7%)
Cashback paid per Rs100 spent: Prime 1.15 vs Non-Prime 0.79


### What does Prime cost? (spend vs cashback bill)

**Procedure**
Cashback is a cost to Amazon Pay, and spend is what could pay for it. We compared average monthly spend and average estimated cashback cost per customer for Prime vs non-Prime, with bootstrap confidence intervals for the percentage gap. Cashback cost uses the published Amazon Pay ICICI card rates (5% Prime / 3% non-Prime on eligible Amazon.in orders, 1% on partners and elsewhere) and an assumed spend mix (40% / 30% / 30%) with 85% of Amazon.in spend eligible.

### Observations 
- **Prime members do not spend more.** Average monthly spend is ₹367,330 for Prime vs ₹383,371 for Non-Prime, a difference of -4.2% (95% CI -8.6% to +0.6%). The interval includes zero, so Prime members spend about the same, and if anything slightly less.
- **Prime members cost clearly more.** Average monthly cashback cost is ₹4,214 for Prime vs ₹3,043 for Non-Prime: +38.5% (95% CI +29.7% to +47.7%). The whole interval is above zero.
- **Per ₹100 spent, Prime members earn about 1.15 in redeemed cashback vs 0.79 for Non-Prime** (about 45% more). The three ways of stating the Prime premium agree: +42% in the blended rate, +45% per ₹100 spent (Prime also redeems slightly more) and +38% per customer (Prime members spend 4% less, which offsets part of it). The gap comes from the higher Amazon.in cashback rate for Prime (5% vs 3%) applied to similar spending and redemption.
- **Business reading:** Prime membership is a cost multiplier, not a spending or loyalty driver, in this data. The 5% vs 3% rate difference is paid for, but no behavioural advantage shows up in return.
- **Caveat:** the exact percentage depends on the assumed rate card and spend mix, but the direction holds in every scenario of Cell 7A (+28% to +51%).

### Q2 Conclusion: How does Prime membership affect transaction frequency and loyalty?

**Answer:** In this data, Prime membership does not increase transaction frequency or loyalty, and it carries a higher cashback cost.

| Question | Finding |
|---|---|
| Does Prime raise transaction frequency? | No: 74.8 vs 76.5 transactions a month (not significant) |
| Does Prime raise spend or ticket size? | No: spend 4% lower (not significant) |
| Does Prime improve loyalty (tenure, churn, NPS, engagement)? | No: all differences negligible |
| Does the answer change by customer stage or city tier? | No clear pattern; one new-customer churn gap to monitor |
| What does Prime cost? | About 38% more cashback per customer (₹1,171 more a month), or 45% more per ₹100 spent |

**Recommendations**
1. **Do not assume Prime-linked cashback drives loyalty.** The data shows no loyalty or spend return on the extra cashback.
2. **Test before changing anything:** run an A/B test where a random subset of Prime members gets a lower or conditional Amazon Pay cashback tier, and measure spend, frequency and retention.
3. **Link the Prime bonus to behaviour:** pay the higher Prime rate only after activity milestones (for example a minimum number of transactions a month), so cashback rewards real engagement.
4. **Collect the missing data:** the share of spend on Amazon.in, Prime join date and fee, and Prime-specific behaviour, so the full value of Prime can be assessed.

**Caveats**
- This compares existing members and does not prove cause: customers chose whether to join Prime.
- Prime's subscription revenue and Amazon.in shopping value are outside this dataset.
- Cashback cost relies on assumed rates and an assumed spend mix.

In [19]:
# ===== CELL 13: HOW WE MEASURE VALUE (Q3) AND HOW CONCENTRATED IT IS =====
# BIZ: the data has no revenue or margin, so we measure value as PAYMENT VOLUME, the money moved
# through Amazon Pay, which is what any fee or interchange revenue would scale with.

# ASSUMPTIONS, kept in one place so they are easy to state and change:
CHURN_WINDOW_MONTHS = 12      # we assume the churn flag means "left within 12 months"
LIFETIME_CAP_MONTHS = 60      # cap the expected lifetime at 5 years (we don't extrapolate further)
TAKE_RATES = [0.010, 0.015, 0.020]   # ILLUSTRATIVE revenue per Rs100 of spend, NOT Amazon Pay's real margin

# TECH: if 19% of customers leave in a year, the steady monthly churn is 1 - (1 - 0.19)^(1/12).
# Expected lifetime in months = 1 / monthly churn. (Customer leaving each month at a constant rate.)
annual_churn    = fe.churned.mean()
monthly_churn   = 1 - (1 - annual_churn) ** (1 / CHURN_WINDOW_MONTHS)
lifetime_months = min(1 / monthly_churn, LIFETIME_CAP_MONTHS)
print(f"Churn {annual_churn:.1%} over {CHURN_WINDOW_MONTHS} months -> monthly churn {monthly_churn:.2%} "
      f"-> expected lifetime {lifetime_months:.0f} months")

# SENSITIVITY: how the expected lifetime changes with the assumed churn window.
# BIZ: the segment ranking does not change, only the absolute size of projected value.
for w in [6, 12, 24]:
    mc = 1 - (1 - annual_churn) ** (1 / w)          # steady monthly churn for that window
    lt = min(1 / mc, LIFETIME_CAP_MONTHS)            # expected lifetime, capped at 5 years
    print(f"Window {w:>2} months -> monthly churn {mc:.2%} -> expected lifetime {lt:.0f} months")

# IMPORTANT: churn was flat across every segment (Q1), so we use ONE overall lifetime for everyone.
# Segment-specific churn rates would only inject noise (they differ by chance, not by real effect).
# Consequence: the projected value ranks customers by monthly spend; tenure adds no future value.
fe["value_projected"] = fe.monthly_spend * lifetime_months          # projected volume over lifetime (INR)
fe["value_index"] = fe.monthly_spend / fe.monthly_spend.mean() * 100   # 100 = the average customer
fe["volume_to_date"] = fe.monthly_spend * fe.tenure_months          # past volume (for reference)

# TECH: concentration. Sort customers from highest to lowest spend and track the running share of volume.
# BIZ: if a few customers carry most of the volume, retention effort should focus on them.
v = np.sort(fe.monthly_spend.values)[::-1]
cum_vol  = np.cumsum(v) / v.sum()
cum_cust = np.arange(1, len(v) + 1) / len(v)
for q in [0.10, 0.20, 0.50]:
    print(f"Top {q:.0%} of customers -> {cum_vol[int(q * len(v)) - 1]:.1%} of payment volume")
n50 = (cum_vol < 0.5).sum() + 1
print(f"Customers needed for 50% of volume: {n50:,} ({n50 / len(v):.0%})")

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(cum_cust * 100, cum_vol * 100, color=C_PRIME, lw=2.5, label="Actual")
ax.plot([0, 100], [0, 100], color=C_NON, ls="--", lw=1.5, label="If every customer were equal")
i20 = int(0.20 * len(v)) - 1
ax.scatter([20], [cum_vol[i20] * 100], color=C_PRIME, s=60, zorder=3)
ax.annotate(f"Top 20% of customers\n= {cum_vol[i20]:.0%} of volume", (20, cum_vol[i20] * 100),
            xytext=(32, 35), arrowprops=dict(arrowstyle="->", color="#333"), fontsize=11)
ax.set_xlabel("Share of customers, highest spenders first (%)")
ax.set_ylabel("Share of total payment volume (%)")
ax.set_title("Customer value is concentrated in a minority of customers", weight="bold")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

Churn 19.0% over 12 months -> monthly churn 1.74% -> expected lifetime 57 months
Window  6 months -> monthly churn 3.45% -> expected lifetime 29 months
Window 12 months -> monthly churn 1.74% -> expected lifetime 57 months
Window 24 months -> monthly churn 0.87% -> expected lifetime 60 months
Top 10% of customers -> 28.2% of payment volume
Top 20% of customers -> 48.2% of payment volume
Top 50% of customers -> 84.2% of payment volume
Customers needed for 50% of volume: 1,052 (21%)


### How we measure customer value (Q3) and how concentrated it is

**Procedure**
The dataset has no revenue or margin, so we measure value as **payment volume**: monthly spend (transactions x average value), which is what any fee or interchange revenue would scale with. To turn it into a lifetime figure we need an expected lifetime. Because the dataset does not say when customers churned (Cell 13A), we **assume the churn flag covers 12 months**. A 19% annual churn then gives a steady monthly churn of 1.74% and an expected lifetime of about 57 months (capped at 60). We use one overall lifetime for every customer, because churn was flat across all segments (Q1), so segment-specific churn rates would only add noise. We then built a **value index** (monthly spend relative to the average customer, 100 = average) and a **concentration curve**: customers sorted from highest to lowest spender, with the running share of total volume.

**Observations**
- **Value is concentrated, but not extreme.** The top 10% of customers account for 28% of payment volume, the top 20% for 48%, and the top 50% for 84%. The bottom half of customers contribute only about 16%.
- **About 1,050 customers (21%) generate half of all volume.** These are the customers where retention effort and personalised offers have the largest payoff.
- **Expected lifetime is about 57 months,** from a monthly churn of 1.74%. It rests on the 12-month assumption, which affects the absolute size of lifetime value but not the ranking of segments.
- **Lifetime is the same for every segment,** so projected value ranks customers by monthly spend. Tenure so far adds no future value, because churn does not vary with tenure.
- **The concentration is partly a result of how spend is built.** Multiplying two evenly spread features (transactions and ticket size) creates a long tail of heavy spenders (see Cell 7), so we describe it as a concentration of payment volume, not as proof of a natural "whale" customer group.

In [20]:
# ===== CELL 13A: DIAGNOSTIC, IS TENURE THE "TIME UNTIL CHURN"? =====
from scipy.stats import ks_2samp
from scipy.optimize import brentq

# BIZ: if tenure were "months until the customer left", we could measure true customer lifetimes
# (survival analysis). We test that reading before relying on it.
# TECH: CENSORING. For churned customers tenure would be the full time they stayed; for active
# customers it is only "so far" (they may still leave). That is how survival analysis treats it.

edges  = [0, 12, 24, 36, 48, 60, 72]
labels = ["1-12", "13-24", "25-36", "37-48", "49-60", "61-72"]
fe["tenure_band"] = pd.cut(fe.tenure_months, edges, labels=labels)

# --- Check 1: do churned customers have shorter tenure than active ones? ---
# TECH: Kolmogorov-Smirnov test = "do two groups have the same distribution?" (p > 0.05 = no evidence they differ)
ten_c = fe.loc[fe.churned == 1, "tenure_months"]
ten_a = fe.loc[fe.churned == 0, "tenure_months"]
ks_p = ks_2samp(ten_c, ten_a).pvalue
print(f"REAL tenure: churned median {ten_c.median():.0f} vs active {ten_a.median():.0f} months | KS test p = {ks_p:.2f}")

# --- Check 2: share churned by tenure band (real data, with 95% CI) ---
real = fe.groupby("tenure_band", observed=True).churned.agg(["sum", "count"])
real["rate"] = real["sum"] / real["count"]
real[["lo", "hi"]] = [wilson(k, n) for k, n in zip(real["sum"], real["count"])]

# --- Simulation: what a GENUINE time-to-churn dataset would look like ---
# ASSUMPTIONS: customers joined evenly over the past 71 months, and each leaves at a constant monthly rate.
# TECH: we pick the exit rate h so that 19% have left, the same as in our data (brentq = root finder).
# Fixed random numbers (seeded) keep the result repeatable.
N = 200_000
rng = np.random.default_rng(SEED)
join_age = rng.uniform(1, 71, N)               # months since each simulated customer joined
u = rng.uniform(size=N)
def churn_share(h):                            # share who would have left within their time on the platform
    return ((-np.log(u) / h) < join_age).mean()
h = brentq(lambda h: churn_share(h) - fe.churned.mean(), 1e-4, 0.1)
T = -np.log(u) / h                             # simulated time until each customer leaves
sim_churn  = T < join_age                      # left before the snapshot
sim_tenure = np.where(sim_churn, T, join_age)  # churned: time to exit; active: time so far
sim_band   = pd.cut(sim_tenure, edges, labels=labels)
sim_rate   = pd.Series(sim_churn).groupby(sim_band, observed=True).mean()
print(f"SIMULATED (true time-to-churn, monthly exit rate {h:.2%}): tenure churned median "
      f"{np.median(sim_tenure[sim_churn]):.0f} vs active {np.median(sim_tenure[~sim_churn]):.0f} months")

# --- Survival view of the REAL data (Kaplan-Meier style hazard) ---
# TECH: monthly exit rate in a band = exits / customer-months "at risk" (still with us when the month started).
# BIZ: if this rate balloons with tenure, the survival reading gives implausible results.
d = fe.groupby("tenure_months").agg(events=("churned", "sum"), n=("churned", "size"))
d["at_risk"] = d.n[::-1].cumsum()[::-1]        # customers with tenure >= this month
d["band"] = pd.cut(d.index, edges, labels=labels)
km_haz = d.groupby("band", observed=True).apply(lambda g: g.events.sum() / g.at_risk.sum())

cmp_tbl = pd.DataFrame({"real_share_churned_%": (real.rate * 100).round(1),
                        "simulated_share_churned_%": (sim_rate * 100).round(1),
                        "survival_view_monthly_exit_%": (km_haz * 100).round(2)})
print(cmp_tbl.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
x = np.arange(len(labels))
ax.bar(x, real.rate * 100, color=C_PRIME, width=0.55, label="Real data")
ax.errorbar(x, real.rate * 100, yerr=[(real.rate - real.lo) * 100, (real.hi - real.rate) * 100],
            fmt="none", ecolor="#222", capsize=4)
ax.plot(x, sim_rate.values * 100, color=C_NON, marker="o", lw=2.5, label="Expected if tenure = time until churn")
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_xlabel("Tenure band (months)")
ax.set_ylabel("Share churned (%)"); ax.set_ylim(0, 35)
ax.set_title("Churn share does not fall with tenure", weight="bold"); ax.legend(loc="upper right")

ax = axes[1]
ax.bar(x, km_haz.values * 100, color=C_PRIME, width=0.55)
for xi, vv in zip(x, km_haz.values * 100):
    ax.text(xi, vv + 0.05, f"{vv:.2f}%", ha="center", fontsize=10, weight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_xlabel("Tenure band (months)")
ax.set_ylabel("Implied monthly exit rate (%)")
ax.set_title("Survival view: exit rate would rise ~10x with tenure", weight="bold")
plt.tight_layout(); plt.show()

REAL tenure: churned median 37 vs active 37 months | KS test p = 0.88
SIMULATED (true time-to-churn, monthly exit rate 0.61%): tenure churned median 19 vs active 32 months
       real_share_churned_%  simulated_share_churned_%  survival_view_monthly_exit_%
1-12                   18.7                       30.0                          0.27
13-24                  18.8                       24.3                          0.33
25-36                  18.7                       20.1                          0.46
37-48                  21.2                       15.0                          0.72
49-60                  18.2                        9.6                          1.06
61-72                  18.4                        3.2                          2.97


### Diagnostic: is tenure the "time until churn"?

**Procedure**
The dataset does not say when customers churned. If tenure meant "months until the customer left", we could measure true customer lifetimes with survival analysis, which would be better than assuming a time window. We tested that reading in three ways:
1. **Compared tenure of churned and active customers** (Kolmogorov-Smirnov test). If tenure were time-to-exit, churned customers should show visibly shorter tenure.
2. **Simulated what a genuine time-to-churn dataset would look like.** We assumed customers joined evenly over the past 71 months and each leaves at a constant monthly rate, set so that 19% have left, as in our data. We then compared the simulated pattern with the real one.
3. **Computed the exit rate that the survival reading would imply.** For each tenure band, this is customers leaving divided by customers still with us at the start of each month.

**Observations**
- **Churned and active customers have identical tenure:** both have a median of 37 months (KS test p = 0.88). A genuine time-to-churn process would show churned customers at about 19 months vs 32 for active customers.
- **The share churned does not fall as tenure rises.** It is flat at 18% to 21% in every tenure band. In the simulated time-to-churn dataset, it falls from about 30% for tenure of 1 to 12 months to about 3% for 61 to 72 months, because long tenures are only possible for early joiners.

| Tenure band (months) | Real share churned | Simulated if tenure = time-to-exit | Exit rate implied by survival view |
|---|---|---|---|
| 1-12 | 18.7% | 30.0% | 0.27% a month |
| 13-24 | 18.8% | 24.3% | 0.33% |
| 25-36 | 18.7% | 20.1% | 0.46% |
| 37-48 | 21.2% | 15.0% | 0.72% |
| 49-60 | 18.2% | 9.6% | 1.06% |
| 61-72 | 18.4% | 3.2% | 2.97% |

- **The survival reading gives implausible results.** It would mean the monthly exit rate rises about tenfold, from 0.27% in the first year to about 3% after five years, so customers who have stayed five years would be ten times likelier to leave than newcomers. That also contradicts the flat churn we found in Q1.
- **Conclusion:** in this dataset, tenure behaves as if it were generated independently of churn. It is "how long the customer has been with us", not "how long until they left". We therefore do not use survival analysis. For customer lifetime we use a stated assumption (churn over a 12-month window), with a sensitivity table, and note this in the limitations.
- **Caveat:** this is an inference from how the data behaves. If the organisers state that tenure means time until churn, we would switch to survival analysis.

In [21]:
# ===== CELL 14: WHICH SEGMENTS CARRY THE MOST VALUE? =====
from scipy.stats import kruskal
from matplotlib.lines import Line2D

# TECH: CIRCULARITY. Some segments are built FROM spend ingredients (payment style uses transactions and
# ticket size; cashback segment and say-do gap use the engagement index, which includes transactions).
# They will rank by value BY DESIGN, so that ranking is not a discovery. Profile segments (age, city,
# tenure stage, gender, Prime, NPS) are NOT built from spend, so any gap there is a genuine finding.
PROFILE_SEGS   = ["age_band", "city_tier", "lifecycle_stage", "gender", "amazon_prime_member", "nps_group"]
BEHAVIOUR_SEGS = ["payment_style", "cashback_segment", "say_do_gap"]

total_vol = fe.monthly_spend.sum()
rows, tests = [], []
for seg in PROFILE_SEGS + BEHAVIOUR_SEGS:
    kind = "profile" if seg in PROFILE_SEGS else "behaviour"
    for level, d in fe.groupby(seg, observed=True):
        m, lo, hi = mean_ci(d.value_index)                     # mean value index with 95% CI
        rows.append({"segment": seg, "level": str(level), "kind": kind, "n": len(d),
                     "pct_customers": len(d) / len(fe) * 100, "value_index": m, "ci_lo": lo, "ci_hi": hi,
                     "pct_volume": d.monthly_spend.sum() / total_vol * 100})
    # TECH: Kruskal-Wallis = "do these groups differ in spend?" for 2+ groups, no bell-curve assumption.
    p = kruskal(*[d.monthly_spend.values for _, d in fe.groupby(seg, observed=True)]).pvalue
    tests.append({"segment": seg, "kind": kind, "kruskal_p": round(p, 4)})
seg_tbl, test_tbl = pd.DataFrame(rows), pd.DataFrame(tests)

# TECH: 9 tests at once -> Bonferroni correction (as in Cell 8): threshold 0.05 / 9.
alpha_adj = 0.05 / len(test_tbl)
test_tbl["significant_after_correction"] = test_tbl.kruskal_p < alpha_adj
print(test_tbl.to_string(index=False)); print(f"Corrected threshold: {alpha_adj:.4f}\n")
print(seg_tbl.round(1).to_string(index=False))

# Dot plot: value index per segment (100 = average customer), bar = 95% CI.
seg_tbl["label"] = seg_tbl.segment.str.replace("_", " ") + ": " + seg_tbl.level
fig, ax = plt.subplots(figsize=(10, 10))
y = np.arange(len(seg_tbl))[::-1]
ax.axvline(100, color="black", ls="--", lw=1)
for yi, r in zip(y, seg_tbl.itertuples()):
    c = C_PRIME if r.kind == "profile" else C_NON
    ax.errorbar(r.value_index, yi, xerr=[[r.value_index - r.ci_lo], [r.ci_hi - r.value_index]],
                fmt="o", color=c, ecolor=c, capsize=3, lw=1.8, ms=6)
ax.set_yticks(y); ax.set_yticklabels(seg_tbl.label, fontsize=9)
ax.set_xlabel("Value index (100 = average customer; bar = 95% CI)")
ax.set_title("Which segments carry the most value?", weight="bold")
ax.legend(handles=[Line2D([0], [0], marker="o", color=C_PRIME, lw=0, label="Profile segments (not built from spend)"),
                   Line2D([0], [0], marker="o", color=C_NON, lw=0, label="Behaviour segments (partly built from transactions)")],
          loc="upper right")
plt.tight_layout(); plt.show()

            segment      kind  kruskal_p  significant_after_correction
           age_band   profile     0.7637                         False
          city_tier   profile     0.0221                         False
    lifecycle_stage   profile     0.8524                         False
             gender   profile     0.7472                         False
amazon_prime_member   profile     0.1735                         False
          nps_group   profile     0.0659                         False
      payment_style behaviour     0.0000                          True
   cashback_segment behaviour     0.0000                          True
         say_do_gap behaviour     0.0000                          True
Corrected threshold: 0.0056

            segment                 level      kind    n  pct_customers  value_index  ci_lo  ci_hi  pct_volume
           age_band                 18-25   profile  874           17.5         99.7   93.8  105.5        17.4
           age_band                 26-

### Which customer segments carry the most value?

**Procedure**
We compared nine segmentations on a **value index**: average monthly spend relative to the average customer (100 = average), with 95% confidence intervals and the share of total payment volume each segment holds. We split them into two kinds:
- **Profile segments** (age band, city tier, lifecycle stage, gender, Prime, NPS group), which are not built from spend. Any gap here would be a genuine finding.
- **Behaviour segments** (payment style, cashback segment, say-do gap), which are built partly from transaction data. They rank by value **by design**, so that ranking describes where value sits, not a discovered driver.

We tested whether segments differ in spend with a Kruskal-Wallis test (it does not assume a bell curve). Because we tested nine segmentations, we used a Bonferroni-corrected threshold of 0.0056 (0.05 / 9).

**Observations**
- **Profile does not predict value.** Every profile segment sits between about 96 and 103 on the index, and none is significant after correction. The closest are city tier (Tier 1 at 103 vs Tier 3 at 96, p = 0.022) and NPS group (p = 0.066), and neither clears the 0.0056 threshold. Prime (98 vs 102, p = 0.17), age, gender and tenure stage show nothing. Demographics and tenure cannot be used to find high-value customers.
- **Value follows behaviour.** Power users (high frequency and high ticket size) are 25% of customers and 55% of payment volume (index 222). Light users are 25% of customers but only 6.5% of volume (index 26).
- **Frequent small payers and occasional big shoppers are worth the same.** Both sit at about 76 and hold 19% of volume each, so both styles deserve retention attention.
- **Two-thirds of volume comes from high-engagement customers, whatever their cashback use:** organic loyalists (index 135) and real engagers (133) each hold about a third of volume. Deal-seekers (67) and passives (65) hold about a sixth each. Real engagers and deal-seekers use cashback to the same degree but differ twofold in value, so cashback use alone does not identify valuable customers (this sets up Q4). Because the engagement index includes transactions, this gap is partly built in.
- **High-value customers who are unhappy are a retention priority.** The "active but unhappy" group (17% of customers) holds 23% of volume (index 136). The "happy but idle" group (17% of customers) holds only 11% (index 65).
- **Gender "Other" is too small to interpret** (97 customers; confidence interval about 85 to 118).
- **Business meaning:** to find high-value customers we must look at what they do (transaction behaviour), not who they are. Retention effort should target active, high-volume customers, especially those showing dissatisfaction.

In [22]:
# ===== CELL 15: VALUE AFTER CASHBACK (BREAK-EVEN AND SCENARIOS) =====
import seaborn as sns, textwrap

# BIZ: volume is not profit. Cashback is paid out of whatever Amazon Pay earns per Rs100 of spend.
# BREAK-EVEN revenue rate = cashback paid per Rs100 of spend. If real revenue per Rs100 is below this,
# the segment LOSES money after cashback; above it, the segment makes money.
fe["prime_label"] = fe.amazon_prime_member.map({1: "Prime", 0: "Non-Prime"})
order = ["Passive", "Organic loyalist", "Deal-seeker", "Real engager"]

g  = fe.groupby("cashback_segment").agg(spend=("monthly_spend", "sum"), cost=("est_cashback_cost", "sum")).loc[order]
gp = fe.groupby("prime_label").agg(spend=("monthly_spend", "sum"), cost=("est_cashback_cost", "sum"))
g["breakeven"], gp["breakeven"] = g.cost / g.spend * 100, gp.cost / gp.spend * 100
print("Break-even revenue per Rs100 of spend (%):\n", pd.concat([g.breakeven, gp.breakeven]).round(2).to_string())

# SCENARIOS: net contribution per customer per month = spend x revenue rate - cashback cost.
# The revenue rates are ILLUSTRATIVE assumptions (see TAKE_RATES), shown as a range, not a forecast.
net = pd.DataFrame({
    f"{tr:.1%} revenue": fe.groupby("cashback_segment").apply(
        lambda d, tr=tr: (d.monthly_spend * tr - d.est_cashback_cost).mean()).loc[order]
    for tr in TAKE_RATES})
print("\nNet contribution (Rs per customer per month):\n", net.round(0).to_string())
neg = {f"{tr:.1%}": f"{((fe.monthly_spend * tr - fe.est_cashback_cost) < 0).mean():.0%}" for tr in TAKE_RATES}
print("\nShare of customers losing money after cashback:", neg)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]
labels = order + ["Prime", "Non-Prime"]
vals = list(g.breakeven) + [gp.breakeven["Prime"], gp.breakeven["Non-Prime"]]
x = list(range(4)) + [4.6, 5.6]
ax.bar(x, vals, color=C_PRIME, width=0.7)
for xi, vv in zip(x, vals): ax.text(xi, vv + 0.04, f"{vv:.2f}%", ha="center", fontsize=10, weight="bold")
for tr in TAKE_RATES:                                   # dashed lines = the assumed revenue scenarios
    ax.axhline(tr * 100, color="#333", ls="--", lw=1)
    ax.text(-0.6, tr * 100 + 0.02, f"{tr:.1%} revenue", fontsize=9, ha="left", va="bottom")
ax.set_xticks(x); ax.set_xticklabels([textwrap.fill(l, 10) for l in labels], fontsize=9)
ax.set_ylabel("Cashback cost per Rs100 of spend (%)")
ax.set_title("Break-even revenue rate by segment", weight="bold"); ax.set_ylim(0, 2.3)

ax = axes[1]
sns.heatmap(net / 1000, annot=True, fmt=".1f", cmap="RdBu", center=0, linewidths=1,
            cbar_kws={"label": "Rs '000 per customer per month"}, ax=ax)
ax.set_title("Net contribution after cashback (Rs '000 / customer / month)", weight="bold")
ax.set_ylabel(""); ax.set_xlabel("Assumed revenue per Rs100 of spend (take rate)")
plt.tight_layout(); plt.show()

Break-even revenue per Rs100 of spend (%):
 Passive             0.45
Organic loyalist    0.46
Deal-seeker         1.43
Real engager        1.43
Non-Prime           0.79
Prime               1.15

Net contribution (Rs per customer per month):
                   1.0% revenue  1.5% revenue  2.0% revenue
cashback_segment                                          
Passive                 1348.0        2568.0        3788.0
Organic loyalist        2714.0        5249.0        7783.0
Deal-seeker            -1085.0         179.0        1444.0
Real engager           -2152.0         344.0        2839.0

Share of customers losing money after cashback: {'1.0%': '46%', '1.5%': '19%', '2.0%': '6%'}


### Value after cashback: break- even and scenarios

**Procedure**
Payment volume is not profit, because cashback is paid out of whatever Amazon Pay earns on each ₹100 of spend. We measured two things:
- **Break-even revenue rate:** cashback paid per ₹100 of spend for each segment (using the Amazon Pay ICICI card rates and the assumed spend mix). If the real revenue per ₹100 is below this figure, that segment loses money after cashback.
- **Net contribution per customer per month** under three illustrative revenue rates (1%, 1.5% and 2% of spend): spend x revenue rate - cashback cost.
The revenue rates are assumptions to show a range, not Amazon Pay's actual margin.

**Observations**

- **Break-even differs about threefold across segments:** Passive 0.45%, Organic loyalist 0.46%, Non-Prime 0.79%, Prime 1.15%, Deal-seeker 1.43% and Real engager 1.43%. The gap comes from how much earned cashback customers redeem (cost per ₹100 = rate x redemption): the two high-cost segments redeem about three times as much (74% vs 24%).
- **The highest-volume segment is not the most profitable.** Real engagers hold a third of payment volume, but at 1% revenue they lose about ₹2,200 per customer per month, and at 1.5% they only just break even (+₹340). They turn clearly profitable only at about 2% revenue per ₹100 spent (+₹2,800).
- **Organic loyalists are the most valuable segment after cashback.** They hold the same volume as real engagers (about a third) but cost one-third as much per ₹100 spent, so they earn about ₹2,700 per customer per month at 1% revenue and about ₹7,800 at 2%.
- **Deal-seekers are loss-making at low revenue rates.** They carry half the volume of the high-engagement segments and the same high cost per ₹100: -₹1,100 at 1% and about break-even at 1.5% (+₹180).
- **Passive customers are profitable even at low revenue rates,** because they redeem little cashback, but their volume is small.
- **Prime needs about 45% more revenue per ₹100 spent than non-Prime to break even** (1.15% vs 0.79%), which matches the per-₹100 cost gap found in Q2.
- **Share of customers losing money after cashback:** 46% at 1% revenue, 19% at 1.5% and 6% at 2%. Whether cashback is "worth it" depends heavily on the true revenue per ₹100, which the data cannot tell us. This is the key number to confirm with Amazon Pay.
- **Caveats:** the revenue rates are illustrative, cashback cost relies on the assumed ICICI rates and spend mix, and rupee amounts are overstated because many customers' spend exceeds their income. We use this chart to compare segments and rank where cashback is least efficient, not to quote absolute profit.


### Q3 Conclusion: Which customer segments deliver the highest lifetime value?

**Answer:** Value follows what customers do, not who they are. Two-thirds of payment volume comes from highly engaged customers, but after cashback the most valuable segment is the **organic loyalists**: high-engagement customers who redeem little cashback.

| Question | Finding |
|---|---|
| Does profile (age, city, gender, tenure stage, Prime, NPS) identify high-value customers? | No: all within 96 to 103 on the value index; none significant after correction |
| Where does volume sit? | Top 20% of customers = 48% of volume; about 1,050 customers (21%) = half |
| Which behaviour segments carry the most volume? | Power users (25% of customers, 55% of volume); high-engagement customers (two-thirds of volume) |
| Who is most profitable after cashback? | Organic loyalists (positive at every revenue rate tested) |
| Who is least profitable per ₹ spent? | Real engagers and deal-seekers (break-even about 1.4% of spend vs about 0.5% for the others) |
| Retention priority group | "Active but unhappy" customers: 17% of customers, 23% of volume |

**Recommendations**
1. **Prioritise retention effort on high-volume, high-engagement customers,** especially the unhappy ones (detractors who are still very active).
2. **Do not assume the highest-volume customers are the most profitable:** real engagers cost the most per ₹100 spent. Restructure their cashback (for example tiered rates that reward activity, not blanket redemption) so the cost per ₹100 falls.
3. **Protect organic loyalists with non-cashback benefits** (recognition, priority service, early access), since they are low-cost and high-value and do not need cashback to stay.
4. **Review cashback for deal-seekers:** they redeem heavily for half the volume and are loss-making at low revenue rates.
5. **Confirm Amazon Pay's actual revenue per ₹100 of spend,** because it decides which segments are profitable (the break-even chart shows exactly where the lines cross).

**Caveats**
- Value is measured as payment volume, and cashback uses assumed ICICI rates and an assumed spend mix.
- Lifetime assumes the churn flag covers 12 months, and it is the same for all segments because churn does not vary.
- Behaviour segments rank by value partly by design (they are built from transaction data), so they describe where value sits, not what causes it.

In [23]:
# ===== CELL 16: DOES CASHBACK REDEMPTION GO WITH REAL ENGAGEMENT? (Q4) =====
from scipy.stats import spearmanr

# BIZ: if cashback drives real engagement, customers who redeem MORE should also be more engaged,
# transact more, stay longer and be happier. If it only attracts deal-hunters, they would redeem a lot
# without showing any of that. So we look at what changes as redemption rises.
# TECH: cut redemption (% of earned cashback used) into 5 equal-sized bands (quintiles).
fe["cashback_band"] = pd.qcut(fe.cashback_used_pct, q=5, duplicates="drop",
                              labels=["Lowest 20%", "20-40%", "40-60%", "60-80%", "Highest 20%"])
print(fe.groupby("cashback_band", observed=True).cashback_used_pct.agg(["min", "max", "size"])
        .rename(columns={"min": "redeems_from_%", "max": "redeems_to_%", "size": "customers"}))

METRICS = {"App engagement score": "app_engagement_score", "Transactions / month": "avg_monthly_transactions",
           "Monthly spend (INR)": "monthly_spend", "Tenure (months)": "tenure_months",
           "Net promoter score": "net_promoter_score", "Support contacts": "customer_support_contacts",
           "Churned (0/1)": "churned"}

# TECH: Spearman correlation (rank-based) between redemption and each metric, with a bootstrap 95% CI.
# Bootstrap = re-draw customers with replacement 500 times and recompute, to see how much it wobbles.
# BIZ: a correlation whose CI covers 0 means "no detectable link".
rng = np.random.default_rng(SEED); idx = np.arange(len(fe)); rows = []
for label, col in METRICS.items():
    rho, p = spearmanr(fe.cashback_used_pct, fe[col])
    boots = []
    for _ in range(500):
        i = rng.choice(idx, len(idx))
        boots.append(spearmanr(fe.cashback_used_pct.values[i], fe[col].values[i])[0])
    rows.append({"metric": label, "spearman_rho": round(rho, 3), "ci_lo": round(np.percentile(boots, 2.5), 3),
                 "ci_hi": round(np.percentile(boots, 97.5), 3), "p_value": round(p, 3)})
print(pd.DataFrame(rows).to_string(index=False))

# Direct comparison: light vs heavy redeemers (split at the median) on the engagement index.
heavy = fe.cashback_used_pct >= fe.cashback_used_pct.median()
print("\nAverage engagement index: light redeemers %.3f vs heavy redeemers %.3f"
      % (fe.loc[~heavy, "engagement_index"].mean(), fe.loc[heavy, "engagement_index"].mean()))

# Small multiples: average of each metric across the 5 redemption bands (bar = 95% CI).
# TECH: y-axis ranges are fixed to a sensible span so that tiny wiggles are not visually exaggerated.
YLIMS = {"App engagement score": (4, 7), "Transactions / month": (60, 90), "Monthly spend (INR)": (300000, 450000),
         "Tenure (months)": (30, 43), "Net promoter score": (-10, 10), "Churned (0/1)": (0.10, 0.30)}
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, label in zip(axes.ravel(), list(YLIMS)):
    col = METRICS[label]; xs, ms, los, his = [], [], [], []
    for i, (band, d) in enumerate(fe.groupby("cashback_band", observed=True)):
        if col == "churned":
            k, n = d[col].sum(), len(d); m = k / n; lo, hi = wilson(k, n)      # Wilson CI for a rate (Cell 3)
        else:
            m, lo, hi = mean_ci(d[col])
        xs.append(i); ms.append(m); los.append(m - lo); his.append(hi - m)
    ax.errorbar(xs, ms, yerr=[los, his], fmt="o-", color=C_PRIME, capsize=4, lw=2, ms=7)
    ax.axhline(fe[col].mean(), color="black", ls="--", lw=1)
    ax.set_ylim(*YLIMS[label])
    ax.set_xticks(range(5)); ax.set_xticklabels(["Lowest\n20%", "20-40%", "40-60%", "60-80%", "Highest\n20%"], fontsize=9)
    ax.set_title(label, weight="bold", fontsize=11)
axes[0, 0].set_ylabel("Average"); axes[1, 0].set_ylabel("Average")
fig.suptitle("Customers who redeem more cashback are not more engaged, higher-spending or more loyal "
             "(bars = 95% CI, dashed = overall)", fontsize=12.5, weight="bold")
plt.tight_layout(); plt.show()


               redeems_from_%  redeems_to_%  customers
cashback_band                                         
Lowest 20%                  0            19       1022
20-40%                     20            40       1038
40-60%                     41            60        963
60-80%                     61            79        997
Highest 20%                80           100        980


              metric  spearman_rho  ci_lo  ci_hi  p_value
App engagement score         0.018 -0.009  0.045    0.212
Transactions / month        -0.012 -0.041  0.015    0.391
 Monthly spend (INR)        -0.008 -0.035  0.017    0.549
     Tenure (months)        -0.012 -0.041  0.018    0.407
  Net promoter score         0.011 -0.014  0.039    0.420
    Support contacts        -0.015 -0.041  0.012    0.291
       Churned (0/1)        -0.016 -0.043  0.014    0.250

Average engagement index: light redeemers 0.499 vs heavy redeemers 0.501


### Does cashback redemption go with real engagement? (Q4)

**Procedure**
If cashback drives real engagement, customers who redeem more of their earned cashback should also be more engaged, transact more, spend more, stay longer, be happier and churn less. If it only attracts deal-hunters, they would redeem heavily without showing any of that. We split customers into five equal-sized bands by redemption (from the lowest 20% to the highest 20%) and compared the average of each metric across the bands. We also computed the Spearman correlation between redemption and each metric, with a bootstrap 95% confidence interval (the data re-sampled 500 times), and compared light and heavy redeemers (split at the median) on the engagement index.

**Observations**
- **Redemption goes with nothing.** All correlations are within ±0.02 and every confidence interval includes zero (app engagement 0.018, transactions −0.012, spend −0.008, tenure −0.012, NPS 0.011, support contacts −0.015, churn −0.016; all p-values above 0.2).
- **Light and heavy redeemers are equally engaged:** the engagement index is 0.499 for light and 0.501 for heavy redeemers.
- **In the charts, every band sits within the uncertainty range of the overall average** for engagement, transactions, spend, tenure, NPS and churn.
- **Business meaning:** customers who use more cashback are not more engaged, higher-spending, longer-staying, happier or less likely to leave. Redemption does not signal real engagement, and the data gives no evidence that it builds it.
- **Caveat:** this compares customers and does not show what would happen to an individual if their cashback changed. That is why we recommend testing before changing the scheme.

In [24]:
fe.cashback_band.hist()

<Axes: title={'center': 'Churned (0/1)'}>

In [25]:
# ===== CELL 17: WHO GETS THE CASHBACK, AND WHAT DO THEY BRING? =====
import textwrap

# BIZ: the cashback segments from Cell 7 cross two things: how much cashback customers redeem
# (heavy vs light) and how engaged they are (high vs low). We now compare each segment's share of
# customers, of payment volume, and of the cashback BILL (estimated cost, Cell 7 assumptions).
order = ["Real engager", "Deal-seeker", "Organic loyalist", "Passive"]
g = (fe.groupby("cashback_segment")
       .agg(customers=("churned", "size"), spend=("monthly_spend", "sum"), cost=("est_cashback_cost", "sum"),
            redemption=("cashback_used_pct", "mean"), engagement_index=("engagement_index", "mean"),
            transactions=("avg_monthly_transactions", "mean"), app_score=("app_engagement_score", "mean"),
            tenure=("tenure_months", "mean"), nps=("net_promoter_score", "mean"), churn=("churned", "mean"))
       .loc[order])
g["pct_customers"]      = g.customers / g.customers.sum() * 100
g["pct_volume"]         = g.spend / g.spend.sum() * 100
g["pct_cashback_bill"]  = g.cost / g.cost.sum() * 100
# BIZ: "volume per unit of cashback" = share of volume / share of bill. Above 1 = brings more than its
# share of the cost; below 1 = costs more than it brings. (A fair-share ratio is 1.0.)
g["volume_per_cashback"] = g.pct_volume / g.pct_cashback_bill

print(g[["redemption", "engagement_index", "transactions", "app_score", "tenure", "nps", "churn"]].round(2).to_string())
print(g[["pct_customers", "pct_volume", "pct_cashback_bill", "volume_per_cashback"]].round(2).to_string())
hi = g.loc[["Real engager", "Deal-seeker"]]
print(f"\nHeavy redeemers (real engagers + deal-seekers): {hi.pct_customers.sum():.0f}% of customers, "
      f"{hi.pct_volume.sum():.0f}% of volume, {hi.pct_cashback_bill.sum():.0f}% of the cashback bill")

# Colours: grey = customers, blue = volume, orange = cashback bill (always direct-labelled, never colour alone).
C_COST, C_CUST = "#D2591A", "#BDBDBD"
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [1.5, 1]})
ax = axes[0]; x = np.arange(len(order)); w = 0.26
for k, (col, color, name) in enumerate([("pct_customers", C_CUST, "Share of customers"),
                                        ("pct_volume", C_PRIME, "Share of payment volume"),
                                        ("pct_cashback_bill", C_COST, "Share of cashback bill")]):
    bars = ax.bar(x + (k - 1) * w, g[col], w, color=color, label=name)
    for b, v in zip(bars, g[col]):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.8, f"{v:.0f}%", ha="center", fontsize=9, weight="bold")
ax.set_xticks(x); ax.set_xticklabels([textwrap.fill(o, 12) for o in order])
ax.set_ylabel("% of total"); ax.set_ylim(0, 60)
ax.set_title("Who gets the cashback, and what they bring", weight="bold"); ax.legend(loc="upper right")

ax = axes[1]
bars = ax.bar(x, g.volume_per_cashback, color=C_PRIME, width=0.55); ax.axhline(1, color="black", ls="--", lw=1)
for b, v in zip(bars, g.volume_per_cashback):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.04, f"{v:.2f}", ha="center", fontsize=10, weight="bold")
ax.set_xticks(x); ax.set_xticklabels([textwrap.fill(o, 12) for o in order]); ax.set_ylim(0, 2.6)
ax.set_ylabel("Volume share / cashback-bill share")
ax.set_title("Volume per unit of cashback (1.0 = fair share)", weight="bold")
plt.tight_layout(); plt.show()

                  redemption  engagement_index  transactions  app_score  tenure   nps  churn
cashback_segment                                                                            
Real engager           74.26              0.67        100.58       7.24   36.74 -2.28   0.18
Deal-seeker            74.34              0.33         50.04       3.89   35.76  2.92   0.19
Organic loyalist       24.09              0.67        101.36       7.12   37.06 -0.26   0.20
Passive                23.73              0.33         50.76       3.85   36.43 -0.13   0.19
                  pct_customers  pct_volume  pct_cashback_bill  volume_per_cashback
cashback_segment                                                                   
Real engager              25.40       33.71              50.71                 0.66
Deal-seeker               25.26       16.99              25.52                 0.67
Organic loyalist          24.74       33.34              16.28                 2.05
Passive               

### Who gets the cashback, and what do they bring?

**Procedure**
Cell 7 divided customers into four cashback segments by crossing how much cashback they redeem (heavy or light) with how engaged they are (high or low): real engagers, deal-seekers, organic loyalists and passives. We profiled each segment, then compared its share of customers, of payment volume and of the estimated cashback bill (using the assumed Amazon Pay ICICI rates and spend mix). "Volume per unit of cashback" divides the volume share by the bill share (1.0 = fair share).

**Observations**
- **Real engagers and deal-seekers redeem the same amount of cashback (74%) but differ in activity,** not in loyalty. Real engagers make about 101 transactions a month vs 50 for deal-seekers, with app engagement of 7.2 vs 3.9. Their tenure (36.7 vs 35.8 months), NPS (−2.3 vs +2.9) and churn (18% vs 19%) are alike.
- **Heavy redeemers take most of the bill without bringing more volume.** Real engagers and deal-seekers are 51% of customers and 51% of volume, but **76% of the cashback bill.**
- **Real engagers are the largest cost centre:** 25% of customers, 34% of volume, 51% of the bill (0.66 volume per unit of cashback).
- **Deal-seekers cost more than they bring:** 25% of customers, 17% of volume, 26% of the bill (0.67).
- **Organic loyalists are as valuable as real engagers at a third of the cost:** 33% of volume for only 16% of the bill (2.05). High engagement does not require heavy cashback redemption.
- **Passives are cheap and small:** 16% of volume for 7% of the bill (2.13).
- **Business meaning:** because cashback redemption is unrelated to engagement (Cell 16), a large part of the cashback paid to real engagers may be rewarding behaviour that would occur anyway. This needs confirming with a test. Deal-seekers receive a quarter of the bill without a loyalty advantage.
- **Caveat:** cashback cost relies on the assumed ICICI card rates and spend mix, so we compare shares, not absolute rupee amounts.

In [26]:
# ===== CELL 17D: HOW MUCH OF THE 76% IS ARITHMETIC, AND HOW SURE ARE WE? =====
# BIZ: "heavy redeemers take 76% of the bill" is the number the whole savings case rests on. Two honest questions:
#   (1) Is it a discovery, or does it follow from how we defined "heavy"?   (2) How much could it move by chance?
# TECH: bill = spend x rate x redemption%. Heavy redeemers are DEFINED as redemption above the median, so their
# share of the bill is mostly their share of redemption. We test that by ignoring spend and rate entirely.
heavy = HEAVY.values
cost  = fe.est_cashback_cost.values
earn  = (fe.monthly_spend * fe.cashback_rate).values
spend = fe.monthly_spend.values
red   = fe.cashback_used_pct.values

print("Average redemption: heavy {:.0f}% vs light {:.0f}%".format(red[heavy].mean(), red[~heavy].mean()))
print("Heavy share of the bill (spend x rate x redemption): {:.1%}".format(cost[heavy].sum() / cost.sum()))
print("Heavy share of redemption points alone (no spend, no rate): {:.1%}".format(red[heavy].sum() / red.sum()))
print("Heavy share of earned cashback / of spend: {:.1%} / {:.1%}".format(earn[heavy].sum() / earn.sum(),
                                                                          spend[heavy].sum() / spend.sum()))

# TECH: bootstrap = re-draw 5,000 customers WITH replacement 1,000 times and recompute the number each time.
# The 2.5th and 97.5th percentiles give a 95% interval for how much sampling luck could move it.
rng = np.random.default_rng(SEED); n = len(fe); out = []
for _ in range(1000):
    ix = rng.integers(0, n, n); h = heavy[ix]
    out.append((cost[ix][h].sum() / cost[ix].sum(),                       # share of redeemed bill
                cost[ix][h].sum() / spend[ix][h].sum() * 100,             # cost per Rs100, heavy group
                earn[ix][h].sum() / earn[ix].sum()))                      # share of earned cashback
out = np.array(out); lo, hi = np.percentile(out, [2.5, 97.5], axis=0)
print("\n95% bootstrap intervals:")
print(f"  Heavy share of redeemed bill : {lo[0]:.1%} to {hi[0]:.1%}   -> 25% cut saves {0.25*lo[0]:.1%} to {0.25*hi[0]:.1%}")
print(f"  Heavy share of earned        : {lo[2]:.1%} to {hi[2]:.1%}   -> 25% cut saves {0.25*lo[2]:.1%} to {0.25*hi[2]:.1%}")
print(f"  Heavy cost per Rs100         : Rs{lo[1]:.2f} to Rs{hi[1]:.2f}")

# BIZ: what is NOT arithmetic is whether heavy redeemers are more loyal. That is a real test of the data.
from scipy.stats import chi2_contingency
ch_h, ch_l = fe.churned[HEAVY].mean(), fe.churned[~HEAVY].mean()
pv = chi2_contingency(pd.crosstab(HEAVY, fe.churned), correction=False)[1]
se = np.sqrt(ch_h*(1-ch_h)/HEAVY.sum() + ch_l*(1-ch_l)/(~HEAVY).sum())
print(f"\nChurn: heavy {ch_h:.1%} vs light {ch_l:.1%} (gap {100*(ch_h-ch_l):+.1f} pts, "
      f"95% CI {100*(ch_h-ch_l-1.96*se):+.1f} to {100*(ch_h-ch_l+1.96*se):+.1f}; p = {pv:.2f})")

Average redemption: heavy 74% vs light 24%
Heavy share of the bill (spend x rate x redemption): 76.2%
Heavy share of redemption points alone (no spend, no rate): 76.1%
Heavy share of earned cashback / of spend: 50.8% / 50.7%

95% bootstrap intervals:
  Heavy share of redeemed bill : 74.7% to 77.6%   -> 25% cut saves 18.7% to 19.4%
  Heavy share of earned        : 49.0% to 52.6%   -> 25% cut saves 12.3% to 13.2%
  Heavy cost per Rs100         : Rs1.41 to Rs1.45

Churn: heavy 18.4% vs light 19.6% (gap -1.1 pts, 95% CI -3.3 to +1.0; p = 0.30)


### Procedure
1. Split the bill into its parts (bill = spend x rate x redemption) and compared the heavy redeemers' share of the bill with their share of redemption alone, ignoring spend and rate.
2. Re-drew the 5,000 customers 1,000 times with replacement (a **bootstrap**) to see how far the headline shares could move by sampling luck alone. The middle 95% of results is the interval.
3. Tested the part that is not arithmetic: are heavy redeemers more loyal? We compared their churn with light redeemers.

### Observations
- **The 76% is mostly arithmetic, not a discovery.** Heavy redeemers are defined as redemption above the median, and they redeem 74% of earned cashback against 24% for light redeemers. Their share of redemption alone is 76.1%, almost identical to their 76.2% share of the bill. They hold 50.7% of spend and 50.8% of earned cashback, so spend and rate add almost nothing. We therefore present 76% as where the money goes, not as a surprising result.
- **The real finding is what that money buys.** Heavy redeemers churn at 18.4% against 19.6% for light redeemers (gap -1.1 points, 95% interval -3.3 to +1.0, p = 0.30). Together with Cell 16 (redemption is unrelated to engagement, NPS, tenure and churn), three times the payout brings no measurable loyalty.
- **Sampling luck moves the numbers very little.** The heavy share of the redeemed bill is 74.7% to 77.6%, so a 25% cut saves 18.7% to 19.4%. On the earned basis the share is 49.0% to 52.6%, a saving of 12.3% to 13.2%. Cost per ₹100 is ₹1.41 to ₹1.45. The 13% to 19% range therefore comes from the choice of basis (earned or redeemed), not from chance.
- **What the interval does not cover:** the assumed rate card, the assumption that `cashback_used_pct` is the share of earned cashback redeemed, and customer reaction to a cut. The saving also assumes volume stays unchanged (see Cell 19).
- **Business reading:** the case for re-tiering rests on "cashback is concentrated and buys no measurable loyalty", which the data supports, and not on 76% being surprising.

### Q4 Conclusion: Is cashback driving real engagement or only deal-seeking behaviour?

**Answer:** In this data, cashback is not associated with real engagement. Cashback redemption shows no association with engagement, spending or loyalty: heavy and light redeemers are statistically indistinguishable on all three. Heavy redeemers are an even mix of highly engaged customers and low-engagement "deal-seekers", and the two groups look alike on loyalty.

| Question | Finding |
|---|---|
| Does redeeming more cashback go with higher engagement, spend or loyalty? | No: all correlations within ±0.02; light and heavy redeemers have the same engagement (0.499 vs 0.501) |
| How many heavy redeemers are "real engagers"? | About half (1,270 real engagers vs 1,263 deal-seekers). This is built in: redemption and engagement were each split at the median and are unrelated, so each group comes out near 25% |
| Where does the cashback bill go? | 76% goes to heavy redeemers, who bring 51% of volume (largely by definition, Cell 17D), and their churn is no lower (18.4% vs 19.6%) |
| Which segment is the largest cost centre? | Real engagers: 51% of the bill for 34% of volume |
| Is high engagement possible without heavy cashback? | Yes: organic loyalists match real engagers' volume at a third of the cost |

**Recommendations**
1. **Stop treating cashback redemption as a sign of loyalty.** It is unrelated to engagement, spend, tenure, NPS or churn in this data.
2. **Test cashback reductions on real engagers and deal-seekers** (for example, a lower or tiered cashback rate for a random subset) and measure whether engagement, spend and retention change. Because redemption does not track engagement, the bill may be paying for behaviour that would happen anyway.
3. **Tie rewards to behaviour,** such as transaction milestones or consistent monthly activity, instead of blanket cashback, so spend rewards real engagement.
4. **Keep organic loyalists and passives as they are:** they are low-cost, and over-incentivising them is not needed.

**Caveats**
- This is an association across customers, not proof of what cashback does for an individual; the A/B test is what settles it.
- The cashback bill uses assumed ICICI card rates and an assumed spend mix, so we compare shares, not absolute rupees.
- Segments combine redemption with an engagement index that includes transactions, so some differences between segments are built in.

In [27]:
# ===== CELL 17A: DO CUSTOMERS FORM NATURAL CLUSTERS? (DIAGNOSTIC) =====
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

# BIZ: before building personalisation on clusters, check whether customers genuinely fall into groups.
# If they do not, any clusters are arbitrary slices and the actions built on them would be unreliable.
# TECH: StandardScaler puts every feature on the same scale (mean 0, spread 1), so rupees and scores
# count equally. K-means measures distance, so unscaled features would let the biggest numbers dominate.
CLUSTER_FEATURES = ["avg_monthly_transactions", "avg_transaction_value_inr", "cashback_used_pct",
                    "app_engagement_score", "customer_support_contacts", "net_promoter_score", "tenure_months"]
X = StandardScaler().fit_transform(fe[CLUSTER_FEATURES])
rng = np.random.default_rng(SEED)

# --- Test 1: Hopkins statistic = "is the data clumped, or evenly spread?" ---
# TECH: compare distances from real customers to their nearest neighbour with distances from random
# points to their nearest customer. About 0.5 = evenly spread (no clusters); closer to 1 = clumped.
def hopkins(X, m=500, repeats=5):
    n, d = X.shape
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    scores = []
    for _ in range(repeats):
        pick = rng.choice(n, m, replace=False)
        w = nn.kneighbors(X[pick], 2)[0][:, 1]                  # real customer -> nearest other customer
        U = rng.uniform(X.min(axis=0), X.max(axis=0), (m, d))   # random points in the same space
        u = nn.kneighbors(U, 1)[0][:, 0]                        # random point -> nearest customer
        scores.append(u.sum() / (u.sum() + w.sum()))
    return float(np.mean(scores))

# CONTROL: the same test on pure random noise of the same shape shows what "no structure" looks like.
noise = StandardScaler().fit_transform(rng.uniform(0, 1, X.shape))
print(f"Hopkins statistic - our data: {hopkins(X):.3f} | pure random noise: {hopkins(noise):.3f}  (0.5 = no cluster structure)")

# --- Tests 2 and 3: for k = 2..8 clusters ---
# TECH: silhouette score (-1 to 1) = how much closer each customer is to its own cluster than to the
# next one. About 0.5+ = clear clusters; about 0.1 = overlapping blobs. We compare with the noise control.
# TECH: STABILITY = re-draw the customers (bootstrap), re-cluster, and measure how well the new clusters
# match the original ones (Adjusted Rand Index: 1 = identical, 0 = no better than chance).
# BIZ: personalisation needs groups that stay the same; unstable clusters cannot carry a fixed action.
rows = []
for k in range(2, 9):
    km = KMeans(k, n_init=10, random_state=SEED).fit(X)
    sil_real = silhouette_score(X, km.labels_, sample_size=3000, random_state=1)
    km_n = KMeans(k, n_init=10, random_state=SEED).fit(noise)
    sil_noise = silhouette_score(noise, km_n.labels_, sample_size=3000, random_state=1)
    aris = []
    for _ in range(5):
        i = rng.choice(len(X), len(X))
        again = KMeans(k, n_init=5, random_state=int(rng.integers(1_000_000))).fit(X[i])
        aris.append(adjusted_rand_score(km.labels_, again.predict(X)))
    rows.append({"k": k, "silhouette_real": sil_real, "silhouette_noise": sil_noise, "stability_ari": float(np.mean(aris))})
diag = pd.DataFrame(rows); print(diag.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ax = axes[0]
ax.plot(diag.k, diag.silhouette_real, color=C_PRIME, marker="o", lw=2.5, label="Our data")
ax.plot(diag.k, diag.silhouette_noise, color=C_NON, marker="s", lw=2, ls="--", label="Pure random noise")
ax.axhline(0.5, color="black", ls=":", lw=1); ax.text(8, 0.51, "clear structure starts around 0.5", ha="right", fontsize=9)
ax.set_ylim(0, 0.6); ax.set_xlabel("Number of clusters (k)"); ax.set_ylabel("Silhouette score")
ax.set_title("Our data clusters no better than random noise", weight="bold"); ax.legend(loc="center right")
ax = axes[1]
ax.bar(diag.k, diag.stability_ari, color=C_PRIME, width=0.6)
ax.axhline(0.8, color="black", ls=":", lw=1); ax.text(8.4, 0.82, "stable (0.8+)", ha="right", fontsize=9)
ax.set_ylim(0, 1); ax.set_xlabel("Number of clusters (k)"); ax.set_ylabel("Stability across re-samples (ARI)")
ax.set_title("Cluster assignments change when the data is re-drawn", weight="bold")
plt.tight_layout(); plt.show()

Hopkins statistic - our data: 0.495 | pure random noise: 0.499  (0.5 = no cluster structure)


 k  silhouette_real  silhouette_noise  stability_ari
 2            0.106             0.109          0.343
 3            0.095             0.098          0.134
 4            0.102             0.099          0.285
 5            0.101             0.101          0.230
 6            0.102             0.102          0.257
 7            0.104             0.106          0.248
 8            0.108             0.109          0.242


### Do customers form natural clusters? (diagnostic)

**Procedure**
Before building personalisation on clusters, we checked whether customers genuinely fall into groups. We standardised seven behavioural features (transactions, ticket size, cashback redemption, app engagement, support contacts, NPS and tenure) so each counts equally, then ran three checks, each against a control of pure random noise with the same shape:
1. **Hopkins statistic:** compares distances between real customers with distances from random points. About 0.5 means evenly spread (no clusters); closer to 1 means clumped.
2. **Silhouette score** for 2 to 8 clusters (K-means): how much closer customers are to their own cluster than to the next one. Around 0.5 or more indicates clear clusters; around 0.1 indicates overlapping blobs.
3. **Stability:** we re-drew the customers 5 times, re-clustered and measured how well the new clusters matched the originals (Adjusted Rand Index; 1 = identical, 0 = chance).

**Observations**
- **The data is as structureless as random noise.** The Hopkins statistic is 0.495 for our data vs 0.499 for pure noise (0.5 = no structure).
- **Clusters are not separated.** The silhouette score is about 0.10 for every number of clusters from 2 to 8, identical to the noise control and far below the roughly 0.5 that signals clear groups.
- **Clusters are unstable.** Stability is between 0.13 and 0.34 (it varies a little between runs), far below the 0.8 we would want. The same customer lands in different clusters each time the data is re-drawn.
- **Business meaning:** K-means will always return groups, but here they would be arbitrary slices of an evenly spread cloud. A customer labelled "Cluster 3" today could be in a different cluster on the next run, so no fixed action can be attached to it.
- **Decision:** we do not use clustering for personalisation. We use transparent business-rule segments (value, engagement, cashback redemption, satisfaction), which are stable, explainable and easy to act on.

In [28]:
# ===== CELL 17B: CAN PCA COMPRESS THE FEATURES OR REVEAL HIDDEN DIRECTIONS? (DIAGNOSTIC) =====
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import spearmanr

# TECH: PCA rotates the data to find "components": new axes ordered by how much variance they capture.
# If features move together, the first few components capture most of the variation and we can drop
# the rest. BIZ: this is how "hidden dimensions" (for example "affluent heavy user") are usually found.
PCA_FEATURES = ["age", "monthly_income_inr", "tenure_months", "avg_monthly_transactions", "avg_transaction_value_inr",
                "cashback_used_pct", "app_engagement_score", "customer_support_contacts", "net_promoter_score"]
Xp = StandardScaler().fit_transform(fe[PCA_FEATURES])    # scale first so rupee columns do not dominate
pca = PCA().fit(Xp)
real_ev = pca.explained_variance_                         # eigenvalue = variance captured by each component
share = pca.explained_variance_ratio_ * 100
cum = np.cumsum(share)

# TECH: PARALLEL ANALYSIS. After scaling, each feature has variance 1, so a component is only
# interesting if its eigenvalue is clearly above what random data produces. We shuffle each column
# independently (this destroys any real relationships), re-run PCA 200 times, and use the 95th
# percentile of those noise eigenvalues as the bar a real component must clear.
rng = np.random.default_rng(SEED)
null_ev = []
for _ in range(200):
    shuffled = np.column_stack([rng.permutation(Xp[:, j]) for j in range(Xp.shape[1])])
    null_ev.append(PCA().fit(shuffled).explained_variance_)
null95 = np.percentile(np.array(null_ev), 95, axis=0)

out = pd.DataFrame({"component": [f"PC{i+1}" for i in range(len(share))], "variance_%": share.round(1),
                    "cumulative_%": cum.round(1), "eigenvalue": real_ev.round(3),
                    "noise_95th": null95.round(3), "beats_noise": real_ev > null95})
print(out.to_string(index=False))
print("\nComponents needed for 80% / 90% of variance:", int(np.argmax(cum >= 80)) + 1, "/", int(np.argmax(cum >= 90)) + 1,
      "out of", len(share))

# BIZ: does any of the top components (the "directions" PCA found) relate to churn?
scores = pca.transform(Xp)
rows = []
for i in range(3):
    rho, p = spearmanr(scores[:, i], fe.churned)
    rows.append({"component": f"PC{i+1}", "spearman_with_churn": round(rho, 3), "p_value": round(p, 3)})
print("\n", pd.DataFrame(rows).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ax = axes[0]; x = np.arange(1, len(real_ev) + 1)
ax.plot(x, real_ev, color=C_PRIME, marker="o", lw=2.5, label="Our data")
ax.plot(x, null95, color=C_NON, marker="s", lw=2, ls="--", label="Random noise (95th percentile)")
ax.axhline(1, color="black", ls=":", lw=1); ax.text(9, 1.08, "1.0 = one feature's worth of variance", ha="right", fontsize=9)
ax.set_ylim(0, 2.5)          # wide range on purpose: a zoomed axis would make a flat curve look steep
ax.set_xlabel("Component"); ax.set_ylabel("Eigenvalue (variance explained)")
ax.set_title("No component carries more than noise would", weight="bold"); ax.legend(loc="upper right")
ax = axes[1]
ax.bar(x, share, color=C_PRIME, width=0.6); ax.plot(x, cum, color=C_COST, marker="o", lw=2)
ax.axhline(100 / len(share), color="black", ls=":", lw=1)
ax.text(len(share) + 0.4, 100 / len(share) + 2, "equal share", ha="right", fontsize=9)
ax.set_xlabel("Component"); ax.set_ylabel("% of variance (bars) and cumulative % (line)"); ax.set_ylim(0, 105)
ax.set_title("Variance is spread evenly: no compression possible", weight="bold")
plt.tight_layout(); plt.show()

component  variance_%  cumulative_%  eigenvalue  noise_95th  beats_noise
      PC1        11.9          11.9       1.072       1.082        False
      PC2        11.6          23.6       1.048       1.056        False
      PC3        11.4          34.9       1.025       1.038        False
      PC4        11.3          46.2       1.014       1.022        False
      PC5        11.1          57.3       0.998       1.010        False
      PC6        11.0          68.3       0.991       0.995        False
      PC7        10.7          78.9       0.960       0.982        False
      PC8        10.6          89.5       0.951       0.971        False
      PC9        10.5         100.0       0.944       0.952        False

Components needed for 80% / 90% of variance: 8 / 9 out of 9

 component  spearman_with_churn  p_value
      PC1                0.014    0.309
      PC2               -0.002    0.902
      PC3               -0.018    0.207


### Can PCA compress the features or reveal hidden directions? (diagnostic)

**Procedure**
PCA rotates the data into new axes ("components") ordered by how much variation they capture. If features move together, a few components capture most of the information and the rest can be dropped. We ran PCA on nine standardised numeric features. To check whether any component is meaningful, we used **parallel analysis**: shuffle each column independently (which destroys any real relationship), re-run PCA 200 times, and take the 95th percentile of those noise eigenvalues as the bar a real component must clear. We also checked whether the first three components relate to churn.

**Observations**
- **No compression is possible.** Each component captures about 11% of the variance (an equal share of nine is 11.1%). Eight of the nine components are needed to keep 80% of the information, and all nine to reach 90%.
- **No component beats noise.** Every real eigenvalue is at or just below its noise benchmark (for example PC1: 1.072 vs 1.082), so none of the nine components carries more than random data would.
- **No hidden direction relates to churn.** The first three components have correlations with churn of 0.014, −0.002 and −0.018 (p = 0.31, 0.90, 0.21). The sign of a component is arbitrary, so only the size matters.
- **Business meaning:** the features are independent of each other, so there are no hidden combined dimensions (such as "affluent heavy user") to find, and nothing to simplify. This is consistent with the near-zero correlations in Cell 5.
- **Conclusion for the report:** we tested for natural customer groups (clustering) and for hidden combined dimensions (PCA) and found neither, which is why the personalisation uses transparent business-rule segments. This also reinforces the Q1 finding that no combination of attributes explains churn.

In [29]:
# ===== CELL 17C: REWARD PULL = how strongly rewards pull a customer into the app =====
# TECH: reward_pull = (% of cashback redeemed) x (transactions per month). It rises with transactions
#       by construction, so we do NOT ask "does it track activity?" (it must).
#       We ask: does a higher reward pull bring lower churn or higher satisfaction?
# BIZ:  If heavy rewards created loyalty, high-pull customers would churn less and score higher on NPS.
from scipy.stats import chi2_contingency
C_PRIME, C_NON, C_COST = "#2B7BBF", "#7F7F7F", "#D2591A"
def mean_ci(s):
    m, se = s.mean(), s.std(ddof=1) / np.sqrt(len(s)); return m, m - 1.96 * se, m + 1.96 * se

fe["pull_band"] = pd.qcut(fe.reward_pull, q=5, labels=["Lowest 20%", "20-40%", "40-60%", "60-80%", "Highest 20%"])
q = fe.groupby("pull_band", observed=True).agg(customers=("churned", "size"), pull_from=("reward_pull", "min"), pull_to=("reward_pull", "max"),
        spend=("monthly_spend", "sum"), cost=("est_cashback_cost", "sum"), churn=("churned", "mean"),
        nps=("net_promoter_score", "mean"), engagement=("engagement_index", "mean"))
q["pct_volume"] = q.spend / q.spend.sum() * 100
q["pct_bill"] = q.cost / q.cost.sum() * 100
print(q[["customers", "pull_from", "pull_to", "pct_volume", "pct_bill", "churn", "nps", "engagement"]].round(2).to_string())
print(f"\nChurn differs across reward-pull bands? chi-square p = {chi2_contingency(pd.crosstab(fe.pull_band, fe.churned))[1]:.3f}")
print(f"Top 20% cost {q.cost.iloc[-1] / q.cost.iloc[0]:.0f}x the bottom 20% in cashback; churn {q.churn.iloc[-1]:.1%} vs {q.churn.iloc[0]:.1%}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8)); xs = np.arange(5); lab = ["Lowest\n20%", "20-40%", "40-60%", "60-80%", "Highest\n20%"]
ax = axes[0]; w = 0.38
b1 = ax.bar(xs - w/2, q.pct_volume, w, color=C_NON, label="Share of payment volume")
b2 = ax.bar(xs + w/2, q.pct_bill, w, color=C_COST, label="Share of cashback bill")
for b in list(b1) + list(b2): ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.8, f"{b.get_height():.0f}%", ha="center", fontsize=9, weight="bold")
ax.set_xticks(xs); ax.set_xticklabels(lab, fontsize=9); ax.set_ylim(0, 60); ax.legend(loc="upper left", fontsize=9)
ax.set_title("Cashback cost concentrates at the top", weight="bold", fontsize=11); ax.set_ylabel("% of total")
ax = axes[1]
for i, (b, d) in enumerate(fe.groupby("pull_band", observed=True)):
    k, n = d.churned.sum(), len(d); m = k / n; lo, hi = wilson(k, n)
    ax.errorbar(i, m, yerr=[[m - lo], [hi - m]], fmt="o", color=C_PRIME, capsize=4, ms=7, lw=2)
ax.axhline(BASE, color="black", ls="--", lw=1); ax.set_ylim(0.10, 0.30); ax.set_xticks(xs); ax.set_xticklabels(lab, fontsize=9)
ax.set_title("...but churn does not fall (95% CI)", weight="bold", fontsize=11); ax.set_ylabel("Churn rate")
ax = axes[2]
for i, (b, d) in enumerate(fe.groupby("pull_band", observed=True)):
    m, lo, hi = mean_ci(d.net_promoter_score)
    ax.errorbar(i, m, yerr=[[m - lo], [hi - m]], fmt="o", color=C_PRIME, capsize=4, ms=7, lw=2)
ax.axhline(fe.net_promoter_score.mean(), color="black", ls="--", lw=1); ax.set_ylim(-20, 20); ax.set_xticks(xs); ax.set_xticklabels(lab, fontsize=9)
ax.set_title("...and customers are not happier (95% CI)", weight="bold", fontsize=11); ax.set_ylabel("Average NPS")
fig.suptitle("Reward pull (share of cashback redeemed x transactions): cost rises steeply, loyalty and satisfaction stay flat", fontsize=12.5, weight="bold", y=1.03)
plt.tight_layout(); plt.show()

             customers  pull_from  pull_to  pct_volume  pct_bill  churn   nps  engagement
pull_band                                                                                
Lowest 20%        1003       0.00     7.20       10.08      1.72   0.19 -1.38        0.37
20-40%             999       7.22    19.84       14.94      6.92   0.19  0.26        0.43
40-60%             998      19.88    37.45       19.63     15.38   0.19  2.05        0.50
60-80%            1001      37.53    64.77       24.17     27.08   0.18 -0.91        0.55
Highest 20%        999      64.80   149.00       31.18     48.91   0.20  0.31        0.65

Churn differs across reward-pull bands? chi-square p = 0.847
Top 20% cost 28x the bottom 20% in cashback; churn 19.7% vs 19.1%


### Procedure 
1. Used `reward_pull` (share of earned cashback redeemed x transactions per month) and split customers into five equal bands, from the lowest 20% to the highest 20%.
2. For each band, measured its share of payment volume, share of the cashback bill, churn rate (with 95% confidence intervals), average NPS and average engagement.
3. Tested whether churn differs across the five bands (chi-square test).
4. Note: reward pull rises with transactions by construction, so the real question is not whether it tracks activity but whether it brings loyalty or satisfaction.

### Observations 
- **Cashback cost is highly concentrated.** The top 20% by reward pull account for 31% of payment volume but 49% of the cashback bill, and cost 28 times the bottom 20%. The bottom 20% generate 10% of volume for only 2% of the bill.
- **Churn does not fall as reward pull rises.** Churn is about 19% in every band (19.1% in the lowest, 19.7% in the highest; chi-square p = 0.85). All intervals overlap the overall average.
- **Customers are not happier either.** Average NPS stays between about -1.4 and +2.1 across bands, with every interval overlapping the overall average (zero). A bigger reward pull does not lift satisfaction.
- **Engagement rises with reward pull (0.37 to 0.65),** but that is partly built in, because the measure includes transactions per month and the engagement index does too. It shows the rewards are used by active customers, not that they make customers stay.
- **Business reading:** the customers who redeem the most cashback cost far more but are no more loyal or satisfied. Heavy rewards buy activity that is largely already there, which supports re-tiering cashback by activity milestones instead of paying more to everyone.

In [30]:
# ===== CELL 18: THE CUSTOMER PLAYBOOK (Q5) =====
import textwrap
from matplotlib.patches import FancyBboxPatch

# BIZ: personalisation means a different action per customer group. We use the four cashback segments
# (heavy/light redemption x high/low engagement) because they differ in value AND in cashback cost,
# then overlay satisfaction flags (unhappy but active, happy but idle) inside each one.
seg_stats = fe.groupby("cashback_segment").agg(customers=("churned", "size"), spend=("monthly_spend", "sum"),
                                               cost=("est_cashback_cost", "sum"))
seg_stats["pct_customers"] = seg_stats.customers / seg_stats.customers.sum() * 100
seg_stats["pct_volume"]    = seg_stats.spend / seg_stats.spend.sum() * 100
seg_stats["pct_bill"]      = seg_stats.cost / seg_stats.cost.sum() * 100
# TECH: crosstab counts how many customers fall in each segment x flag combination.
flags = pd.crosstab(fe.cashback_segment, fe.say_do_gap)
playbook = seg_stats.join(flags)
print(playbook[["customers", "pct_customers", "pct_volume", "pct_bill", "Active but unhappy", "Happy but idle"]].round(1).to_string())

# The action for each segment, based on what Cells 14 to 17 showed.
ACTIONS = {
    "Real engager":     ("Re-tier cashback", "Keep the headline reward but pay higher rates only for activity milestones. Service recovery for unhappy active customers."),
    "Deal-seeker":      ("Make cashback conditional", "Link rewards to repeat use (bill pay, recharges). Cap the redemption subsidy. Habit nudges for happy but idle customers."),
    "Organic loyalist": ("Reward without cashback", "No cashback increase. Recognise with perks (priority support, early access). Fix the experience for unhappy customers."),
    "Passive":          ("Low-cost activation", "First-use prompts and a small one-time reward. Nudge happy but idle customers."),
}
GRID = {"Deal-seeker": (0, 1), "Real engager": (1, 1), "Passive": (0, 0), "Organic loyalist": (1, 0)}   # column, row

# A 2x2 map: rows = how much cashback they redeem, columns = how engaged they are.
fig, ax = plt.subplots(figsize=(12, 8)); ax.set_xlim(0, 2); ax.set_ylim(0, 2); ax.axis("off")
for seg, (cx, cy) in GRID.items():
    row = playbook.loc[seg]; heavy_cost = seg in ["Deal-seeker", "Real engager"]
    ax.add_patch(FancyBboxPatch((cx + 0.03, cy + 0.03), 0.94, 0.94, boxstyle="round,pad=0.0,rounding_size=0.04",
                                fc="#FBE3D6" if heavy_cost else "#DDEBF7", ec="#444", lw=1.2))
    ax.text(cx + 0.5, cy + 0.88, seg, ha="center", fontsize=15, weight="bold")
    ax.text(cx + 0.5, cy + 0.79, f"{row.pct_customers:.0f}% of customers | {row.pct_volume:.0f}% of volume | "
            f"{row.pct_bill:.0f}% of cashback bill", ha="center", fontsize=9.5)
    ax.text(cx + 0.5, cy + 0.66, ACTIONS[seg][0], ha="center", fontsize=13, weight="bold", color="#222")
    ax.text(cx + 0.5, cy + 0.36, textwrap.fill(ACTIONS[seg][1], 46), ha="center", va="center", fontsize=10.5)
    parts = []                                   # show only the satisfaction flag that exists in this segment
    if row.get("Active but unhappy", 0) > 0: parts.append(f"Unhappy but active: {int(row['Active but unhappy']):,}")
    if row.get("Happy but idle", 0) > 0:     parts.append(f"Happy but idle: {int(row['Happy but idle']):,}")
    ax.text(cx + 0.5, cy + 0.10, " | ".join(parts), ha="center", fontsize=9, color="#555")
ax.text(1, 2.04, "Customer playbook (orange = high cashback cost, blue = low)", ha="center", fontsize=14, weight="bold")
ax.text(0.5, -0.04, "Lower engagement", ha="center", fontsize=11); ax.text(1.5, -0.04, "Higher engagement", ha="center", fontsize=11)
ax.text(-0.04, 0.5, "Light redeemers", va="center", rotation=90, fontsize=11)
ax.text(-0.04, 1.5, "Heavy redeemers", va="center", rotation=90, fontsize=11)
plt.tight_layout(); plt.show()

                  customers  pct_customers  pct_volume  pct_bill  Active but unhappy  Happy but idle
cashback_segment                                                                                    
Deal-seeker            1263           25.3        17.0      25.5                   0             439
Organic loyalist       1237           24.7        33.3      16.3                 413               0
Passive                1230           24.6        16.0       7.5                   0             419
Real engager           1270           25.4        33.7      50.7                 448               0


### Procedure
1. Split customers into four behaviour segments using two things: how much cashback they redeem (heavy vs light) and whether their activity looks genuine (engaged vs not).
2. For each segment, measured its share of customers, its share of payment volume and its share of the total cashback bill.
3. Flagged two mismatches inside each segment: "active but unhappy" (high activity, low NPS) and "happy but idle" (high NPS, low activity).
4. Attached one action to each segment, drawn as a 2x2 map (orange = high cashback cost, blue = low).

### Observations
- **The four segments are equal in size (about 25% of customers each), but not in cost or value.**
- **Real engagers** (heavy redeemers who are also active) carry 34% of volume but 51% of the cashback bill. They are the biggest cost, and the action is to re-tier cashback so higher rates are paid only for activity milestones.
- **Deal-seekers** (heavy redeemers with weak activity) take 26% of the bill for only 17% of volume. This is the weakest return, and the action is to make cashback conditional on repeat use.
- **Organic loyalists** (light redeemers, high activity) bring 33% of volume for only 16% of the bill. They are the most profitable group, and the action is to reward them without more cashback (perks, priority support, early access).
- **Passive customers** use the app little and cost little (7.5% of the bill). The action is a low-cost activation nudge, not a cashback increase.
- **Experience gaps exist within segments:** 413 organic loyalists and 448 real engagers are active but unhappy, which is a service-recovery risk. 439 deal-seekers and 419 passive customers are happy but idle, which is a cheap re-activation opportunity.
- **Business reading:** cashback is concentrated where it buys the least. Half the bill goes to real engagers, and a quarter goes to deal-seekers who add little volume.

In [31]:
# ===== CELL 18A: TARGETING RULES FOR THE PLAYBOOK =====
from scipy.stats import mannwhitneyu
C_PRIME, C_COST = "#2B7BBF", "#D2591A"

# 1) Can support logs find the "active but unhappy" customers?
# BIZ: If unhappy customers contacted support more, we could find them from service logs for free.
gap = fe.groupby("say_do_gap").agg(customers=("churned", "size"), avg_support_contacts=("customer_support_contacts", "mean"),
                                    avg_friction_per_year=("friction_per_year", "mean"), churn=("churned", "mean"))
print(gap.round(2).to_string())
ua = fe.say_do_gap == "Active but unhappy"
p_ua = mannwhitneyu(fe.loc[ua, "customer_support_contacts"], fe.loc[~ua, "customer_support_contacts"], alternative="two-sided")[1]
print(f"Support contacts, unhappy-but-active vs everyone else: p = {p_ua:.3f}")

# 2) Concentration curve: how few customers drive the bill?
# TECH: rank customers from most to least costly and plot the running share of cost and volume
#       (a Lorenz / Pareto curve). The more it bows away from the diagonal, the more concentrated.
ranked = fe.sort_values("est_cashback_cost", ascending=False)
cum_bill = ranked.est_cashback_cost.cumsum().values / ranked.est_cashback_cost.sum()
cum_vol = ranked.monthly_spend.cumsum().values / ranked.monthly_spend.sum()
share = np.arange(1, len(ranked) + 1) / len(ranked)
for s in [0.2, 0.4]:
    i = int(s * len(ranked)) - 1
    print(f"Top {s:.0%} of customers by cashback cost: {cum_bill[i]:.0%} of the bill, {cum_vol[i]:.0%} of volume")

# 3) A simple, operational targeting rule: reward pull (top 20%)
top = fe.reward_pull >= fe.reward_pull.quantile(0.8)
print(f"Top 20% by reward pull: {fe.est_cashback_cost[top].sum() / fe.est_cashback_cost.sum():.0%} of the bill | "
      f"churn {fe.churned[top].mean():.1%} vs {fe.churned[~top].mean():.1%} for the rest")
print(pd.crosstab(top.map({True: "Top 20% reward pull", False: "Rest"}), fe.cashback_segment, normalize="index").round(2).to_string())

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.plot(share * 100, cum_bill * 100, color=C_COST, lw=2.8); ax.plot(share * 100, cum_vol * 100, color=C_PRIME, lw=2.8)
ax.plot([0, 100], [0, 100], color="#999", ls="--", lw=1)
ax.text(22, 92, "Cashback bill", color="#222", fontsize=11, weight="bold"); ax.text(70, 66, "Payment volume", color="#222", fontsize=11, weight="bold")
ax.axvline(20, color="black", ls=":", lw=1)
ax.set_xlabel("Customers, ranked from highest to lowest cashback cost (%)"); ax.set_ylabel("Cumulative share of total (%)")
ax.set_title("A fifth of customers drive most of the cashback bill", weight="bold")
plt.tight_layout(); plt.show()

                    customers  avg_support_contacts  avg_friction_per_year  churn
say_do_gap                                                                       
Active but unhappy        861                  4.56                   3.73   0.19
Aligned                  3281                  4.47                   3.58   0.19
Happy but idle            858                  4.76                   3.92   0.18
Support contacts, unhappy-but-active vs everyone else: p = 0.773
Top 20% of customers by cashback cost: 59% of the bill, 41% of volume
Top 40% of customers by cashback cost: 84% of the bill, 66% of volume
Top 20% by reward pull: 49% of the bill | churn 19.7% vs 18.8% for the rest
cashback_segment     Deal-seeker  Organic loyalist  Passive  Real engager
reward_pull                                                              
Rest                        0.27              0.31     0.31          0.12
Top 20% reward pull         0.19              0.01     0.00          0.79


### Procedure 
1. Compared support contacts and `friction_per_year` for the three satisfaction groups ("active but unhappy", "aligned", "happy but idle"), with a two-sided Mann-Whitney test of unhappy-but-active customers against everyone else.
2. Ranked all customers from most to least cashback cost and plotted the running share of the cashback bill and of payment volume (a concentration curve).
3. Tested a simple targeting rule: the top 20% of customers by reward pull. Measured their share of the bill, their churn compared with everyone else, and which playbook segments they come from.

### Observations 
- **Support logs cannot find unhappy customers.** The 861 active-but-unhappy customers made 4.56 support contacts on average, against 4.47 for aligned customers (p = 0.77). Their friction per year is also similar (3.73 vs 3.58), and churn is about 19% in every group. Service recovery must be triggered by NPS surveys or feedback, not by support volume.
- **A fifth of customers drive most of the bill.** The top 20% of customers by cashback cost account for 59% of the cashback bill but only 41% of volume, and the top 40% account for 84% of the bill. The gap between the orange and blue curves is the cost that is not matched by volume.
- **Reward pull gives a simple targeting rule.** The top 20% by reward pull take 49% of the bill, and 79% of them are real engagers (19% are deal-seekers). Only 1% are organic loyalists and none are passive. This means a reward-pull ranking picks out the segments the playbook wants to re-tier, while leaving loyalists and light users alone.
- **Churn in the targeted group is not lower.** It is 19.7% vs 18.8% for everyone else, so cutting their cashback does not remove a group that is currently being retained by it (this is correlational, so the A/B test is the proof).
- **Business reading:** the first group to test a cashback change on is the top reward-pull fifth, found by a one-line rule. Unhappy customers have to be found with NPS, since support data does not reveal them.

In [32]:
# ===== CELL 19: SMART CASHBACK WHAT-IF (SAVING VS RISK) =====
# BIZ: heavy redeemers (real engagers + deal-seekers) get 76% of the bill. What if we cut their
# cashback by 10%, 25% or 50%? We cannot observe how customers would react, so instead of guessing we
# ask: HOW MUCH could they cut back their spending before the cut stops paying for itself?
HEAVY = fe.cashback_segment.isin(["Real engager", "Deal-seeker"])
S_h = fe.loc[HEAVY, "monthly_spend"].sum(); C_h = fe.loc[HEAVY, "est_cashback_cost"].sum()
c = C_h / S_h                                          # cashback paid per Rs1 of spend for this group
bill_share = C_h / fe.est_cashback_cost.sum()
CUTS = [0.10, 0.25, 0.50]                               # ILLUSTRATIVE cut sizes
REVENUE_GRID = np.arange(0.010, 0.0351, 0.0025)         # ILLUSTRATIVE revenue per Rs1 of spend (Cell 15)
REINVEST_SHARE = 0.30                                   # ASSUMPTION: share of savings put back into non-cash perks
print(f"Heavy redeemers: {HEAVY.sum():,} customers | cashback per Rs100 spent: {c*100:.2f} | share of bill: {bill_share:.0%}")

# TECH: contribution per Rs of spend = revenue rate r minus cashback c. After a cut of x, cashback falls to
# c(1-x). If volume falls by a fraction L, contribution changes from S(r - c) to S(1-L)(r - c(1-x)).
# Setting the two equal gives the BREAK-EVEN volume loss  L* = c*x / (r - c(1-x)).
# If r is still below c(1-x), the group loses money on every rupee even after the cut, so any volume loss
# only helps and L* = 100%.
def breakeven_loss(r, c, x):
    c_new = c * (1 - x)
    if r - c_new <= 0:
        return 1.0
    return min(1.0, c * x / (r - c_new))

tbl = pd.DataFrame({f"{int(x*100)}% cut": [breakeven_loss(r, c, x) for r in REVENUE_GRID] for x in CUTS},
                   index=[f"{r:.2%}" for r in REVENUE_GRID])
tbl.index.name = "revenue per Rs100"
print((tbl * 100).round(0).astype(int).astype(str).add("%").to_string())

# BIZ: saving if volume is unchanged = cut size x the heavy redeemers' share of the total bill.
saving = pd.DataFrame({"gross_saving_%_of_bill": [x * bill_share * 100 for x in CUTS]},
                      index=[f"{int(x*100)}% cut" for x in CUTS])
saving["net_after_reinvest_%"] = saving["gross_saving_%_of_bill"] * (1 - REINVEST_SHARE)
print(saving.round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
ax = axes[0]
styles = {0.10: (C_NON, ":", "10% cut"), 0.25: (C_PRIME, "-", "25% cut"), 0.50: (C_NON, "--", "50% cut")}
for x in CUTS:
    col, ls, name = styles[x]
    ax.plot(REVENUE_GRID * 100, tbl[f"{int(x*100)}% cut"].values * 100, color=col, ls=ls, lw=2.5, marker="o", ms=5)
    ax.text(REVENUE_GRID[-1] * 100 + 0.04, tbl[f"{int(x*100)}% cut"].values[-1] * 100, name, color="#222", va="center", fontsize=10)
ax.axvline(c * 100, color="black", ls="--", lw=1); ax.text(c * 100 + 0.02, 5, f"break-even\n{c*100:.2f}%", fontsize=9)
ax.set_xlim(0.95, 3.9); ax.set_ylim(0, 105)
ax.set_xlabel("Revenue per Rs100 of spend (%)"); ax.set_ylabel("Max volume loss the cut can absorb (%)")
ax.set_title("A cut pays for itself unless customers react more than this", weight="bold")
ax = axes[1]
x_ = np.arange(len(CUTS)); w = 0.35
b1 = ax.bar(x_ - w/2, saving.iloc[:, 0], w, color=C_PRIME, label="Gross saving")
b2 = ax.bar(x_ + w/2, saving.iloc[:, 1], w, color=C_NON, label=f"After reinvesting {REINVEST_SHARE:.0%}")
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.6, f"{b.get_height():.0f}%", ha="center", fontsize=10, weight="bold")
ax.set_xticks(x_); ax.set_xticklabels([f"{int(x*100)}% cashback cut" for x in CUTS]); ax.set_ylim(0, 45)
ax.set_ylabel("% of the current total cashback bill"); ax.set_title("Saving if volume is unchanged", weight="bold"); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()


Heavy redeemers: 2,533 customers | cashback per Rs100 spent: 1.43 | share of bill: 76%
                  10% cut 25% cut 50% cut
revenue per Rs100                        
1.00%                100%    100%    100%
1.25%                100%    100%    100%
1.50%                 67%     84%     91%
1.75%                 31%     53%     69%
2.00%                 20%     39%     56%
2.25%                 15%     30%     47%
2.50%                 12%     25%     40%
2.75%                 10%     21%     35%
3.00%                  8%     19%     31%
3.25%                  7%     16%     28%
3.50%                  6%     15%     26%
         gross_saving_%_of_bill  net_after_reinvest_%
10% cut                     7.6                   5.3
25% cut                    19.1                  13.3
50% cut                    38.1                  26.7


### Procedure
1. Grouped the two heavy-redeemer segments (real engagers + deal-seekers): 2,533 customers.
2. Simulated cutting their cashback rate by 10%, 25% and 50%, and measured the gross saving as a share of the total cashback bill.
3. Held back 30% of the saving for reinvestment (for example, perks for loyal customers) to get a net saving.
4. Calculated the break-even volume loss: the largest drop in these customers' spending the company could absorb before the cut stops paying for itself. This depends on how much revenue Amazon Pay earns per ₹100 spent, which the data does not contain, so the table shows a range of rates (1.0% to 3.5%).

### Observations
- **Heavy redeemers are 2,533 customers (about half the base) and take 76% of the cashback bill,** at ₹1.43 of cashback per ₹100 spent (base-case rates).
- **If volume does not change, the savings are large:** a 10% cut saves 7.6% of the total bill, a 25% cut saves 19.1% (13.3% after reinvesting 30%), and a 50% cut saves 38.1% (26.7% after reinvesting).
- **Break-even logic:** if revenue per ₹100 is about ₹1.25 or less (below the cashback cost of ₹1.43), the cut pays off whatever happens to volume, because these customers are barely or not profitable already.
- **Above that, the tolerance shrinks as revenue per ₹100 rises.** For a 25% cut, heavy redeemers can reduce their spending by about 84% at ₹1.50 revenue, 39% at ₹2.00, 25% at ₹2.50 and 19% at ₹3.00 before the cut stops paying off. For a 10% cut the tolerance is smaller (20% at ₹2.00, 8% at ₹3.00), and for a 50% cut it is larger (56% at ₹2.00, 31% at ₹3.00).
- **The corrected rates make the case slightly less generous than before.** With the lower cost base (₹1.43 instead of ₹1.83 per ₹100), each rupee cut saves less, so at the same revenue rate the tolerance is smaller (for a 25% cut at ₹3.00 revenue, 19% instead of 28%). A practical rule is: a 25% cut is safe if heavy redeemers lose less than about 20% of their spending, unless revenue per ₹100 is above ₹3.
- **Cautions:** the revenue rates are illustrative (the data has no revenue or margin), spend is inflated for many customers, and the saving and break-even above are on the redeemed basis. Cell 19A repeats them on the earned basis, where the saving is 13%.
- **Business reading:** a moderate cut on heavy redeemers is worth testing. It pays off for any plausible revenue rate unless customers reduce their spending sharply, which is exactly what the A/B test in Cell 20 checks.

In [33]:
# ===== CELL 19A: SAME-BASIS CHECK, REDEEMED vs EARNED CASHBACK =====
# BIZ: Cell 19 prices the cashback the company pays out TODAY (redeemed). But points do not expire, so
# cashback that is earned and not yet redeemed is still a future cost. A fair decision uses ONE basis for
# both the saving and the break-even. This cell reports both side by side.
# TECH: earned = spend x rate (what customers have accrued); redeemed = earned x share of cashback used.
earned_all   = fe.monthly_spend * fe.cashback_rate
red_all      = fe.est_cashback_cost
heavy        = HEAVY.values

bases = {
    "Redeemed (cash-out now)":   red_all,
    "Earned (incl. unused points)": earned_all,
}
rows = []
for name, cost in bases.items():
    c_b   = cost[heavy].sum() / fe.monthly_spend[heavy].sum()          # cost per Rs1 of spend, heavy group
    share = cost[heavy].sum() / cost.sum()                              # heavy group's share of the bill
    row = {"basis": name, "cost per Rs100": round(c_b * 100, 2), "heavy share of bill": f"{share:.0%}",
           "25% cut saves": f"{0.25 * share:.1%}", "after 30% reinvested": f"{0.25 * share * 0.7:.1%}"}
    # BIZ: the volume loss a 25% cut can absorb at three illustrative revenue rates
    for r in (0.020, 0.025, 0.030):
        row[f"absorbs at Rs{r*100:.1f} revenue"] = f"{breakeven_loss(r, c_b, 0.25):.0%}"
    row["cut pays whatever volume does if revenue <="] = f"Rs{c_b*100:.2f}"
    rows.append(row)
same_basis = pd.DataFrame(rows).set_index("basis")
from IPython.display import display
display(same_basis.T)       # transposed so the two bases sit side by side

basis,Redeemed (cash-out now),Earned (incl. unused points)
cost per Rs100,1.43,1.93
heavy share of bill,76%,51%
25% cut saves,19.1%,12.7%
after 30% reinvested,13.3%,8.9%
absorbs at Rs2.0 revenue,39%,87%
absorbs at Rs2.5 revenue,25%,46%
absorbs at Rs3.0 revenue,19%,31%
cut pays whatever volume does if revenue <=,Rs1.43,Rs1.93


### Procedure
1. Repeated the saving and the break-even on two bases: **redeemed** cashback (what is paid out now) and **earned** cashback (what customers have accrued, including unused points that do not expire).
2. For each, calculated the heavy redeemers' cost per ₹100, the saving from a 25% cut, and the volume loss a 25% cut can absorb at ₹2.0, ₹2.5 and ₹3.0 revenue per ₹100.

### Observations
- **Redeemed basis:** cost is ₹1.43 per ₹100. A 25% cut saves 19.1% of the bill (13.3% after reinvesting 30%) and can absorb a spending fall of 39%, 25% and 19% at ₹2.0, ₹2.5 and ₹3.0 revenue.
- **Earned basis:** cost is ₹1.93 per ₹100. A 25% cut saves 12.7% (8.9% after reinvesting) and can absorb a fall of 87%, 46% and 31% at the same revenue rates.
- **Why the two bases differ:** the earned cost base is larger, so each rupee cut frees more margin and the cut tolerates more lost volume. The break-even revenue also moves up, from ₹1.43 to ₹1.93. Below that revenue the group loses money on every rupee, so a cut pays off whatever happens to volume.
- **Use matching pairs.** Quote 19% with the redeemed column and 13% with the earned column. Mixing them (for example 13% with a 39% tolerance) understates the case, and mixing the other way (19% with 87%) overstates it. For the decision rule we use the redeemed column, which is the more demanding.
- **Caveat:** the earned basis assumes every unused point is eventually redeemed, so it is an upper bound on the liability. The truth sits between the two columns.

In [34]:
# ===== CELL 20: A/B TEST DESIGN =====
from scipy.stats import norm
import textwrap

# BIZ: there is no history of how customers react to cashback changes, so the plan must be proven by an
# experiment. This cell sizes it and writes down the design and decision rule BEFORE running it.
# TECH: significance level 5% (false-alarm risk) and power 80% (chance of catching a real effect).
# Sample size = how many customers per group we need to reliably detect an effect of a given size.
ALPHA, POWER = 0.05, 0.80
z = norm.ppf(1 - ALPHA / 2) + norm.ppf(POWER)
base_churn = fe.churned.mean(); sd_spend, mu_spend = fe.monthly_spend.std(), fe.monthly_spend.mean()

def n_churn(delta):       # two-proportion test: churn falls from 19% to (19% - delta)
    p1, p2 = base_churn, base_churn - delta
    return int(np.ceil(z**2 * (p1*(1-p1) + p2*(1-p2)) / delta**2))
def n_spend(pct):         # two-sample test on average spend, detecting a change of pct of the mean
    return int(np.ceil(2 * z**2 * sd_spend**2 / (pct * mu_spend)**2))

churn_deltas = [0.01, 0.02, 0.03, 0.05]; spend_pcts = [0.05, 0.10, 0.15]
print(pd.DataFrame({"churn change detected (pts)": [d*100 for d in churn_deltas],
                    "customers per arm": [n_churn(d) for d in churn_deltas]}).to_string(index=False))
print(pd.DataFrame({"spend change detected (%)": [p*100 for p in spend_pcts],
                    "customers per arm": [n_spend(p) for p in spend_pcts]}).to_string(index=False))

# BIZ: how small an effect could OUR data detect if we split the heavy redeemers into two groups?
avail = int(HEAVY.sum() / 2)
mde_spend = z * np.sqrt(2) * sd_spend / (mu_spend * np.sqrt(avail))
mde_churn = next(d for d in np.arange(0.005, 0.2, 0.0005) if n_churn(d) <= avail)
print(f"\nHere ({HEAVY.sum():,} heavy redeemers, {avail:,} per arm) we could only detect: "
      f"spend change ~{mde_spend:.0%} or churn change ~{mde_churn*100:.1f} pts")

# BIZ: the guardrail that matters most is churn going UP after a cut. A rise from 19% is slightly harder to
# detect than a fall, because the variance is larger at a higher churn rate.
def n_churn_up(delta):
    p1, p2 = base_churn, base_churn + delta
    return int(np.ceil(z**2 * (p1*(1-p1) + p2*(1-p2)) / delta**2))
mde_churn_up = next(d for d in np.arange(0.005, 0.2, 0.0005) if n_churn_up(d) <= avail)
print(f"A RISE in churn needs ~{mde_churn_up*100:.1f} pts to be detected, so the guardrail is blunt at this size.")

design = pd.DataFrame([
 ("Population", "Heavy redeemers (real engagers + deal-seekers)"),
 ("Arms", "Control (current cashback) | A: 25% lower cashback rate | B: higher rate only after activity milestones"),
 ("Assignment", "Random, stratified by cashback segment; same dates for all arms"),
 ("Duration", "8 to 12 weeks (assumption), plus 90-day retention read"),
 ("Primary metric", "Net contribution per customer (volume x revenue per Rs100 - cashback paid)"),
 ("Secondary metrics", "Transactions per month, spend, 90-day retention"),
 ("Guardrails", "NPS, support contacts, complaints, share of customers cutting activity"),
 ("Decision rule", "Roll out if saving holds, volume loss stays below the break-even in Cell 19 and churn rises by at most 1 point"),
], columns=["item", "design"])

for item, text in design.itertuples(index=False):
    print(f"{item:<18}" + ("\n" + " " * 18).join(textwrap.wrap(text, 80)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]; ds = np.arange(0.01, 0.0801, 0.0025)
ax.plot(ds * 100, [n_churn(d) for d in ds], color=C_PRIME, lw=2.5); ax.set_yscale("log")
ax.axhline(avail, color=C_COST, ls="--", lw=1.5)
ax.text(7.9, avail * 1.1, f"available here: {avail:,} per arm", ha="right", fontsize=10, color="#222")
ax.set_xlabel("Smallest churn change to detect (percentage points)"); ax.set_ylabel("Customers needed per arm (log scale)")
ax.set_title("Detecting a change in churn", weight="bold")
ax = axes[1]; ps = np.arange(0.03, 0.2001, 0.005)
ax.plot(ps * 100, [n_spend(p) for p in ps], color=C_PRIME, lw=2.5); ax.set_yscale("log")
ax.axhline(avail, color=C_COST, ls="--", lw=1.5)
ax.text(19.8, avail * 1.1, f"available here: {avail:,} per arm", ha="right", fontsize=10, color="#222")
ax.set_xlabel("Smallest change in spend to detect (%)"); ax.set_ylabel("Customers needed per arm (log scale)")
ax.set_title("Detecting a change in spend", weight="bold")
plt.tight_layout(); plt.show()

 churn change detected (pts)  customers per arm
                         1.0              23665
                         2.0               5789
                         3.0               2515
                         5.0                862
 spend change detected (%)  customers per arm
                       5.0               4670
                      10.0               1168
                      15.0                519

Here (2,533 heavy redeemers, 1,266 per arm) we could only detect: spend change ~10% or churn change ~4.2 pts
A RISE in churn needs ~4.6 pts to be detected, so the guardrail is blunt at this size.
Population        Heavy redeemers (real engagers + deal-seekers)
Arms              Control (current cashback) | A: 25% lower cashback rate | B: higher rate only
                  after activity milestones
Assignment        Random, stratified by cashback segment; same dates for all arms
Duration          8 to 12 weeks (assumption), plus 90-day retention read
Primary metric    N

### Procedure
1. Calculated how many customers each test group (arm) would need to detect a given change in churn or spend, at 5% significance and 80% power.
2. Compared that with what we have: 2,533 heavy redeemers split into two arms gives 1,266 customers per arm.
3. Wrote the test design as a one-page term sheet: population, arms, assignment, duration, metrics, guardrails and decision rule.

### Observations
- **Sample size needed per arm:**

| Smallest change to detect | Customers per arm |
|---|---|
| Churn: 1 point | 23,665 |
| Churn: 2 points | 5,789 |
| Churn: 3 points | 2,515 |
| Churn: 5 points | 862 |
| Spend: 5% | 4,670 |
| Spend: 10% | 1,168 |
| Spend: 15% | 519 |

- **Our own data limits what a test could detect.** With 1,266 per arm we can detect a spend change of about 10% or a churn change of about 4.2 points, and nothing smaller. A rise in churn, the harmful direction, needs about 4.6 points to be detected.
- **Spend and net contribution are the right primary metrics.** Churn moves too slowly for a test this size, so it should be monitored as a guardrail and not used as the deciding result.
- **The decision rule needs a caveat.** It says roll out only if churn rises by at most 1 point, but a test this size cannot reliably detect a 1-point change. A real test needs a much larger population (the whole customer base, not only heavy redeemers) or a longer follow-up before we can make that claim.
- **Design:** control vs A (25% lower cashback rate) vs B (higher rate only after activity milestones). Randomise within each cashback segment, run for 8 to 12 weeks (assumed), and track NPS, support contacts and complaints as guardrails.
- **Business reading:** testing is cheap compared with the 19% bill saving at stake, but the test must be sized for the effect we care about, otherwise a "no difference" result would be meaningless.

## Q5 conclusion: how to personalise offers to improve retention

- **Personalise by behaviour, not by profile.** Age, city, gender, tenure and Prime status do not predict churn or value. Redemption and activity patterns do separate customers, so the playbook is built on them.
- **Do not increase cashback for everyone.** Heavy redeemers take 76% of the bill, and redemption is unrelated to engagement.
- **Four actions:** make cashback conditional for deal-seekers, re-tier it for real engagers, reward organic loyalists without extra cashback, and use low-cost nudges for passive customers.
- **A 25% reduction for heavy redeemers saves about 19% of the redeemed bill (13% after reinvesting 30%), or 13% if every earned point is eventually redeemed (about 9% after reinvesting).** It pays off unless heavy redeemers cut their spending by more than the break-even.
- **Next step: run the A/B test**, sized for spend and net contribution, with churn as a guardrail.

## Executive summary

**Bottom line:** we could not find who will leave, but we can see where money is wasted. Fixing that is a decision the data supports.

| Question | Answer | Evidence | How robust |
|---|---|---|---|
| **Q1. What drives churn?** | Nothing in this data. About 19% of customers churn, regardless of who they are or how they behave. | Single features, pairs of features (0 of 21 pairs significant), engineered segments, service friction (support contacts vs churn: p = 0.94), reward pull (p = 0.85), predictive models (at chance level), clustering and PCA (no structure). | Eight independent lenses agree, and a group of 1,000 customers would need a churn gap of 4 to 5 points to be flagged, so large drivers are ruled out and small ones are not. The honest conclusion is "no driver found", not "no churn". |
| **Q2. Does Prime help?** | No behavioural advantage. Prime members transact, spend and stay the same, yet cost about 38% more in cashback per customer (45% more per ₹100 spent). | Standardised differences near zero on every behaviour. Cashback cost per customer is about ₹4,214 (Prime) vs ₹3,043 (Non-Prime). | The premium ranges from +28% to +51% across assumption sets. It is always positive. |
| **Q3. Who is most valuable?** | Value follows behaviour, not profile. Organic loyalists (33% of volume for 16% of the bill) are the most profitable segment. | Age, city, gender, tenure and Prime do not separate value. Behaviour segments do (by design, so profile segments are the real test). | Rates are identical across segments, so shares do not change with the rate card. |
| **Q4. Is cashback driving real engagement?** | Mostly not. Heavy redeemers take 76% of redeemed cashback for 51% of volume. That share follows largely from how "heavy" is defined, so the finding is what it buys: their churn is 18.4% vs 19.6% for light redeemers, no lower. Higher reward pull costs far more (top 20% cost 28x the bottom 20%) but brings no lower churn and no higher NPS. | Redemption is unrelated to engagement, tenure, NPS or churn. Reward pull rises with activity only by construction. | The 76% share is the same in all five assumption sets. Counting earned (not yet redeemed) points, it is 51%. |
| **Q5. How to personalise?** | A behaviour-based cashback playbook: four segments, one action each. Cut cashback rates 25% for heavy redeemers, tested first with an A/B test. | Saves 19% of the redeemed bill (13% if every earned point is eventually redeemed), about 13% / 9% after reinvesting 30%. A 25% cut still pays off unless heavy redeemers reduce their spending by more than about 20% (at ₹3 revenue per ₹100) to 39% (at ₹2). | Range of 13% to 19% is the honest headline. It comes from the redeemed vs earned basis (sampling luck moves it by under one point). Each saving is paired with its own break-even (Cell 19A). |

**How the pieces fit together:** customers do not differ in who they are, so profile-based targeting would be guesswork. They differ sharply in how they use rewards, and that is where the cost sits: a fifth of customers drive 59% of the bill, and extra reward pull does not buy loyalty. So the lever is not predicting churn but paying cashback for activity and rewarding loyal customers without extra cashback.

**Service and satisfaction:** support contacts do not predict churn or unhappiness (unhappy-but-active customers contact support no more than others), and `friction_per_year` is distorted for new customers. Unhappy customers must be found with NPS surveys, not support logs. 861 active-but-unhappy customers are the first service-recovery list.

**What to do next:**
1. Run the A/B test on heavy redeemers (control vs 25% lower rate vs milestone-based rate), sized for spend and net contribution.
2. Treat churn as a guardrail, not a decision metric. With about 1,266 customers per arm the test can only detect a churn change of about 4 points.
3. Replace the illustrative revenue rates with Amazon Pay's actual revenue per ₹100 before final sign-off.

**Caveats:**
- The cashback rates come from a blog summary of the ICICI card (not the issuer's terms). The 40/30/30 spend mix and the 85% eligible share are assumptions. Stress test: Cell 7A.
- Rewards are non-expiring points per the same source, so the 13% to 19% range reflects both cash-out and earned views.
- There is no revenue or margin in the data, and spend exceeds income for many customers, so read shares and percentages, not absolute rupee values.
- The data is a single snapshot with no time dimension, the 12-month churn window is assumed, and the features look independently generated.

In [35]:
# ===== CELL 21: EXPORT TABLES FOR THE DASHBOARD =====
# TECH: to_csv() writes a DataFrame to a plain-text table that any tool (HTML/JavaScript, Excel) can read.
#       index=True keeps row labels such as segment names; we reset them into proper columns first.
# BIZ:  The dashboard needs the same numbers you have just seen, so we save each result once,
#       in one folder, instead of retyping figures by hand.
import os, json
OUT = "../outputs/dashboard_data"; os.makedirs(OUT, exist_ok=True)

def save(df_, name):
    df_.to_csv(f"{OUT}/{name}.csv", index=False); print(f"saved {name}.csv  {df_.shape}")

# 1) One row per customer (5,000 rows): lets the dashboard filter, count and chart by any field
cust_cols = ["customer_id", "age", "gender", "city_tier", "monthly_income_inr", "tenure_months", "amazon_prime_member",
             "avg_monthly_transactions", "avg_transaction_value_inr", "cashback_used_pct", "app_engagement_score",
             "customer_support_contacts", "net_promoter_score", "churned", "monthly_spend", "engagement_index",
             "reward_pull", "est_cashback_cost", "cashback_segment", "say_do_gap", "nps_group", "lifecycle_stage", "age_band"]
cust_cols = [c for c in cust_cols if c in fe.columns]               # skip any column your file names differently
save(fe[cust_cols].assign(reward_pull_band=fe.pull_band.astype(str)).round(4), "customers")

# 2) Playbook: one row per behaviour segment (Cell 18)
save(playbook.reset_index().rename(columns={"index": "cashback_segment"}), "playbook_segments")

# 3) Cashback what-if: break-even table (Cell 19) in long format = easy to chart
be = (tbl * 100).round(1).reset_index().rename(columns={"revenue per Rs100": "revenue_per_100"})
be = be.melt(id_vars="revenue_per_100", var_name="cashback_cut", value_name="max_volume_loss_pct")
save(be, "breakeven")
save(saving.reset_index().rename(columns={"index": "cashback_cut"}), "savings")

# 4) A/B test sizing (Cell 20)
save(pd.DataFrame({"metric": "churn change (pts)", "effect": [d * 100 for d in churn_deltas], "customers_per_arm": [n_churn(d) for d in churn_deltas]})
     .pipe(lambda a: pd.concat([a, pd.DataFrame({"metric": "spend change (%)", "effect": [p * 100 for p in spend_pcts], "customers_per_arm": [n_spend(p) for p in spend_pcts]})])), "ab_sample_sizes")
save(design, "ab_design")

# 5) Assumption stress test (Cell 7A) and reward-pull bands (Cell 17C)
save(sens.reset_index().rename(columns={"index": "metric"}), "assumption_scenarios")
save(q.reset_index(), "reward_pull_bands")

# 6) Prime vs Non-Prime summary
prime = fe.groupby("amazon_prime_member").agg(customers=("churned", "size"), avg_spend=("monthly_spend", "mean"),
        avg_cashback_cost=("est_cashback_cost", "mean"), avg_txns=("avg_monthly_transactions", "mean"),
        avg_tenure=("tenure_months", "mean"), avg_nps=("net_promoter_score", "mean"), churn=("churned", "mean")).reset_index()
prime["amazon_prime_member"] = prime.amazon_prime_member.map({1: "Prime", 0: "Non-Prime"})
save(prime, "prime_vs_nonprime")

# 7) Headline numbers and key assumptions in one small JSON file
heavy_mask = fe.cashback_segment.isin(["Real engager", "Deal-seeker"])
red_all = fe.est_cashback_cost.sum(); earn = fe.monthly_spend * fe.cashback_rate   # redeemed and earned cashback
kpis = {
    "customers": int(len(fe)), "churn_rate": round(float(fe.churned.mean()), 4),
    "heavy_redeemers": int(heavy_mask.sum()),
    "heavy_share_of_redeemed_bill": round(float(fe.est_cashback_cost[heavy_mask].sum() / red_all), 4),
    "heavy_cost_per_100": round(float(fe.est_cashback_cost[heavy_mask].sum() / fe.monthly_spend[heavy_mask].sum() * 100), 3),
    "saving_25pct_cut_redeemed_view": round(float(0.25 * fe.est_cashback_cost[heavy_mask].sum() / red_all), 4),
    "saving_25pct_cut_earned_view": round(float(0.25 * earn[heavy_mask].sum() / earn.sum()), 4),
    "reinvest_share": 0.30,
    "assumptions": {"rate_amazon": CFG["rate_amazon"], "rate_partner": CFG["rate_partner"], "rate_other": CFG["rate_other"],
                    "mix_amazon": CFG["mix_amazon"], "mix_partner": CFG["mix_partner"], "mix_other": CFG["mix_other"],
                    "eligible_share_amazon": CFG["eligible_share_amazon"]},
}
with open(f"{OUT}/kpis.json", "w") as f: json.dump(kpis, f, indent=2)
print("saved kpis.json"); print(json.dumps(kpis, indent=2)[:600])
print("\nFolder:", os.path.abspath(OUT)); print(sorted(os.listdir(OUT)))

saved customers.csv  (5000, 24)
saved playbook_segments.csv  (4, 10)
saved breakeven.csv  (33, 3)
saved savings.csv  (3, 3)
saved ab_sample_sizes.csv  (7, 3)
saved ab_design.csv  (8, 2)
saved assumption_scenarios.csv  (8, 6)
saved reward_pull_bands.csv  (5, 11)
saved prime_vs_nonprime.csv  (2, 8)
saved kpis.json
{
  "customers": 5000,
  "churn_rate": 0.19,
  "heavy_redeemers": 2533,
  "heavy_share_of_redeemed_bill": 0.7622,
  "heavy_cost_per_100": 1.43,
  "saving_25pct_cut_redeemed_view": 0.1906,
  "saving_25pct_cut_earned_view": 0.127,
  "reinvest_share": 0.3,
  "assumptions": {
    "rate_amazon": {
      "prime": 0.05,
      "nonprime": 0.03
    },
    "rate_partner": 0.01,
    "rate_other": 0.01,
    "mix_amazon": 0.4,
    "mix_partner": 0.3,
    "mix_other": 0.3,
    "eligible_share_amazon": 0.85
  }
}

Folder: /home/claude/hz/v2/dashboard_data
['ab_design.csv', 'ab_sample_sizes.csv', 'assumption_scenarios.csv', 'breakeven.csv', 'customers.csv', 'kpis.json', 'playbook_segments.csv'